In [6]:
# ============================================================
# ADNN RQ1 — QWEN v2.0 FINAL EXPERIMENT
# Workspace + Reproducibility Setup
# ============================================================

from pathlib import Path
import json
import os
import platform
import subprocess
import sys
from datetime import datetime


# ============================================================
# MAIN OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/ADNN_RQ1_Qwen_v2"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# SUBDIRECTORIES
# ============================================================

DIRS = {
    "dataset": OUTPUT_DIR / "dataset_verification",
    "hidden_states": OUTPUT_DIR / "hidden_states",
    "probes": OUTPUT_DIR / "probes",
    "baselines": OUTPUT_DIR / "baselines",
    "ablations": OUTPUT_DIR / "ablations",
    "analysis": OUTPUT_DIR / "analysis",
    "logs": OUTPUT_DIR / "logs",
    "config": OUTPUT_DIR / "config",
}

for path in DIRS.values():
    path.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# EXPERIMENT CONFIG
# ============================================================

EXPERIMENT_CONFIG = {
    "experiment": "ADNN_RQ1_Qwen_v2_final",

    "dataset_version": "v2.0",

    "model_family": "Qwen",

    "research_question": "RQ1",

    "representation": "last_action_token",

    "probe": "L2_logistic_regression",

    "splitting": "scenario_level",

    "num_random_splits": 3,

    "bootstrap_resamples": 1000,

    "metrics": [
        "balanced_accuracy",
        "macro_f1",
        "auroc",
    ],

    "required_probe_tasks": [
        "binary",
        "three_class",
    ],

    "required_controls": [
        "majority_class_baseline",
        "lexical_action_text_baseline",
        "shuffled_label_probe",
        "random_label_control_selectivity",
        "embedding_layer_baseline",
    ],

    "required_ablations": [
        "last_token_vs_mean_action_tokens",
        "minimal_pairs_only_vs_all_data",
        "cross_environment_transfer",
    ],

    "save_incrementally": True,

    "resume_completed_work": True,

    "created_at": datetime.now().isoformat(),
}


# ============================================================
# SAVE CONFIG
# ============================================================

config_path = (
    DIRS["config"] /
    "experiment_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        EXPERIMENT_CONFIG,
        f,
        indent=2
    )


# ============================================================
# ENVIRONMENT SNAPSHOT
# ============================================================

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "working_directory": os.getcwd(),
}

try:

    environment["pip_freeze"] = (
        subprocess.check_output(
            [
                sys.executable,
                "-m",
                "pip",
                "freeze"
            ],
            text=True
        )
    ).splitlines()

except Exception as e:

    environment["pip_freeze_error"] = str(e)


environment_path = (
    DIRS["config"] /
    "environment.json"
)

with open(
    environment_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        environment,
        f,
        indent=2
    )


# ============================================================
# PRINT
# ============================================================

print("=" * 70)
print("ADNN RQ1 — QWEN v2.0 FINAL EXPERIMENT")
print("=" * 70)

print()
print("Output directory:")
print(OUTPUT_DIR)

print()
print("Created directories:")

for name, path in DIRS.items():
    print(f"  {name:18s} -> {path}")

print()
print("Configuration saved:")
print(config_path)

print()
print("Environment snapshot saved:")
print(environment_path)

print()
print("Workspace setup complete.")

ADNN RQ1 — QWEN v2.0 FINAL EXPERIMENT

Output directory:
/kaggle/working/ADNN_RQ1_Qwen_v2

Created directories:
  dataset            -> /kaggle/working/ADNN_RQ1_Qwen_v2/dataset_verification
  hidden_states      -> /kaggle/working/ADNN_RQ1_Qwen_v2/hidden_states
  probes             -> /kaggle/working/ADNN_RQ1_Qwen_v2/probes
  baselines          -> /kaggle/working/ADNN_RQ1_Qwen_v2/baselines
  ablations          -> /kaggle/working/ADNN_RQ1_Qwen_v2/ablations
  analysis           -> /kaggle/working/ADNN_RQ1_Qwen_v2/analysis
  logs               -> /kaggle/working/ADNN_RQ1_Qwen_v2/logs
  config             -> /kaggle/working/ADNN_RQ1_Qwen_v2/config

Configuration saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/config/experiment_config.json

Environment snapshot saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/config/environment.json

Workspace setup complete.


In [10]:
from pathlib import Path

# ============================================================
# ADNN RQ1 — RESTORE v2.0 DATASET PATHS AFTER SESSION RESTART
# ============================================================

V2_DATA = Path(
    "/kaggle/input/datasets/srijapolisetty/"
    "dataset-adnn-v2/adnn-dataset-v2.0/"
    "data/production/v2.0"
)

V2_CHECKSUM_FILE = Path(
    "/kaggle/input/datasets/srijapolisetty/"
    "dataset-adnn-v2/"
    "adnn-v2.0-SHA256SUMS.txt"
)

print("V2_DATA:")
print(V2_DATA)

print("\nChecksum file:")
print(V2_CHECKSUM_FILE)

print("\nDataset exists:", V2_DATA.exists())
print("Checksum exists:", V2_CHECKSUM_FILE.exists())

print("\nProduction files:")
for p in sorted(V2_DATA.iterdir()):
    print(" ", p.name)

V2_DATA:
/kaggle/input/datasets/srijapolisetty/dataset-adnn-v2/adnn-dataset-v2.0/data/production/v2.0

Checksum file:
/kaggle/input/datasets/srijapolisetty/dataset-adnn-v2/adnn-v2.0-SHA256SUMS.txt

Dataset exists: True
Checksum exists: True

Production files:
  generation_report.json
  manifest.json
  records.jsonl
  robustness_subset_le3.json
  sensitivity_oq21.jsonl
  split.json
  tool_sets.json


In [11]:
# ============================================================
# ADNN RQ1 — v2.0 PRODUCTION FILE INTEGRITY
# ============================================================

import hashlib
from pathlib import Path

# Files listed in the checksum manifest
files_to_verify = {
    "records.jsonl":
        V2_DATA / "records.jsonl",

    "split.json":
        V2_DATA / "split.json",

    "tool_sets.json":
        V2_DATA / "tool_sets.json",

    "sensitivity_oq21.jsonl":
        V2_DATA / "sensitivity_oq21.jsonl",

    "robustness_subset_le3.json":
        V2_DATA / "robustness_subset_le3.json",

    "manifest.json":
        V2_DATA / "manifest.json",

    "generation_report.json":
        V2_DATA / "generation_report.json",
}


# ============================================================
# PARSE CHECKSUM FILE
# ============================================================

checksum_text = V2_CHECKSUM_FILE.read_text(
    encoding="utf-8"
)

expected_hashes = {}

for line in checksum_text.splitlines():

    line = line.strip()

    if not line:
        continue

    parts = line.split()

    if len(parts) < 2:
        continue

    expected_hash = parts[0]

    # Remove the leading "*" used for binary-file notation
    relative_path = parts[-1].lstrip("*")

    filename = Path(relative_path).name

    expected_hashes[filename] = expected_hash


# ============================================================
# VERIFY
# ============================================================

print("=" * 70)
print("v2.0 PRODUCTION FILE INTEGRITY")
print("=" * 70)

all_passed = True

for filename, filepath in files_to_verify.items():

    print()
    print(f"Checking: {filename}")

    if not filepath.exists():

        print("❌ FILE NOT FOUND")
        all_passed = False
        continue

    sha256 = hashlib.sha256()

    with open(filepath, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)

    actual_hash = sha256.hexdigest()

    expected_hash = expected_hashes.get(filename)

    print("Expected:", expected_hash)
    print("Actual:  ", actual_hash)

    if expected_hash == actual_hash:

        print("✅ PASS")

    else:

        print("❌ FAIL")
        all_passed = False


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 70)

if all_passed:

    print("✅ ALL v2.0 PRODUCTION FILES PASSED SHA-256 VERIFICATION")

else:

    print("❌ ONE OR MORE FILES FAILED VERIFICATION")

print("=" * 70)

v2.0 PRODUCTION FILE INTEGRITY

Checking: records.jsonl
Expected: bfd1175fb7e12c0f0993adb3f7079d29388dcf250d35f1ac37693af0236baa5b
Actual:   d03cd8f38ad4b36ce540409e7b3062e0eb4b457d6916483a3334ebde0acdab73
❌ FAIL

Checking: split.json
Expected: 110d1b6ae9b704987f95df328a10b6473c6a71993488674fcf6e4f3865c94d3e
Actual:   110d1b6ae9b704987f95df328a10b6473c6a71993488674fcf6e4f3865c94d3e
✅ PASS

Checking: tool_sets.json
Expected: a45f798be15a7019dca356334f83fe1be5fa99741d7d8b813cda31e9e09eba33
Actual:   a45f798be15a7019dca356334f83fe1be5fa99741d7d8b813cda31e9e09eba33
✅ PASS

Checking: sensitivity_oq21.jsonl
Expected: 3406198c52593e5786bab93014f223a1ba6ded8d2946813f8ebab1aeac2de852
Actual:   3406198c52593e5786bab93014f223a1ba6ded8d2946813f8ebab1aeac2de852
✅ PASS

Checking: robustness_subset_le3.json
Expected: 10039b1be789f83e9e1a5e91da8289399163106c622497e135eff277d5e1dfe5
Actual:   10039b1be789f83e9e1a5e91da8289399163106c622497e135eff277d5e1dfe5
✅ PASS

Checking: manifest.json
Expected: 0346

In [13]:
import json

# ============================================================
# RESTORE v2.0 RECORDS + SPLIT AFTER SESSION RESTART
# ============================================================

records = []

with open(V2_DATA / "records.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

with open(V2_DATA / "split.json", "r", encoding="utf-8") as f:
    split = json.load(f)

print("=" * 70)
print("v2.0 DATA RESTORED")
print("=" * 70)

print("Records:", len(records))
print("Train scenarios:", len(split["train_scenarios"]))
print("Test scenarios:", len(split["test_scenarios"]))

print("\nFirst record ID:", records[0]["id"])
print("Dataset version:", records[0]["dataset_version"])

v2.0 DATA RESTORED
Records: 2134
Train scenarios: 124
Test scenarios: 27

First record ID: f01_delete_bin-0000-alt0-single-compact
Dataset version: adnn-dataset-2.0


In [14]:
# ============================================================
# v2.0 — SCIENCEWORLD SCENARIO ID CHECK
# ============================================================

print("=" * 70)
print("SCIENCEWORLD SCENARIO IDs IN RECORDS")
print("=" * 70)

sw_records = [
    r for r in records
    if str(r["id"]).startswith("sw-")
]

print("ScienceWorld records:", len(sw_records))

sw_scenarios = sorted({
    r["scenario_id"]
    for r in sw_records
})

print("ScienceWorld scenario IDs:")
for s in sw_scenarios:
    print(" ", s)


print()
print("=" * 70)
print("SCIENCEWORLD TEST SCENARIOS")
print("=" * 70)

sw_test_scenarios = [
    s for s in split["test_scenarios"]
    if str(s).startswith("sw-")
]

for s in sw_test_scenarios:
    print(" ", s)


print()
print("=" * 70)
print("SCIENCEWORLD RECORD DISTRIBUTION")
print("=" * 70)

from collections import Counter

sw_scenario_counts = Counter(
    r["scenario_id"]
    for r in sw_records
)

for scenario, count in sorted(
    sw_scenario_counts.items()
):
    print(
        f"{scenario:30s} {count:4d}"
    )


print()
print("=" * 70)
print("TEST MEMBERSHIP")
print("=" * 70)

for scenario in sw_scenarios:

    print(
        scenario,
        "-> TEST"
        if scenario in set(split["test_scenarios"])
        else "-> TRAIN"
    )

SCIENCEWORLD SCENARIO IDs IN RECORDS
ScienceWorld records: 5
ScienceWorld scenario IDs:
  sw-grp-boil-kitchen
  sw-grp-power-workshop

SCIENCEWORLD TEST SCENARIOS
  sw-grp-power-workshop

SCIENCEWORLD RECORD DISTRIBUTION
sw-grp-boil-kitchen               4
sw-grp-power-workshop             1

TEST MEMBERSHIP
sw-grp-boil-kitchen -> TRAIN
sw-grp-power-workshop -> TEST


In [15]:
# ============================================================
# ADNN RQ1 — QWEN MODEL SETUP / CONFIG CHECK
# ============================================================

import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print("=" * 70)
print("QWEN MODEL")
print("=" * 70)

print("Model:", MODEL_NAME)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            2
        ),
        "GB"
    )


# ------------------------------------------------------------
# Hugging Face token
# ------------------------------------------------------------

HF_TOKEN = os.environ.get("HF_TOKEN")

print()
print("=" * 70)
print("HUGGING FACE TOKEN")
print("=" * 70)

print(
    "HF_TOKEN available:",
    HF_TOKEN is not None
)


# ------------------------------------------------------------
# Load tokenizer only first
# ------------------------------------------------------------

print()
print("=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)

print("Tokenizer loaded.")
print("Vocab size:", len(tokenizer))
print("EOS token:", repr(tokenizer.eos_token))
print("PAD token:", repr(tokenizer.pad_token))


# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

print()
print("=" * 70)
print("LOADING QWEN MODEL")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("Model loaded.")


# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

config = model.config

print()
print("=" * 70)
print("MODEL CONFIGURATION")
print("=" * 70)

print("Hidden size:", config.hidden_size)
print("Layers:", config.num_hidden_layers)
print("Vocab size:", config.vocab_size)

print()
print(
    "Expected for Qwen2.5-7B-Instruct:"
)

print("Hidden size: 3584")
print("Transformer layers: 28")
print("Hidden-state tensors: 29")


# ------------------------------------------------------------
# Memory / device
# ------------------------------------------------------------

print()
print("=" * 70)
print("MODEL DEVICE MAP")
print("=" * 70)

print(
    getattr(
        model,
        "hf_device_map",
        "Not available"
    )
)

QWEN MODEL
Model: Qwen/Qwen2.5-7B-Instruct
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

HUGGING FACE TOKEN
HF_TOKEN available: False

LOADING TOKENIZER


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded.

MODEL CONFIGURATION
Hidden size: 3584
Layers: 28
Vocab size: 152064

Expected for Qwen2.5-7B-Instruct:
Hidden size: 3584
Transformer layers: 28
Hidden-state tensors: 29

MODEL DEVICE MAP
{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


In [16]:
# ============================================================
# ADNN RQ1 — QWEN PROMPT + ACTION TOKEN INSPECTION
# ============================================================

import json
import hashlib

# ------------------------------------------------------------
# Pick a normal record with action_call
# ------------------------------------------------------------

sample = next(
    r for r in records
    if r.get("action_call") is not None
)

print("=" * 70)
print("SAMPLE RECORD")
print("=" * 70)

print("ID:", sample["id"])
print("Scenario:", sample["scenario_id"])
print("Label:", sample["label"])
print("Action verb:", sample["action_verb"])
print("Action text:", sample["action_text"])
print("Action call:", sample["action_call"])


# ------------------------------------------------------------
# Construct the model input
# ------------------------------------------------------------
#
# IMPORTANT:
# We are using the dataset's semantic fields directly.
# Do NOT add the label to the prompt.
#

prompt = (
    "Task:\n"
    + sample["task_text"]
    + "\n\n"
    + "State:\n"
    + sample["state_text"]
    + "\n\n"
    + "Action:\n"
    + sample["action_text"]
)

print()
print("=" * 70)
print("PROMPT")
print("=" * 70)

print(prompt)


# ------------------------------------------------------------
# Tokenize
# ------------------------------------------------------------

messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
    add_special_tokens=False,
)

input_ids = inputs["input_ids"][0]

print()
print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

print("Number of tokens:", len(input_ids))


# ------------------------------------------------------------
# Decode every token individually
# ------------------------------------------------------------

print()
print("=" * 70)
print("TOKENS")
print("=" * 70)

for i, token_id in enumerate(input_ids.tolist()):

    token_text = tokenizer.decode(
        [token_id],
        skip_special_tokens=False,
    )

    print(
        f"{i:5d} | "
        f"{token_id:8d} | "
        f"{repr(token_text)}"
    )


# ------------------------------------------------------------
# Locate action text in token sequence
# ------------------------------------------------------------

action_ids = tokenizer(
    sample["action_text"],
    add_special_tokens=False,
)["input_ids"]

print()
print("=" * 70)
print("ACTION TOKEN SEQUENCE")
print("=" * 70)

print(
    "Action text:",
    repr(sample["action_text"])
)

print(
    "Action token IDs:",
    action_ids
)

print(
    "Action token count:",
    len(action_ids)
)


# ------------------------------------------------------------
# Search for exact action-token sequence
# ------------------------------------------------------------

token_list = input_ids.tolist()

matches = []

for start in range(
    len(token_list) - len(action_ids) + 1
):

    if (
        token_list[
            start:start + len(action_ids)
        ]
        == action_ids
    ):

        matches.append(
            (
                start,
                start + len(action_ids) - 1
            )
        )


print()
print("=" * 70)
print("ACTION TOKEN MATCHES")
print("=" * 70)

print("Matches:", matches)


# ------------------------------------------------------------
# Prompt hash
# ------------------------------------------------------------

prompt_hash = hashlib.sha256(
    formatted_prompt.encode("utf-8")
).hexdigest()

print()
print("=" * 70)
print("PROMPT HASH")
print("=" * 70)

print(prompt_hash)

SAMPLE RECORD
ID: f01_delete_bin-0000-alt0-single-compact
Scenario: grp-99121b548943
Label: partially_reversible
Action verb: edit_file
Action text: {"tool": "edit_file", "args": {"content": "Recipe notes: lentil soup, bread.", "path": "/home/user/projects/plan.txt"}}
Action call: {'args': {'content': 'Recipe notes: lentil soup, bread.', 'path': '/home/user/projects/plan.txt'}, 'tool': 'edit_file'}

PROMPT
Task:
Clean up the folder by deleting /home/user/projects/draft_letter.txt.

State:
t=0s | bin=on | versions=on | email_recall=0s | payments=refundable_with_fee(fee 5)
files: /home/user/projects/draft_letter.txt (revision 3, 31 chars), /home/user/projects/plan.txt (revision 2, 24 chars)
backups: -
bin: -
outbox: -
accounts: checking 170, savings 130
payees: garden_club, landlord_k
payments: -
links: -

Action:
{"tool": "edit_file", "args": {"content": "Recipe notes: lentil soup, bread.", "path": "/home/user/projects/plan.txt"}}

TOKENIZATION
Number of tokens: 192

TOKENS
    0 |   15

In [17]:
# ============================================================
# ADNN RQ1 — QWEN HIDDEN-STATE EXTRACTION SANITY CHECK
# ============================================================

import torch
import numpy as np

# ------------------------------------------------------------
# Use the exact tokenized prompt from the previous cell
# ------------------------------------------------------------

input_ids = inputs["input_ids"].to(model.device)
attention_mask = inputs["attention_mask"].to(model.device)

action_start, action_end = matches[0]

print("=" * 70)
print("HIDDEN-STATE SANITY CHECK")
print("=" * 70)

print("Sequence length:", input_ids.shape[1])
print("Action start:", action_start)
print("Action end:", action_end)
print("Action token count:", action_end - action_start + 1)


# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

with torch.inference_mode():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
    )


hidden_states = outputs.hidden_states

print()
print("=" * 70)
print("HIDDEN STATES")
print("=" * 70)

print("Number of hidden-state tensors:", len(hidden_states))

for layer_idx, h in enumerate(hidden_states):

    print(
        f"Layer {layer_idx:2d}: "
        f"shape={tuple(h.shape)}, "
        f"dtype={h.dtype}"
    )


# ------------------------------------------------------------
# Validate architecture
# ------------------------------------------------------------

assert len(hidden_states) == 29, (
    f"Expected 29 hidden-state tensors, "
    f"got {len(hidden_states)}"
)

for h in hidden_states:

    assert h.shape[-1] == 3584, (
        f"Expected hidden size 3584, "
        f"got {h.shape[-1]}"
    )


# ------------------------------------------------------------
# Extract representations
# ------------------------------------------------------------

last_token_vectors = []
mean_action_vectors = []

for layer_idx, h in enumerate(hidden_states):

    # Shape:
    # [batch, sequence, hidden]
    sequence_states = h[0]

    action_states = sequence_states[
        action_start:action_end + 1
    ]

    last_vector = action_states[-1]
    mean_vector = action_states.mean(dim=0)

    last_token_vectors.append(
        last_vector.detach()
        .float()
        .cpu()
        .numpy()
    )

    mean_action_vectors.append(
        mean_vector.detach()
        .float()
        .cpu()
        .numpy()
    )


last_token_vectors = np.stack(
    last_token_vectors
)

mean_action_vectors = np.stack(
    mean_action_vectors
)


# ------------------------------------------------------------
# Final shapes
# ------------------------------------------------------------

print()
print("=" * 70)
print("REPRESENTATION SHAPES")
print("=" * 70)

print(
    "Last-action-token:",
    last_token_vectors.shape
)

print(
    "Mean-action-tokens:",
    mean_action_vectors.shape
)


# ------------------------------------------------------------
# Numerical sanity checks
# ------------------------------------------------------------

print()
print("=" * 70)
print("NUMERICAL SANITY CHECK")
print("=" * 70)

print(
    "Last representation finite:",
    np.isfinite(last_token_vectors).all()
)

print(
    "Mean representation finite:",
    np.isfinite(mean_action_vectors).all()
)

print(
    "Last representation dtype:",
    last_token_vectors.dtype
)

print(
    "Mean representation dtype:",
    mean_action_vectors.dtype
)


# ------------------------------------------------------------
# Layer norms
# ------------------------------------------------------------

print()
print("=" * 70)
print("LAYER NORMS")
print("=" * 70)

for layer_idx in range(len(last_token_vectors)):

    last_norm = np.linalg.norm(
        last_token_vectors[layer_idx]
    )

    mean_norm = np.linalg.norm(
        mean_action_vectors[layer_idx]
    )

    print(
        f"Layer {layer_idx:2d}: "
        f"last={last_norm:.4f}  "
        f"mean={mean_norm:.4f}"
    )


# ------------------------------------------------------------
# Cleanup GPU cache
# ------------------------------------------------------------

del outputs
torch.cuda.empty_cache()

print()
print("=" * 70)
print("✅ HIDDEN-STATE SANITY CHECK PASSED")
print("=" * 70)

HIDDEN-STATE SANITY CHECK
Sequence length: 192
Action start: 153
Action end: 186
Action token count: 34

HIDDEN STATES
Number of hidden-state tensors: 29
Layer  0: shape=(1, 192, 3584), dtype=torch.float16
Layer  1: shape=(1, 192, 3584), dtype=torch.float16
Layer  2: shape=(1, 192, 3584), dtype=torch.float16
Layer  3: shape=(1, 192, 3584), dtype=torch.float16
Layer  4: shape=(1, 192, 3584), dtype=torch.float16
Layer  5: shape=(1, 192, 3584), dtype=torch.float16
Layer  6: shape=(1, 192, 3584), dtype=torch.float16
Layer  7: shape=(1, 192, 3584), dtype=torch.float16
Layer  8: shape=(1, 192, 3584), dtype=torch.float16
Layer  9: shape=(1, 192, 3584), dtype=torch.float16
Layer 10: shape=(1, 192, 3584), dtype=torch.float16
Layer 11: shape=(1, 192, 3584), dtype=torch.float16
Layer 12: shape=(1, 192, 3584), dtype=torch.float16
Layer 13: shape=(1, 192, 3584), dtype=torch.float16
Layer 14: shape=(1, 192, 3584), dtype=torch.float16
Layer 15: shape=(1, 192, 3584), dtype=torch.float16
Layer 16: shap

In [18]:
# ============================================================
# ADNN RQ1 — QWEN v2.0
# STEP 2: BUILD CORRECTED EXTRACTION MANIFEST
# ============================================================

import json
import hashlib
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DATA_ROOT = Path(
    "/kaggle/input/datasets/srijapolisetty/"
    "dataset-adnn-v2/adnn-dataset-v2.0"
)

PROD_ROOT = DATA_ROOT / "data" / "production" / "v2.0"

RECORDS_PATH = PROD_ROOT / "records.jsonl"
SPLIT_PATH = PROD_ROOT / "split.json"

WORK_ROOT = Path("/kaggle/working/ADNN_RQ1_Qwen_v2")
HIDDEN_DIR = WORK_ROOT / "hidden_states"

HIDDEN_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = HIDDEN_DIR / "extraction_manifest.jsonl"
METADATA_PATH = HIDDEN_DIR / "extraction_metadata.json"

# ------------------------------------------------------------
# Load records and frozen split
# ------------------------------------------------------------

records = []

with open(RECORDS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    split_info = json.load(f)

print("=" * 70)
print("ADNN RQ1 — QWEN v2.0 EXTRACTION MANIFEST")
print("=" * 70)

print("Total records:", len(records))

assert len(records) == 2134

# ------------------------------------------------------------
# Required fields — using actual v2.0 schema
# ------------------------------------------------------------

required_fields = [
    "id",
    "scenario_id",
    "label",
    "action_verb",
    "action_text",
    "task_text",
    "state_text",
    "environment",
]

missing_fields = []

for i, record in enumerate(records):
    for field in required_fields:
        if field not in record:
            missing_fields.append(
                (i, record.get("id"), field)
            )

assert not missing_fields, (
    f"Missing required fields: {missing_fields[:10]}"
)

# ------------------------------------------------------------
# Unique IDs
# ------------------------------------------------------------

record_ids = [r["id"] for r in records]

assert len(record_ids) == len(set(record_ids)), \
    "Duplicate record IDs found"

# ------------------------------------------------------------
# Frozen scenario split
# ------------------------------------------------------------

train_scenarios = set(split_info["train_scenarios"])
test_scenarios = set(split_info["test_scenarios"])

assert train_scenarios.isdisjoint(test_scenarios)

unknown_scenarios = sorted(
    {
        r["scenario_id"]
        for r in records
        if (
            r["scenario_id"] not in train_scenarios
            and r["scenario_id"] not in test_scenarios
        )
    }
)

assert not unknown_scenarios, (
    f"Scenarios missing from frozen split: "
    f"{unknown_scenarios[:20]}"
)

# ------------------------------------------------------------
# Prompt construction
# ------------------------------------------------------------

def build_prompt(record):

    return (
        "Task:\n"
        f"{record['task_text']}\n\n"
        "State:\n"
        f"{record['state_text']}\n\n"
        "Action:\n"
        f"{record['action_text']}"
    )


def find_exact_action_span(input_ids, action_ids):

    input_ids = list(input_ids)
    action_ids = list(action_ids)

    matches = []

    for start in range(
        len(input_ids) - len(action_ids) + 1
    ):
        if (
            input_ids[
                start:start + len(action_ids)
            ]
            == action_ids
        ):
            matches.append(
                (
                    start,
                    start + len(action_ids) - 1
                )
            )

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one action span; "
            f"found {len(matches)}"
        )

    return matches[0]


# ------------------------------------------------------------
# Build manifest
# ------------------------------------------------------------

manifest_rows = []
failed_rows = []

for row_idx, record in enumerate(records):

    try:

        prompt = build_prompt(record)

        formatted_prompt = tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
        )

        encoded = tokenizer(
            formatted_prompt,
            add_special_tokens=False,
            return_attention_mask=False,
        )

        input_ids = encoded["input_ids"]

        action_ids = tokenizer(
            record["action_text"],
            add_special_tokens=False,
            return_attention_mask=False,
        )["input_ids"]

        action_start, action_end = find_exact_action_span(
            input_ids,
            action_ids,
        )

        prompt_hash = hashlib.sha256(
            formatted_prompt.encode("utf-8")
        ).hexdigest()

        scenario_id = record["scenario_id"]

        if scenario_id in test_scenarios:
            split_name = "test"
        elif scenario_id in train_scenarios:
            split_name = "train"
        else:
            raise ValueError(
                f"Scenario {scenario_id} not in frozen split"
            )

        manifest_rows.append(
            {
                "row_index": row_idx,

                "record_id": record["id"],
                "scenario_id": scenario_id,

                "label": record["label"],
                "action_verb": record["action_verb"],
                "environment": record["environment"],

                "split": split_name,

                "prompt_hash": prompt_hash,

                "sequence_length": len(input_ids),

                "action_start": action_start,
                "action_end": action_end,
                "action_token_count": (
                    action_end - action_start + 1
                ),

                "model": "Qwen/Qwen2.5-7B-Instruct",

                "representation_primary":
                    "last_action_token",

                "representation_ablation":
                    "mean_action_tokens",

                "has_action_call":
                    record.get("action_call") is not None,

                "dataset_version":
                    record["dataset_version"],
            }
        )

    except Exception as e:

        failed_rows.append(
            {
                "row_index": row_idx,
                "record_id": record.get("id"),
                "scenario_id": record.get("scenario_id"),
                "error": repr(e),
            }
        )

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print()
print("=" * 70)
print("MANIFEST VALIDATION")
print("=" * 70)

print("Manifest rows:", len(manifest_rows))
print("Failed rows:", len(failed_rows))

if failed_rows:
    print()
    print("FIRST FAILURES:")
    print(json.dumps(
        failed_rows[:10],
        indent=2
    ))

assert len(failed_rows) == 0

assert len(manifest_rows) == len(records)

# Exact source ordering
row_indices = [
    x["row_index"]
    for x in manifest_rows
]

assert row_indices == list(range(2134))

manifest_ids = [
    x["record_id"]
    for x in manifest_rows
]

assert manifest_ids == record_ids

# Action span validity
for row in manifest_rows:

    assert row["action_start"] >= 0

    assert (
        row["action_end"]
        >= row["action_start"]
    )

    assert (
        row["action_token_count"]
        ==
        row["action_end"]
        - row["action_start"]
        + 1
    )

    assert (
        row["action_end"]
        <
        row["sequence_length"]
    )

# ------------------------------------------------------------
# Split counts
# ------------------------------------------------------------

train_count = sum(
    x["split"] == "train"
    for x in manifest_rows
)

test_count = sum(
    x["split"] == "test"
    for x in manifest_rows
)

print("Train records:", train_count)
print("Test records:", test_count)

assert train_count == 1709
assert test_count == 425

# ------------------------------------------------------------
# Action-call check
# ------------------------------------------------------------

missing_action_call = sum(
    not x["has_action_call"]
    for x in manifest_rows
)

print("Records without action_call:", missing_action_call)

# Expected v2.0 behavior
assert missing_action_call == 5

# ------------------------------------------------------------
# Sequence statistics
# ------------------------------------------------------------

sequence_lengths = [
    x["sequence_length"]
    for x in manifest_rows
]

action_lengths = [
    x["action_token_count"]
    for x in manifest_rows
]

print()
print("Sequence length:")
print("  min:", min(sequence_lengths))
print("  max:", max(sequence_lengths))
print("  mean:", round(
    sum(sequence_lengths) / len(sequence_lengths),
    2
))

print("Action token count:")
print("  min:", min(action_lengths))
print("  max:", max(action_lengths))
print("  mean:", round(
    sum(action_lengths) / len(action_lengths),
    2
))

# ------------------------------------------------------------
# Save JSONL manifest
# ------------------------------------------------------------

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    for row in manifest_rows:

        f.write(
            json.dumps(
                row,
                ensure_ascii=False
            )
            + "\n"
        )

# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------

metadata = {
    "experiment":
        "ADNN_RQ1_Qwen_v2_final",

    "dataset_version":
        "adnn-dataset-2.0",

    "num_records":
        2134,

    "model":
        "Qwen/Qwen2.5-7B-Instruct",

    "hidden_size":
        3584,

    "num_hidden_state_tensors":
        29,

    "num_transformer_layers":
        28,

    "dtype":
        "float16",

    "primary_representation":
        "last_action_token",

    "ablation_representation":
        "mean_action_tokens",

    "prompt_template":
        "Task:\\n{task_text}\\n\\n"
        "State:\\n{state_text}\\n\\n"
        "Action:\\n{action_text}",

    "chat_template":
        "tokenizer.apply_chat_template",

    "add_generation_prompt":
        True,

    "add_special_tokens":
        False,

    "action_span_method":
        "exact contiguous token-sequence matching",

    "split_method":
        split_info["method"],

    "split_group_key":
        split_info["group_key"],

    "split_seed":
        split_info["seed"],

    "train_records":
        train_count,

    "test_records":
        test_count,

    "manifest_file":
        str(MANIFEST_PATH),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Display examples
# ------------------------------------------------------------

print()
print("=" * 70)
print("FIRST MANIFEST ROW")
print("=" * 70)

print(
    json.dumps(
        manifest_rows[0],
        indent=2
    )
)

print()
print("=" * 70)
print("LAST MANIFEST ROW")
print("=" * 70)

print(
    json.dumps(
        manifest_rows[-1],
        indent=2
    )
)

print()
print("=" * 70)
print("SAVED")
print("=" * 70)

print("Manifest:")
print(MANIFEST_PATH)

print()
print("Metadata:")
print(METADATA_PATH)

print()
print("✅ EXTRACTION MANIFEST PASSED ALL CHECKS")

ADNN RQ1 — QWEN v2.0 EXTRACTION MANIFEST
Total records: 2134

MANIFEST VALIDATION
Manifest rows: 2134
Failed rows: 0
Train records: 1709
Test records: 425
Records without action_call: 5

Sequence length:
  min: 161
  max: 331
  mean: 225.37
Action token count:
  min: 2
  max: 42
  mean: 27.87

FIRST MANIFEST ROW
{
  "row_index": 0,
  "record_id": "f01_delete_bin-0000-alt0-single-compact",
  "scenario_id": "grp-99121b548943",
  "label": "partially_reversible",
  "action_verb": "edit_file",
  "environment": "sandbox",
  "split": "test",
  "prompt_hash": "954317b33b8bc18838645b15094cedb177a878bc01cb6fd32edebdf5bd696482",
  "sequence_length": 192,
  "action_start": 153,
  "action_end": 186,
  "action_token_count": 34,
  "model": "Qwen/Qwen2.5-7B-Instruct",
  "representation_primary": "last_action_token",
  "representation_ablation": "mean_action_tokens",
  "has_action_call": true,
  "dataset_version": "adnn-dataset-2.0"
}

LAST MANIFEST ROW
{
  "row_index": 2133,
  "record_id": "v2s3_resto

In [19]:
# ============================================================
# ADNN RQ1 — QWEN v2.0
# STEP 3: EXTRACTION BENCHMARK
# ============================================================

import time
import gc
import numpy as np
import torch

# ------------------------------------------------------------
# Load first 4 manifest rows
# ------------------------------------------------------------

benchmark_rows = manifest_rows[:4]

print("=" * 70)
print("QWEN HIDDEN-STATE EXTRACTION BENCHMARK")
print("=" * 70)

print("Benchmark records:", len(benchmark_rows))

# ------------------------------------------------------------
# Reconstruct prompts
# ------------------------------------------------------------

benchmark_inputs = []
benchmark_action_spans = []

for row in benchmark_rows:

    record = records[row["row_index"]]

    prompt = build_prompt(record)

    formatted_prompt = tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": prompt
            }
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    encoded = tokenizer(
        formatted_prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )

    benchmark_inputs.append(
        encoded["input_ids"][0]
    )

    benchmark_action_spans.append(
        (
            row["action_start"],
            row["action_end"]
        )
    )

# ------------------------------------------------------------
# Pad batch
# ------------------------------------------------------------

from torch.nn.utils.rnn import pad_sequence

input_ids = pad_sequence(
    benchmark_inputs,
    batch_first=True,
    padding_value=tokenizer.pad_token_id,
)

attention_mask = (
    input_ids != tokenizer.pad_token_id
).long()

input_ids = input_ids.to(model.device)
attention_mask = attention_mask.to(model.device)

print("Batch shape:", tuple(input_ids.shape))
print("Sequence lengths:", [
    len(x) for x in benchmark_inputs
])

# ------------------------------------------------------------
# GPU memory before
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.synchronize()

    print()
    print("GPU memory before:")

    for device_idx in range(torch.cuda.device_count()):

        allocated = (
            torch.cuda.memory_allocated(device_idx)
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved(device_idx)
            / 1024**3
        )

        print(
            f"GPU {device_idx}: "
            f"allocated={allocated:.2f} GB, "
            f"reserved={reserved:.2f} GB"
        )

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

print()
print("Running forward pass...")

if torch.cuda.is_available():
    torch.cuda.synchronize()

start_time = time.time()

with torch.inference_mode():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start_time

print(f"Forward-pass time: {elapsed:.2f} seconds")

# ------------------------------------------------------------
# Hidden-state validation
# ------------------------------------------------------------

hidden_states = outputs.hidden_states

print()
print("=" * 70)
print("HIDDEN-STATE OUTPUT")
print("=" * 70)

print("Number of tensors:", len(hidden_states))

assert len(hidden_states) == 29

for layer_idx, h in enumerate(hidden_states):

    assert h.dtype == torch.float16
    assert h.shape[0] == 4
    assert h.shape[2] == 3584

    print(
        f"Layer {layer_idx:2d}: "
        f"shape={tuple(h.shape)}, "
        f"dtype={h.dtype}"
    )

# ------------------------------------------------------------
# Extract representations
# ------------------------------------------------------------

batch_last = []
batch_mean = []

for sample_idx, (action_start, action_end) in enumerate(
    benchmark_action_spans
):

    sample_last = []
    sample_mean = []

    for layer_idx, h in enumerate(hidden_states):

        action_states = h[
            sample_idx,
            action_start:action_end + 1
        ]

        last_vector = action_states[-1]
        mean_vector = action_states.mean(dim=0)

        sample_last.append(
            last_vector.detach().cpu().numpy()
        )

        sample_mean.append(
            mean_vector.detach().cpu().numpy()
        )

    batch_last.append(
        np.stack(sample_last)
    )

    batch_mean.append(
        np.stack(sample_mean)
    )

batch_last = np.stack(batch_last)
batch_mean = np.stack(batch_mean)

# ------------------------------------------------------------
# Validate representations
# ------------------------------------------------------------

print()
print("=" * 70)
print("REPRESENTATIONS")
print("=" * 70)

print(
    "Last-action shape:",
    batch_last.shape
)

print(
    "Mean-action shape:",
    batch_mean.shape
)

print(
    "Last-action dtype:",
    batch_last.dtype
)

print(
    "Mean-action dtype:",
    batch_mean.dtype
)

print(
    "Last-action finite:",
    np.isfinite(batch_last).all()
)

print(
    "Mean-action finite:",
    np.isfinite(batch_mean).all()
)

assert batch_last.shape == (4, 29, 3584)
assert batch_mean.shape == (4, 29, 3584)

assert batch_last.dtype == np.float16
assert batch_mean.dtype == np.float16

assert np.isfinite(batch_last).all()
assert np.isfinite(batch_mean).all()

# ------------------------------------------------------------
# GPU memory after
# ------------------------------------------------------------

if torch.cuda.is_available():

    print()
    print("GPU memory after:")

    for device_idx in range(torch.cuda.device_count()):

        allocated = (
            torch.cuda.memory_allocated(device_idx)
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved(device_idx)
            / 1024**3
        )

        print(
            f"GPU {device_idx}: "
            f"allocated={allocated:.2f} GB, "
            f"reserved={reserved:.2f} GB"
        )

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del outputs
del hidden_states
del input_ids
del attention_mask

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print()
print("=" * 70)
print("BENCHMARK COMPLETE")
print("=" * 70)

print(
    f"Time for 4 records: {elapsed:.2f} seconds"
)

print(
    f"Approx. records/minute: "
    f"{4 / elapsed * 60:.1f}"
)

print()
print("✅ EXTRACTION BENCHMARK PASSED")

QWEN HIDDEN-STATE EXTRACTION BENCHMARK
Benchmark records: 4
Batch shape: (4, 236)
Sequence lengths: [192, 236, 227, 234]

GPU memory before:
GPU 0: allocated=7.14 GB, reserved=7.15 GB
GPU 1: allocated=7.10 GB, reserved=7.11 GB

Running forward pass...
Forward-pass time: 0.91 seconds

HIDDEN-STATE OUTPUT
Number of tensors: 29
Layer  0: shape=(4, 236, 3584), dtype=torch.float16
Layer  1: shape=(4, 236, 3584), dtype=torch.float16
Layer  2: shape=(4, 236, 3584), dtype=torch.float16
Layer  3: shape=(4, 236, 3584), dtype=torch.float16
Layer  4: shape=(4, 236, 3584), dtype=torch.float16
Layer  5: shape=(4, 236, 3584), dtype=torch.float16
Layer  6: shape=(4, 236, 3584), dtype=torch.float16
Layer  7: shape=(4, 236, 3584), dtype=torch.float16
Layer  8: shape=(4, 236, 3584), dtype=torch.float16
Layer  9: shape=(4, 236, 3584), dtype=torch.float16
Layer 10: shape=(4, 236, 3584), dtype=torch.float16
Layer 11: shape=(4, 236, 3584), dtype=torch.float16
Layer 12: shape=(4, 236, 3584), dtype=torch.float

In [20]:
# ============================================================
# ADNN RQ1 — QWEN v2.0
# STEP 4: FULL RESUMABLE HIDDEN-STATE EXTRACTION
# ============================================================

import os
import gc
import json
import time
import hashlib
from pathlib import Path

import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

BATCH_SIZE = 8

N_RECORDS = 2134
N_LAYERS = 29
HIDDEN_SIZE = 3584

WORK_ROOT = Path(
    "/kaggle/working/ADNN_RQ1_Qwen_v2"
)

HIDDEN_DIR = WORK_ROOT / "hidden_states"
HIDDEN_DIR.mkdir(parents=True, exist_ok=True)

LAST_PATH = HIDDEN_DIR / "last_action_token.npy"
MEAN_PATH = HIDDEN_DIR / "mean_action_tokens.npy"
PROGRESS_PATH = HIDDEN_DIR / "extraction_progress.npy"
INDEX_PATH = HIDDEN_DIR / "extraction_index.jsonl"
META_PATH = HIDDEN_DIR / "extraction_metadata.json"

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert len(manifest_rows) == N_RECORDS
assert N_LAYERS == 29
assert HIDDEN_SIZE == 3584

print("=" * 70)
print("ADNN RQ1 — FULL QWEN HIDDEN-STATE EXTRACTION")
print("=" * 70)

print("Records:", N_RECORDS)
print("Layers:", N_LAYERS)
print("Hidden size:", HIDDEN_SIZE)
print("Batch size:", BATCH_SIZE)
print("Representation dtype: float16")

# ------------------------------------------------------------
# Create or resume output arrays
# ------------------------------------------------------------

array_shape = (
    N_RECORDS,
    N_LAYERS,
    HIDDEN_SIZE,
)

array_bytes = (
    N_RECORDS
    * N_LAYERS
    * HIDDEN_SIZE
    * np.dtype(np.float16).itemsize
)

print()
print(
    "Each representation:",
    f"{array_bytes / 1024**3:.2f} GB"
)

print(
    "Both representations:",
    f"{2 * array_bytes / 1024**3:.2f} GB"
)

if LAST_PATH.exists():

    print()
    print("Existing last-action array found.")
    print("Resuming extraction.")

    last_vectors = np.lib.format.open_memmap(
        LAST_PATH,
        mode="r+",
        dtype=np.float16,
        shape=array_shape,
    )

else:

    print()
    print("Creating last-action array...")

    last_vectors = np.lib.format.open_memmap(
        LAST_PATH,
        mode="w+",
        dtype=np.float16,
        shape=array_shape,
    )

if MEAN_PATH.exists():

    print("Existing mean-action array found.")
    print("Resuming extraction.")

    mean_vectors = np.lib.format.open_memmap(
        MEAN_PATH,
        mode="r+",
        dtype=np.float16,
        shape=array_shape,
    )

else:

    print("Creating mean-action array...")

    mean_vectors = np.lib.format.open_memmap(
        MEAN_PATH,
        mode="w+",
        dtype=np.float16,
        shape=array_shape,
    )

# ------------------------------------------------------------
# Progress array
# ------------------------------------------------------------

if PROGRESS_PATH.exists():

    progress = np.load(
        PROGRESS_PATH,
        allow_pickle=False,
    )

    assert progress.shape == (N_RECORDS,)
    assert progress.dtype == np.bool_

    print(
        "Existing progress found:",
        int(progress.sum()),
        "/",
        N_RECORDS,
        "records complete"
    )

else:

    progress = np.zeros(
        N_RECORDS,
        dtype=np.bool_,
    )

    np.save(
        PROGRESS_PATH,
        progress,
        allow_pickle=False,
    )

    print("Created new progress tracker.")

# ------------------------------------------------------------
# Prepare index file for resumability
# ------------------------------------------------------------

if not INDEX_PATH.exists():

    with open(
        INDEX_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        for row in manifest_rows:

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )

    print("Created extraction index.")

else:

    print("Existing extraction index found.")

# ------------------------------------------------------------
# Verify array shapes
# ------------------------------------------------------------

assert last_vectors.shape == array_shape
assert mean_vectors.shape == array_shape

# ------------------------------------------------------------
# Determine work
# ------------------------------------------------------------

remaining = np.where(~progress)[0]

print()
print("=" * 70)
print("EXTRACTION STATUS")
print("=" * 70)

print("Completed:", int(progress.sum()))
print("Remaining:", len(remaining))

if len(remaining) == 0:

    print()
    print("✅ ALL RECORDS ARE ALREADY EXTRACTED.")

else:

    # --------------------------------------------------------
    # Extraction loop
    # --------------------------------------------------------

    extraction_start = time.time()

    processed_this_run = 0

    for batch_start in range(
        0,
        len(remaining),
        BATCH_SIZE,
    ):

        batch_indices = remaining[
            batch_start:
            batch_start + BATCH_SIZE
        ]

        # ----------------------------------------------------
        # Build batch
        # ----------------------------------------------------

        batch_input_ids = []
        batch_spans = []

        for row_idx in batch_indices:

            row = manifest_rows[row_idx]
            record = records[row_idx]

            prompt = build_prompt(record)

            formatted_prompt = tokenizer.apply_chat_template(
                [
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                tokenize=False,
                add_generation_prompt=True,
            )

            encoded = tokenizer(
                formatted_prompt,
                add_special_tokens=False,
                return_tensors="pt",
            )

            batch_input_ids.append(
                encoded["input_ids"][0]
            )

            batch_spans.append(
                (
                    row["action_start"],
                    row["action_end"],
                )
            )

        # ----------------------------------------------------
        # Pad batch
        # ----------------------------------------------------

        batch_input_ids = pad_sequence(
            batch_input_ids,
            batch_first=True,
            padding_value=tokenizer.pad_token_id,
        )

        batch_attention_mask = (
            batch_input_ids
            != tokenizer.pad_token_id
        ).long()

        batch_input_ids = batch_input_ids.to(
            model.device
        )

        batch_attention_mask = batch_attention_mask.to(
            model.device
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with torch.inference_mode():

            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )

        hidden_states = outputs.hidden_states

        # ----------------------------------------------------
        # Extract only action representations
        # ----------------------------------------------------

        for local_idx, row_idx in enumerate(
            batch_indices
        ):

            action_start, action_end = (
                batch_spans[local_idx]
            )

            sample_last = np.empty(
                (N_LAYERS, HIDDEN_SIZE),
                dtype=np.float16,
            )

            sample_mean = np.empty(
                (N_LAYERS, HIDDEN_SIZE),
                dtype=np.float16,
            )

            for layer_idx, h in enumerate(
                hidden_states
            ):

                action_states = h[
                    local_idx,
                    action_start:
                    action_end + 1
                ]

                # Last action token
                sample_last[layer_idx] = (
                    action_states[-1]
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float16)
                )

                # Mean action tokens
                sample_mean[layer_idx] = (
                    action_states.mean(dim=0)
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float16)
                )

            # ------------------------------------------------
            # Numerical validation before writing
            # ------------------------------------------------

            assert np.isfinite(
                sample_last
            ).all()

            assert np.isfinite(
                sample_mean
            ).all()

            assert sample_last.shape == (
                N_LAYERS,
                HIDDEN_SIZE,
            )

            assert sample_mean.shape == (
                N_LAYERS,
                HIDDEN_SIZE,
            )

            # ------------------------------------------------
            # Write permanently
            # ------------------------------------------------

            last_vectors[row_idx] = sample_last
            mean_vectors[row_idx] = sample_mean

            # Flush this row before marking complete.
            last_vectors.flush()
            mean_vectors.flush()

            progress[row_idx] = True

            np.save(
                PROGRESS_PATH,
                progress,
                allow_pickle=False,
            )

            processed_this_run += 1

        # ----------------------------------------------------
        # Cleanup batch
        # ----------------------------------------------------

        del outputs
        del hidden_states
        del batch_input_ids
        del batch_attention_mask

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ----------------------------------------------------
        # Progress report
        # ----------------------------------------------------

        completed = int(progress.sum())

        elapsed = time.time() - extraction_start

        rate = (
            processed_this_run / elapsed
            if elapsed > 0
            else 0
        )

        remaining_count = N_RECORDS - completed

        eta_seconds = (
            remaining_count / rate
            if rate > 0
            else float("nan")
        )

        print(
            f"[{completed:4d}/{N_RECORDS}] "
            f"batch={len(batch_indices):2d} | "
            f"rate={rate:.1f} rec/s | "
            f"ETA={eta_seconds / 60:.1f} min"
        )

    # --------------------------------------------------------
    # Final flush
    # --------------------------------------------------------

    last_vectors.flush()
    mean_vectors.flush()

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL EXTRACTION VALIDATION")
print("=" * 70)

completed_count = int(progress.sum())

print(
    "Completed records:",
    completed_count,
    "/",
    N_RECORDS
)

assert completed_count == N_RECORDS

print(
    "Last-action array:",
    last_vectors.shape,
    last_vectors.dtype,
)

print(
    "Mean-action array:",
    mean_vectors.shape,
    mean_vectors.dtype,
)

assert last_vectors.shape == array_shape
assert mean_vectors.shape == array_shape

assert last_vectors.dtype == np.float16
assert mean_vectors.dtype == np.float16

# Check representative rows
check_indices = [
    0,
    N_RECORDS // 2,
    N_RECORDS - 1,
]

for idx in check_indices:

    assert np.isfinite(
        last_vectors[idx]
    ).all()

    assert np.isfinite(
        mean_vectors[idx]
    ).all()

print()
print("Representative rows are finite.")

# ------------------------------------------------------------
# Save final metadata
# ------------------------------------------------------------

final_metadata = {
    "experiment":
        "ADNN_RQ1_Qwen_v2_final",

    "dataset_version":
        "adnn-dataset-2.0",

    "model":
        "Qwen/Qwen2.5-7B-Instruct",

    "num_records":
        N_RECORDS,

    "num_hidden_state_tensors":
        N_LAYERS,

    "hidden_size":
        HIDDEN_SIZE,

    "dtype":
        "float16",

    "batch_size":
        BATCH_SIZE,

    "primary_representation":
        "last_action_token",

    "ablation_representation":
        "mean_action_tokens",

    "last_action_file":
        str(LAST_PATH),

    "mean_action_file":
        str(MEAN_PATH),

    "progress_file":
        str(PROGRESS_PATH),

    "index_file":
        str(INDEX_PATH),

    "manifest_file":
        str(HIDDEN_DIR / "extraction_manifest.jsonl"),

    "all_records_completed":
        True,
}

with open(
    META_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_metadata,
        f,
        indent=2,
    )

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del last_vectors
del mean_vectors

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print()
print("=" * 70)
print("FILES")
print("=" * 70)

print("Last:", LAST_PATH)
print("Mean:", MEAN_PATH)
print("Progress:", PROGRESS_PATH)
print("Index:", INDEX_PATH)
print("Metadata:", META_PATH)

print()
print("🎉 FULL QWEN HIDDEN-STATE EXTRACTION COMPLETE")

ADNN RQ1 — FULL QWEN HIDDEN-STATE EXTRACTION
Records: 2134
Layers: 29
Hidden size: 3584
Batch size: 8
Representation dtype: float16

Each representation: 0.41 GB
Both representations: 0.83 GB

Creating last-action array...
Creating mean-action array...
Created new progress tracker.
Created extraction index.

EXTRACTION STATUS
Completed: 0
Remaining: 2134
[   8/2134] batch= 8 | rate=3.9 rec/s | ETA=9.0 min
[  16/2134] batch= 8 | rate=4.0 rec/s | ETA=8.7 min
[  24/2134] batch= 8 | rate=4.0 rec/s | ETA=8.7 min
[  32/2134] batch= 8 | rate=4.0 rec/s | ETA=8.7 min
[  40/2134] batch= 8 | rate=4.1 rec/s | ETA=8.6 min
[  48/2134] batch= 8 | rate=4.1 rec/s | ETA=8.6 min
[  56/2134] batch= 8 | rate=4.1 rec/s | ETA=8.5 min
[  64/2134] batch= 8 | rate=4.1 rec/s | ETA=8.5 min
[  72/2134] batch= 8 | rate=4.1 rec/s | ETA=8.5 min
[  80/2134] batch= 8 | rate=4.1 rec/s | ETA=8.5 min
[  88/2134] batch= 8 | rate=4.1 rec/s | ETA=8.4 min
[  96/2134] batch= 8 | rate=4.0 rec/s | ETA=8.4 min
[ 104/2134] batch= 

In [21]:
# ============================================================
# ADNN RQ1 — FROZEN SPLIT DIAGNOSTIC
# ============================================================

print("=" * 70)
print("FROZEN SPLIT DIAGNOSTIC")
print("=" * 70)

dataset_scenarios = set(
    record["scenario_id"]
    for record in records
)

train_scenarios = set(
    split_info["train_scenarios"]
)

test_scenarios = set(
    split_info["test_scenarios"]
)

# ------------------------------------------------------------
# Differences
# ------------------------------------------------------------

train_not_in_dataset = sorted(
    train_scenarios - dataset_scenarios
)

test_not_in_dataset = sorted(
    test_scenarios - dataset_scenarios
)

dataset_not_in_split = sorted(
    dataset_scenarios -
    (train_scenarios | test_scenarios)
)

print("\nDataset scenarios:", len(dataset_scenarios))
print("Train scenarios :", len(train_scenarios))
print("Test scenarios  :", len(test_scenarios))

print("\n" + "-" * 70)
print("TRAIN IDs NOT PRESENT IN DATASET")
print("-" * 70)

print(train_not_in_dataset)

print("\n" + "-" * 70)
print("TEST IDs NOT PRESENT IN DATASET")
print("-" * 70)

print(test_not_in_dataset)

print("\n" + "-" * 70)
print("DATASET IDs NOT PRESENT IN SPLIT")
print("-" * 70)

print(dataset_not_in_split)


# ------------------------------------------------------------
# Inspect records belonging to any unmatched dataset scenario
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNMATCHED DATASET RECORDS")
print("=" * 70)

if dataset_not_in_split:

    for sid in dataset_not_in_split:

        matching = [
            r for r in records
            if r["scenario_id"] == sid
        ]

        print(
            f"\nScenario: {sid}"
        )

        print(
            "Number of records:",
            len(matching)
        )

        for r in matching[:10]:

            print(
                "  ID:",
                r["id"]
            )

            print(
                "  Environment:",
                r["environment"]
            )

            print(
                "  Label:",
                r["label"]
            )

            print(
                "  Source:",
                r["source"]
            )


# ------------------------------------------------------------
# Inspect unmatched split IDs
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNMATCHED SPLIT RECORDS")
print("=" * 70)

for sid in train_not_in_dataset:

    print(
        "Train-only split ID:",
        sid
    )

for sid in test_not_in_dataset:

    print(
        "Test-only split ID:",
        sid
    )


# ------------------------------------------------------------
# Check ScienceWorld explicitly
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SCIENCEWORLD CHECK")
print("=" * 70)

scienceworld_records = [
    r for r in records
    if str(r.get("environment", "")).lower()
    == "scienceworld"
]

print(
    "ScienceWorld records:",
    len(scienceworld_records)
)

scienceworld_scenarios = sorted(
    set(r["scenario_id"] for r in scienceworld_records)
)

print(
    "ScienceWorld scenarios:",
    scienceworld_scenarios
)

for sid in scienceworld_scenarios:

    if sid in train_scenarios:
        split_name = "TRAIN"
    elif sid in test_scenarios:
        split_name = "TEST"
    else:
        split_name = "NOT IN SPLIT"

    print(
        f"{sid}: {split_name}"
    )


print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

print(
    "Do NOT change the split yet."
)
print(
    "We will use the result to determine the correct frozen-split handling."
)

FROZEN SPLIT DIAGNOSTIC

Dataset scenarios: 149
Train scenarios : 124
Test scenarios  : 27

----------------------------------------------------------------------
TRAIN IDs NOT PRESENT IN DATASET
----------------------------------------------------------------------
[]

----------------------------------------------------------------------
TEST IDs NOT PRESENT IN DATASET
----------------------------------------------------------------------
['grp-0cc4fd8f73a6', 'grp-f8673bb166d9']

----------------------------------------------------------------------
DATASET IDs NOT PRESENT IN SPLIT
----------------------------------------------------------------------
[]

UNMATCHED DATASET RECORDS

UNMATCHED SPLIT RECORDS
Test-only split ID: grp-0cc4fd8f73a6
Test-only split ID: grp-f8673bb166d9

SCIENCEWORLD CHECK
ScienceWorld records: 5
ScienceWorld scenarios: ['sw-grp-boil-kitchen', 'sw-grp-power-workshop']
sw-grp-boil-kitchen: TRAIN
sw-grp-power-workshop: TEST

DIAGNOSTIC COMPLETE
Do NOT change th

In [23]:
# ============================================================
# RESTORE SCENARIO IDS FOR PROBE SPLIT
# ============================================================

import numpy as np

scenario_ids = np.array(
    [record["scenario_id"] for record in records],
    dtype=object
)

print("=" * 70)
print("SCENARIO IDS RESTORED")
print("=" * 70)

print("Total scenario IDs:", len(scenario_ids))
print("Unique scenarios:", len(set(scenario_ids)))

print("First scenario ID:", scenario_ids[0])
print("Last scenario ID:", scenario_ids[-1])

assert len(scenario_ids) == len(records)
assert scenario_ids[0] == records[0]["scenario_id"]

print("\n✓ scenario_ids aligned with records")

SCENARIO IDS RESTORED
Total scenario IDs: 2134
Unique scenarios: 149
First scenario ID: grp-99121b548943
Last scenario ID: grp-bf1a94f1711e

✓ scenario_ids aligned with records


In [28]:
# ============================================================
# RESTORE ADNN WORKSPACE PATH
# ============================================================

from pathlib import Path
import os

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"

os.makedirs(
    os.path.join(WORK_DIR, "probes"),
    exist_ok=True
)

print("=" * 70)
print("WORKSPACE RESTORED")
print("=" * 70)

print("WORK_DIR:", WORK_DIR)
print("Exists:", os.path.exists(WORK_DIR))
print("Probe directory:", os.path.exists(
    os.path.join(WORK_DIR, "probes")
))

WORKSPACE RESTORED
WORK_DIR: /kaggle/working/ADNN_RQ1_Qwen_v2
Exists: True
Probe directory: True


In [26]:
# ============================================================
# RESTORE EXTRACTION INDEX FROM SAVED FILE
# ============================================================

import json

index_path = "/kaggle/working/ADNN_RQ1_Qwen_v2/hidden_states/extraction_index.jsonl"

extraction_index = []

with open(index_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            extraction_index.append(json.loads(line))

print("=" * 70)
print("EXTRACTION INDEX RESTORED")
print("=" * 70)

print("Entries:", len(extraction_index))

if extraction_index:
    print("\nFirst entry:")
    print(extraction_index[0])

    print("\nLast entry:")
    print(extraction_index[-1])

assert len(extraction_index) == len(records)

assert extraction_index[0]["record_id"] == records[0]["id"]
assert extraction_index[-1]["record_id"] == records[-1]["id"]

print("\n✓ Extraction index length matches records")
print("✓ First record aligned")
print("✓ Last record aligned")

EXTRACTION INDEX RESTORED
Entries: 2134

First entry:
{'row_index': 0, 'record_id': 'f01_delete_bin-0000-alt0-single-compact', 'scenario_id': 'grp-99121b548943', 'label': 'partially_reversible', 'action_verb': 'edit_file', 'environment': 'sandbox', 'split': 'test', 'prompt_hash': '954317b33b8bc18838645b15094cedb177a878bc01cb6fd32edebdf5bd696482', 'sequence_length': 192, 'action_start': 153, 'action_end': 186, 'action_token_count': 34, 'model': 'Qwen/Qwen2.5-7B-Instruct', 'representation_primary': 'last_action_token', 'representation_ablation': 'mean_action_tokens', 'has_action_call': True, 'dataset_version': 'adnn-dataset-2.0'}

Last entry:
{'row_index': 2133, 'record_id': 'v2s3_restore_from_bin-0199-single-single-prose', 'scenario_id': 'grp-bf1a94f1711e', 'label': 'partially_reversible', 'action_verb': 'restore_file', 'environment': 'sandbox', 'split': 'train', 'prompt_hash': 'c47289fa3412068b42c59b08cc7bc5b5714bdb35c3f8d10821f57b0f1fbd8165', 'sequence_length': 224, 'action_start': 19

In [29]:
# ============================================================
# ADNN RQ1 — FINAL PROBE SPLIT CONSTRUCTION
# ============================================================

import os
import json
import numpy as np
from collections import Counter

print("=" * 70)
print("FINAL PROBE SPLIT CONSTRUCTION")
print("=" * 70)

# ------------------------------------------------------------
# Frozen scenario sets from official split.json
# ------------------------------------------------------------

train_scenarios = set(
    split_info["train_scenarios"]
)

test_scenarios = set(
    split_info["test_scenarios"]
)

# Actual scenarios present in records
dataset_scenarios = set(
    record["scenario_id"]
    for record in records
)

# ------------------------------------------------------------
# Verify actual dataset scenarios are fully covered
# ------------------------------------------------------------

assert dataset_scenarios <= (
    train_scenarios | test_scenarios
)

assert train_scenarios.isdisjoint(
    test_scenarios
)

print("✓ All dataset scenarios covered")
print("✓ No train/test scenario overlap")

# ------------------------------------------------------------
# Derive record-level split
# ------------------------------------------------------------

record_split = np.array(
    [
        "train"
        if record["scenario_id"] in train_scenarios
        else "test"
        for record in records
    ],
    dtype=object
)

train_mask = record_split == "train"
test_mask = record_split == "test"

print("\nTrain records:", int(train_mask.sum()))
print("Test records :", int(test_mask.sum()))

assert train_mask.sum() == 1709
assert test_mask.sum() == 425

# ------------------------------------------------------------
# Label mapping
# ------------------------------------------------------------

LABEL_TO_ID = {
    "reversible": 0,
    "partially_reversible": 1,
    "irreversible": 2,
}

ID_TO_LABEL = {
    v: k
    for k, v in LABEL_TO_ID.items()
}

y = np.array(
    [
        LABEL_TO_ID[record["label"]]
        for record in records
    ],
    dtype=np.int64
)

# ------------------------------------------------------------
# Print train/test class distributions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS DISTRIBUTIONS")
print("=" * 70)

for split_name, mask in [
    ("TRAIN", train_mask),
    ("TEST", test_mask),
]:

    print(f"\n{split_name}")

    counts = Counter(
        y[mask].tolist()
    )

    for label_id in range(3):

        print(
            f"  {ID_TO_LABEL[label_id]:25s}: "
            f"{counts.get(label_id, 0):4d}"
        )

# ------------------------------------------------------------
# Scenario counts represented in records
# ------------------------------------------------------------

train_record_scenarios = set(
    scenario_ids[train_mask]
)

test_record_scenarios = set(
    scenario_ids[test_mask]
)

print("\n" + "=" * 70)
print("SCENARIO COUNTS USED BY PROBE")
print("=" * 70)

print(
    "Train scenarios represented:",
    len(train_record_scenarios)
)

print(
    "Test scenarios represented:",
    len(test_record_scenarios)
)

assert len(train_record_scenarios) == 124
assert len(test_record_scenarios) == 25

# ------------------------------------------------------------
# Identify metadata-only test scenarios
# ------------------------------------------------------------

metadata_only_test = sorted(
    test_scenarios - dataset_scenarios
)

print("\nMetadata-only test scenario IDs:")
for sid in metadata_only_test:
    print(" ", sid)

assert len(metadata_only_test) == 2

# ------------------------------------------------------------
# Verify extraction alignment
# ------------------------------------------------------------

for i in range(len(records)):

    assert (
        extraction_index[i]["record_id"]
        == records[i]["id"]
    )

    assert (
        extraction_index[i]["scenario_id"]
        == records[i]["scenario_id"]
    )

print("\n✓ Extraction index aligned with probe records")

# ------------------------------------------------------------
# Save probe split metadata
# ------------------------------------------------------------

probe_dir = os.path.join(
    WORK_DIR,
    "probes"
)

os.makedirs(
    probe_dir,
    exist_ok=True
)

probe_split = {
    "dataset_version": "adnn-dataset-2.0",
    "model": "Qwen/Qwen2.5-7B-Instruct",

    "split_source": "official split.json",
    "split_type": "scenario_level",

    "total_records": int(len(records)),

    "train_records": int(train_mask.sum()),
    "test_records": int(test_mask.sum()),

    "train_scenarios_in_records":
        len(train_record_scenarios),

    "test_scenarios_in_records":
        len(test_record_scenarios),

    "official_train_scenarios":
        len(train_scenarios),

    "official_test_scenarios":
        len(test_scenarios),

    "metadata_only_test_scenarios":
        metadata_only_test,

    "label_mapping":
        LABEL_TO_ID,

    "representations": [
        "last_action_token",
        "mean_action_tokens"
    ]
}

split_output = os.path.join(
    probe_dir,
    "probe_split.json"
)

with open(
    split_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        probe_split,
        f,
        indent=2
    )

print("\nSaved:")
print(split_output)

print("\n" + "=" * 70)
print("✓ FINAL PROBE SPLIT READY")
print("=" * 70)

FINAL PROBE SPLIT CONSTRUCTION
✓ All dataset scenarios covered
✓ No train/test scenario overlap

Train records: 1709
Test records : 425

CLASS DISTRIBUTIONS

TRAIN
  reversible               :  731
  partially_reversible     :  554
  irreversible             :  424

TEST
  reversible               :   68
  partially_reversible     :  204
  irreversible             :  153

SCENARIO COUNTS USED BY PROBE
Train scenarios represented: 124
Test scenarios represented: 25

Metadata-only test scenario IDs:
  grp-0cc4fd8f73a6
  grp-f8673bb166d9

✓ Extraction index aligned with probe records

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/probes/probe_split.json

✓ FINAL PROBE SPLIT READY


In [31]:
# ============================================================
# RESTORE QWEN HIDDEN STATES + PROBE VARIABLES
# ============================================================

import numpy as np

HIDDEN_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2/hidden_states"

last_path = f"{HIDDEN_DIR}/last_action_token.npy"
mean_path = f"{HIDDEN_DIR}/mean_action_tokens.npy"

print("=" * 70)
print("CHECKING SAVED HIDDEN STATES")
print("=" * 70)

print("Last-action file exists:", __import__("os").path.exists(last_path))
print("Mean-action file exists:", __import__("os").path.exists(mean_path))

# Load primary representation
X_all = np.load(
    last_path,
    mmap_mode="r"
)

print("\nX_all shape:", X_all.shape)
print("X_all dtype:", X_all.dtype)

# Verify expected dimensions
assert X_all.shape == (2134, 29, 3584)
assert X_all.dtype == np.float16

# ------------------------------------------------------------
# Restore probe indices
# ------------------------------------------------------------

train_idx = np.where(train_mask)[0]
test_idx = np.where(test_mask)[0]

y_train = y[train_idx]
y_test = y[test_idx]

groups_train = scenario_ids[train_idx]
groups_test = scenario_ids[test_idx]

print("\nTrain indices:", len(train_idx))
print("Test indices :", len(test_idx))

print("Train groups:", len(set(groups_train)))
print("Test groups :", len(set(groups_test)))

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(train_idx) == 1709
assert len(test_idx) == 425
assert X_all.shape[0] == len(records)

assert np.isfinite(
    np.asarray(X_all[0], dtype=np.float32)
).all()

print("\n✓ Hidden states loaded")
print("✓ Shape verified: (2134, 29, 3584)")
print("✓ dtype verified: float16")
print("✓ Train/test indices restored")
print("✓ Scenario groups restored")
print("✓ Finite-value check passed")

CHECKING SAVED HIDDEN STATES
Last-action file exists: True
Mean-action file exists: True

X_all shape: (2134, 29, 3584)
X_all dtype: float16

Train indices: 1709
Test indices : 425
Train groups: 124
Test groups : 25

✓ Hidden states loaded
✓ Shape verified: (2134, 29, 3584)
✓ dtype verified: float16
✓ Train/test indices restored
✓ Scenario groups restored
✓ Finite-value check passed


In [32]:
import time
import warnings
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")

C = 0.001
N_SPLITS = 5
N_LAYERS = X_all.shape[1]

results_screen = []

print("=" * 70)
print("Qwen v2.0 — Layer Screening")
print("=" * 70)
print(f"Layers: {N_LAYERS}")
print(f"C: {C}")
print(f"CV: {N_SPLITS}-fold GroupKFold")
print(f"Training records: {len(train_idx)}")
print()

overall_start = time.time()

for layer in range(N_LAYERS):

    layer_start = time.time()

    # Load one layer
    X = np.asarray(
        X_all[train_idx, layer, :],
        dtype=np.float32
    )

    y = y_train
    groups = groups_train

    gkf = GroupKFold(n_splits=N_SPLITS)

    fold_scores = []

    # 5-fold GroupKFold
    for fold, (tr, va) in enumerate(
        gkf.split(X, y, groups),
        start=1
    ):

        # Fit scaler ONLY on training fold
        scaler = StandardScaler()

        Xtr = scaler.fit_transform(X[tr])
        Xva = scaler.transform(X[va])

        clf = LogisticRegression(
            C=C,
            penalty="l2",
            solver="saga",
            max_iter=500,
            tol=1e-2,
            class_weight="balanced",
            multi_class="multinomial",
            random_state=0
        )

        clf.fit(Xtr, y[tr])

        pred = clf.predict(Xva)

        ba = balanced_accuracy_score(y[va], pred)

        fold_scores.append(ba)

    mean_ba = float(np.mean(fold_scores))
    std_ba = float(np.std(fold_scores))
    elapsed = time.time() - layer_start

    results_screen.append({
        "layer": layer,
        "C": C,
        "cv_balanced_accuracy": mean_ba,
        "cv_std": std_ba,
        "fold_scores": fold_scores,
        "time_sec": elapsed
    })

    total_elapsed = time.time() - overall_start
    avg_layer_time = total_elapsed / (layer + 1)
    remaining = avg_layer_time * (N_LAYERS - layer - 1)

    print(
        f"Layer {layer:2d} | "
        f"BA={mean_ba:.4f} | "
        f"SD={std_ba:.4f} | "
        f"time={elapsed:6.1f}s | "
        f"ETA={remaining/60:5.1f} min"
    )

print()
print("=" * 70)
print("SCREENING COMPLETE")
print("=" * 70)

# Sort by CV BA
ranked_screen = sorted(
    results_screen,
    key=lambda x: x["cv_balanced_accuracy"],
    reverse=True
)

print("\nTop 10 layers:")
print("-" * 70)

for rank, r in enumerate(ranked_screen[:10], start=1):
    print(
        f"{rank:2d}. Layer {r['layer']:2d} | "
        f"BA={r['cv_balanced_accuracy']:.4f} | "
        f"SD={r['cv_std']:.4f}"
    )

print(
    f"\nTotal time: {(time.time()-overall_start)/60:.1f} min"
)

Qwen v2.0 — Layer Screening
Layers: 29
C: 0.001
CV: 5-fold GroupKFold
Training records: 1709

Layer  0 | BA=0.3333 | SD=0.0000 | time=   9.0s | ETA=  4.2 min
Layer  1 | BA=0.6158 | SD=0.0376 | time=  22.6s | ETA=  7.1 min
Layer  2 | BA=0.6150 | SD=0.0344 | time=  32.2s | ETA=  9.2 min
Layer  3 | BA=0.6382 | SD=0.0443 | time=  33.7s | ETA= 10.1 min
Layer  4 | BA=0.6588 | SD=0.0414 | time=  31.6s | ETA= 10.3 min
Layer  5 | BA=0.7231 | SD=0.0254 | time=  28.0s | ETA= 10.0 min
Layer  6 | BA=0.7442 | SD=0.0284 | time=  26.2s | ETA=  9.6 min
Layer  7 | BA=0.7540 | SD=0.0463 | time=  23.5s | ETA=  9.0 min
Layer  8 | BA=0.7792 | SD=0.0485 | time=  22.4s | ETA=  8.5 min
Layer  9 | BA=0.7984 | SD=0.0517 | time=  21.0s | ETA=  7.9 min
Layer 10 | BA=0.8216 | SD=0.0415 | time=  17.8s | ETA=  7.3 min
Layer 11 | BA=0.8190 | SD=0.0415 | time=  18.0s | ETA=  6.8 min
Layer 12 | BA=0.8189 | SD=0.0446 | time=  13.9s | ETA=  6.2 min
Layer 13 | BA=0.8164 | SD=0.0516 | time=  13.7s | ETA=  5.6 min
Layer 14 |

In [33]:
import json
from pathlib import Path

screen_path = Path(
    "/kaggle/working/ADNN_RQ1_Qwen_v2/probes/"
    "qwen_v2_3class_layer_screening.json"
)

screen_path.parent.mkdir(parents=True, exist_ok=True)

with open(screen_path, "w") as f:
    json.dump(results_screen, f, indent=2)

print("Saved:")
print(screen_path)

print("\nBest screening layer:")
best = ranked_screen[0]
print(
    f"Layer {best['layer']} | "
    f"C={best['C']} | "
    f"CV BA={best['cv_balanced_accuracy']:.4f}"
)

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/probes/qwen_v2_3class_layer_screening.json

Best screening layer:
Layer 19 | C=0.001 | CV BA=0.8767


In [47]:
import time
import warnings
import json
import numpy as np

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

TOP_LAYERS = list(range(16, 26))
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
N_SPLITS = 5

probe_dir = Path("/kaggle/working/ADNN_RQ1_Qwen_v2/probes")
probe_dir.mkdir(parents=True, exist_ok=True)

selection_results = []

overall_start = time.time()

print("=" * 70)
print("Qwen v2.0 — Inner CV C Selection")
print("=" * 70)
print(f"Layers: {TOP_LAYERS}")
print(f"C grid: {C_GRID}")
print(f"CV: {N_SPLITS}-fold GroupKFold")
print("Test set: NOT USED")
print()

# ---------------------------------------------------------
# Run layer × C CV
# ---------------------------------------------------------

for layer in TOP_LAYERS:

    layer_start = time.time()

    print(f"\n{'-' * 70}")
    print(f"Layer {layer}")
    print(f"{'-' * 70}")

    # Load layer
    X = np.asarray(
        X_all[train_idx, layer, :],
        dtype=np.float32
    )

    y = y_train
    groups = groups_train

    # Create folds once
    gkf = GroupKFold(n_splits=N_SPLITS)

    folds = list(gkf.split(X, y, groups))

    # -----------------------------------------------------
    # Scale once per fold
    # -----------------------------------------------------

    fold_data = []

    scale_start = time.time()

    for fold, (tr, va) in enumerate(folds):

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(X[tr])
        Xva = scaler.transform(X[va])

        fold_data.append(
            (
                Xtr,
                Xva,
                y[tr],
                y[va]
            )
        )

    scale_time = time.time() - scale_start

    # -----------------------------------------------------
    # C search
    # -----------------------------------------------------

    layer_results = []

    for C in C_GRID:

        c_start = time.time()
        fold_scores = []

        for Xtr, Xva, ytr, yva in fold_data:

            clf = LogisticRegression(
                C=C,
                penalty="l2",
                solver="saga",
                max_iter=500,
                tol=1e-2,
                class_weight="balanced",
                multi_class="multinomial",
                random_state=0
            )

            clf.fit(Xtr, ytr)

            pred = clf.predict(Xva)

            ba = balanced_accuracy_score(
                yva,
                pred
            )

            fold_scores.append(float(ba))

        mean_ba = float(np.mean(fold_scores))
        std_ba = float(np.std(fold_scores))

        result = {
            "layer": layer,
            "C": C,
            "cv_balanced_accuracy": mean_ba,
            "cv_std": std_ba,
            "fold_scores": fold_scores
        }

        layer_results.append(result)
        selection_results.append(result)

        print(
            f"C={C:<6} | "
            f"BA={mean_ba:.4f} | "
            f"SD={std_ba:.4f} | "
            f"time={time.time()-c_start:.1f}s"
        )

    # Best C for this layer
    best_layer_result = max(
        layer_results,
        key=lambda x: x["cv_balanced_accuracy"]
    )

    print(
        f"BEST Layer {layer}: "
        f"C={best_layer_result['C']} | "
        f"BA={best_layer_result['cv_balanced_accuracy']:.4f}"
    )

    print(
        f"Layer time: {(time.time()-layer_start)/60:.1f} min"
    )

    # Save incrementally
    with open(
        probe_dir / "qwen_v2_3class_inner_cv_progress.json",
        "w"
    ) as f:
        json.dump(selection_results, f, indent=2)

# ---------------------------------------------------------
# Final ranking
# ---------------------------------------------------------

ranked_selection = sorted(
    selection_results,
    key=lambda x: x["cv_balanced_accuracy"],
    reverse=True
)

print("\n")
print("=" * 70)
print("INNER CV COMPLETE")
print("=" * 70)

print("\nTop 10 layer/C combinations:")
print("-" * 70)

for rank, r in enumerate(
    ranked_selection[:10],
    start=1
):
    print(
        f"{rank:2d}. "
        f"Layer {r['layer']:2d} | "
        f"C={r['C']:<6} | "
        f"BA={r['cv_balanced_accuracy']:.4f} | "
        f"SD={r['cv_std']:.4f}"
    )

best_overall = ranked_selection[0]

print("\nSelected by inner CV:")
print(
    f"Layer = {best_overall['layer']}"
)
print(
    f"C = {best_overall['C']}"
)
print(
    f"CV balanced accuracy = "
    f"{best_overall['cv_balanced_accuracy']:.4f}"
)

print(
    f"\nTotal time: "
    f"{(time.time()-overall_start)/60:.1f} min"
)

Qwen v2.0 — Inner CV C Selection
Layers: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
C grid: [0.001, 0.01, 0.1, 1.0, 10.0]
CV: 5-fold GroupKFold
Test set: NOT USED


----------------------------------------------------------------------
Layer 16
----------------------------------------------------------------------
C=0.001  | BA=0.8458 | SD=0.0425 | time=12.5s
C=0.01   | BA=0.8631 | SD=0.0421 | time=24.2s
C=0.1    | BA=0.8682 | SD=0.0406 | time=29.5s
C=1.0    | BA=0.8676 | SD=0.0415 | time=30.1s
C=10.0   | BA=0.8676 | SD=0.0415 | time=30.3s
BEST Layer 16: C=0.1 | BA=0.8682
Layer time: 2.1 min

----------------------------------------------------------------------
Layer 17
----------------------------------------------------------------------
C=0.001  | BA=0.8461 | SD=0.0548 | time=12.6s
C=0.01   | BA=0.8731 | SD=0.0395 | time=24.1s
C=0.1    | BA=0.8781 | SD=0.0343 | time=29.1s
C=1.0    | BA=0.8804 | SD=0.0317 | time=30.4s
C=10.0   | BA=0.8804 | SD=0.0317 | time=30.0s
BEST Layer 17: C=1.0 

In [34]:
import json
import warnings
import numpy as np

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    classification_report
)

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# FINAL SELECTED CONFIGURATION
# ---------------------------------------------------------

BEST_LAYER = 20
BEST_C = 0.1

CLASS_NAMES = [
    "reversible",
    "partially_reversible",
    "irreversible"
]

print("=" * 70)
print("Qwen v2.0 — FINAL 3-CLASS TEST EVALUATION")
print("=" * 70)

print(f"Selected layer: {BEST_LAYER}")
print(f"Selected C: {BEST_C}")
print(f"Training records: {len(train_idx)}")
print(f"Test records: {len(test_idx)}")
print("Test set used ONLY for final evaluation")
print()

# ---------------------------------------------------------
# Load selected representation
# ---------------------------------------------------------

X_train = np.asarray(
    X_all[train_idx, BEST_LAYER, :],
    dtype=np.float32
)

X_test = np.asarray(
    X_all[test_idx, BEST_LAYER, :],
    dtype=np.float32
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# ---------------------------------------------------------
# Fit scaler ONLY on full training set
# ---------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# Fit final classifier
# ---------------------------------------------------------

clf = LogisticRegression(
    C=BEST_C,
    penalty="l2",
    solver="saga",
    max_iter=500,
    tol=1e-2,
    class_weight="balanced",
    multi_class="multinomial",
    random_state=0
)

clf.fit(X_train_scaled, y_train)

# ---------------------------------------------------------
# Test predictions
# ---------------------------------------------------------

y_pred = clf.predict(X_test_scaled)
y_prob = clf.predict_proba(X_test_scaled)

# ---------------------------------------------------------
# Metrics
# ---------------------------------------------------------

test_ba = balanced_accuracy_score(
    y_test,
    y_pred
)

test_macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

precision, recall, f1, support = precision_recall_fscore_support(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    zero_division=0
)

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1, 2]
)

# Multiclass one-vs-rest AUROC
try:
    test_auroc = roc_auc_score(
        y_test,
        y_prob,
        multi_class="ovr",
        average="macro"
    )
except Exception:
    test_auroc = None

# ---------------------------------------------------------
# Print results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Balanced Accuracy : {test_ba:.4f}")
print(f"Macro-F1          : {test_macro_f1:.4f}")

if test_auroc is not None:
    print(f"Macro AUROC       : {test_auroc:.4f}")
else:
    print("Macro AUROC       : unavailable")

print("\nPer-class metrics:")
print("-" * 70)

for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name:20s} | "
        f"Precision={precision[i]:.4f} | "
        f"Recall={recall[i]:.4f} | "
        f"F1={f1[i]:.4f} | "
        f"Support={support[i]}"
    )

print("\nConfusion Matrix")
print("(rows = true, columns = predicted)")
print("-" * 70)
print("                 " + "  ".join(
    f"{x[:8]:>10s}" for x in CLASS_NAMES
))
print()

for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name[:16]:16s} "
        + "  ".join(f"{x:10d}" for x in cm[i])
    )

# ---------------------------------------------------------
# Save predictions
# ---------------------------------------------------------

probe_dir = Path(
    "/kaggle/working/ADNN_RQ1_Qwen_v2/probes"
)

pred_path = probe_dir / "qwen_v2_3class_test_predictions.jsonl"

with open(pred_path, "w") as f:

    for row_idx, original_idx in enumerate(test_idx):

        f.write(json.dumps({
            "test_row": int(row_idx),
            "dataset_row_index": int(original_idx),
            "record_id": str(
                extraction_index[original_idx]["record_id"]
            ),
            "scenario_id": str(
                extraction_index[original_idx]["scenario_id"]
            ),
            "true_label": CLASS_NAMES[int(y_test[row_idx])],
            "predicted_label": CLASS_NAMES[int(y_pred[row_idx])],
            "prob_reversible": float(y_prob[row_idx, 0]),
            "prob_partially_reversible": float(y_prob[row_idx, 1]),
            "prob_irreversible": float(y_prob[row_idx, 2])
        }) + "\n")

# ---------------------------------------------------------
# Save metrics
# ---------------------------------------------------------

metrics = {
    "experiment": "ADNN_RQ1_Qwen_v2_final",
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "representation": "last_action_token",
    "task": "three_class",
    "layer_selected": BEST_LAYER,
    "C_selected": BEST_C,
    "selection_method": "inner GroupKFold CV",
    "n_inner_folds": 5,
    "class_weight": "balanced",
    "solver": "saga",
    "penalty": "l2",
    "tol": 1e-2,
    "max_iter": 500,
    "train_records": len(train_idx),
    "test_records": len(test_idx),
    "inner_cv_balanced_accuracy": float(
        best_overall["cv_balanced_accuracy"]
    ),
    "inner_cv_balanced_accuracy_sd": float(
        best_overall["cv_std"]
    ),
    "test_balanced_accuracy": float(test_ba),
    "test_macro_f1": float(test_macro_f1),
    "test_macro_auroc": (
        float(test_auroc)
        if test_auroc is not None else None
    ),
    "per_class": {
        CLASS_NAMES[i]: {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i])
        }
        for i in range(3)
    },
    "confusion_matrix": cm.tolist()
}

metrics_path = probe_dir / "qwen_v2_3class_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\nSaved:")
print(pred_path)
print(metrics_path)

Qwen v2.0 — FINAL 3-CLASS TEST EVALUATION
Selected layer: 20
Selected C: 0.1
Training records: 1709
Test records: 425
Test set used ONLY for final evaluation

X_train: (1709, 3584)
X_test : (425, 3584)

FINAL TEST RESULTS
Balanced Accuracy : 0.9216
Macro-F1          : 0.9209
Macro AUROC       : 0.9951

Per-class metrics:
----------------------------------------------------------------------
reversible           | Precision=0.8788 | Recall=0.8529 | F1=0.8657 | Support=68
partially_reversible | Precision=0.9896 | Recall=0.9314 | F1=0.9596 | Support=204
irreversible         | Precision=0.8982 | Recall=0.9804 | F1=0.9375 | Support=153

Confusion Matrix
(rows = true, columns = predicted)
----------------------------------------------------------------------
                   reversib    partiall    irrevers

reversible               58           2           8
partially_revers          5         190           9
irreversible              3           0         150


NameError: name 'best_overall' is not defined

In [35]:
import json
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# Recover Qwen v2.0 probe split + extraction index
# ---------------------------------------------------------

ROOT = Path("/kaggle/working/ADNN_RQ1_Qwen_v2")
HS_DIR = ROOT / "hidden_states"
PROBE_DIR = ROOT / "probes"

# Load extraction index
index_path = HS_DIR / "extraction_index.jsonl"

extraction_index = []

with open(index_path, "r") as f:
    for line in f:
        extraction_index.append(json.loads(line))

print("Extraction index:", len(extraction_index))

# Load official-derived probe split
split_path = PROBE_DIR / "probe_split.json"

with open(split_path, "r") as f:
    split_info = json.load(f)

print("Probe split loaded")
print("Train records:", split_info["train_records"])
print("Test records :", split_info["test_records"])

# Build indices from extraction index
train_idx = []
test_idx = []

for i, row in enumerate(extraction_index):

    if row["split"] == "train":
        train_idx.append(i)

    elif row["split"] == "test":
        test_idx.append(i)

    else:
        raise ValueError(
            f"Unknown split at row {i}: {row['split']}"
        )

train_idx = np.asarray(train_idx, dtype=np.int64)
test_idx = np.asarray(test_idx, dtype=np.int64)

# Load labels
label_mapping = {
    "reversible": 0,
    "partially_reversible": 1,
    "irreversible": 2
}

y_all = np.asarray([
    label_mapping[row["label"]]
    for row in extraction_index
], dtype=np.int64)

y_train = y_all[train_idx]
y_test = y_all[test_idx]

# Scenario groups for training CV
groups_train = np.asarray([
    extraction_index[i]["scenario_id"]
    for i in train_idx
])

print("\nRecovered:")
print("train_idx:", train_idx.shape)
print("test_idx :", test_idx.shape)
print("y_train  :", y_train.shape)
print("y_test   :", y_test.shape)
print("groups   :", groups_train.shape)

# Verify against frozen split
assert len(train_idx) == split_info["train_records"]
assert len(test_idx) == split_info["test_records"]

assert len(train_idx) == 1709
assert len(test_idx) == 425

print("\nSplit recovery PASSED.")

Extraction index: 2134
Probe split loaded
Train records: 1709
Test records : 425

Recovered:
train_idx: (1709,)
test_idx : (425,)
y_train  : (1709,)
y_test   : (425,)
groups   : (1709,)

Split recovery PASSED.


In [36]:
# ============================================================
# SAVE QWEN v2.0 FINAL 3-CLASS RESULTS
# ============================================================

import json
import os

results_dir = os.path.join(
    WORK_DIR,
    "probes"
)

os.makedirs(results_dir, exist_ok=True)

final_results = {
    "experiment": "ADNN_RQ1_Qwen_v2_final",
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "dataset_version": "adnn-dataset-2.0",

    "task": "three_class",

    "selected_layer": 20,
    "selected_C": 0.1,

    "inner_cv_balanced_accuracy": 0.8928,

    "train_records": 1709,
    "test_records": 425,

    "test_metrics": {
        "balanced_accuracy": 0.9216,
        "macro_f1": 0.9209,
        "macro_auroc": 0.9951
    },

    "per_class": {
        "reversible": {
            "precision": 0.8788,
            "recall": 0.8529,
            "f1": 0.8657,
            "support": 68
        },
        "partially_reversible": {
            "precision": 0.9896,
            "recall": 0.9314,
            "f1": 0.9596,
            "support": 204
        },
        "irreversible": {
            "precision": 0.8982,
            "recall": 0.9804,
            "f1": 0.9375,
            "support": 153
        }
    },

    "confusion_matrix": [
        [58, 2, 8],
        [5, 190, 9],
        [3, 0, 150]
    ],

    "cv_selection_source": "inner_group_kfold",
    "test_used_only_for_final_evaluation": True
}

output_path = os.path.join(
    results_dir,
    "qwen_v2_final_3class_results.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        final_results,
        f,
        indent=2
    )

print("=" * 70)
print("FINAL 3-CLASS RESULTS SAVED")
print("=" * 70)
print(output_path)
print("\n✓ Layer 20 / C=0.1 recorded")
print("✓ Inner-CV BA = 0.8928 recorded")
print("✓ Test metrics recorded")
print("✓ Per-class metrics recorded")
print("✓ Confusion matrix recorded")

FINAL 3-CLASS RESULTS SAVED
/kaggle/working/ADNN_RQ1_Qwen_v2/probes/qwen_v2_final_3class_results.json

✓ Layer 20 / C=0.1 recorded
✓ Inner-CV BA = 0.8928 recorded
✓ Test metrics recorded
✓ Per-class metrics recorded
✓ Confusion matrix recorded


In [37]:
# ============================================================
# ADNN RQ1 — COMPLETE PROJECT BACKUP
# ============================================================

import os
import shutil
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/ADNN_RQ1_Qwen_v2")
ZIP_BASE = Path("/kaggle/working/ADNN_RQ1_Qwen_v2_BACKUP")

print("=" * 70)
print("ADNN RQ1 — CREATING COMPLETE BACKUP")
print("=" * 70)

assert PROJECT_DIR.exists(), f"Project folder not found: {PROJECT_DIR}"

# ------------------------------------------------------------
# Verify critical files BEFORE creating backup
# ------------------------------------------------------------

critical_files = [
    PROJECT_DIR / "hidden_states" / "last_action_token.npy",
    PROJECT_DIR / "hidden_states" / "mean_action_tokens.npy",
    PROJECT_DIR / "hidden_states" / "extraction_index.jsonl",
    PROJECT_DIR / "hidden_states" / "extraction_manifest.jsonl",
    PROJECT_DIR / "hidden_states" / "extraction_metadata.json",
    PROJECT_DIR / "hidden_states" / "extraction_progress.npy",
    PROJECT_DIR / "probes" / "probe_split.json",
    PROJECT_DIR / "probes" / "qwen_v2_final_3class_results.json",
]

print("\nChecking critical files:")

for path in critical_files:
    exists = path.exists()
    print(f"{'✓' if exists else '✗'} {path.relative_to(PROJECT_DIR)}")
    assert exists, f"Missing critical file: {path}"

# ------------------------------------------------------------
# Calculate project size
# ------------------------------------------------------------

total_bytes = sum(
    p.stat().st_size
    for p in PROJECT_DIR.rglob("*")
    if p.is_file()
)

print(
    f"\nProject size: "
    f"{total_bytes / (1024**3):.2f} GB"
)

# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------

print("\nCreating ZIP...")
print("This may take a few minutes because the hidden-state arrays")
print("are large.")

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=PROJECT_DIR.parent,
    base_dir=PROJECT_DIR.name
)

zip_size = os.path.getsize(zip_path)

print("\n" + "=" * 70)
print("BACKUP COMPLETE")
print("=" * 70)

print("ZIP:")
print(zip_path)

print(
    f"\nZIP size: "
    f"{zip_size / (1024**3):.2f} GB"
)

print(
    f"ZIP size: "
    f"{zip_size / (1024**2):.1f} MB"
)

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

assert os.path.exists(zip_path)
assert zip_size > 0

print("\n✓ Critical files existed before backup")
print("✓ Hidden states included")
print("✓ Probe results included")
print("✓ Extraction metadata included")
print("✓ ZIP successfully created")

print("\nDOWNLOAD THIS FILE FROM KAGGLE:")
print(zip_path)

ADNN RQ1 — CREATING COMPLETE BACKUP

Checking critical files:
✓ hidden_states/last_action_token.npy
✓ hidden_states/mean_action_tokens.npy
✓ hidden_states/extraction_index.jsonl
✓ hidden_states/extraction_manifest.jsonl
✓ hidden_states/extraction_metadata.json
✓ hidden_states/extraction_progress.npy
✓ probes/probe_split.json
✓ probes/qwen_v2_final_3class_results.json

Project size: 0.83 GB

Creating ZIP...
This may take a few minutes because the hidden-state arrays
are large.

BACKUP COMPLETE
ZIP:
/kaggle/working/ADNN_RQ1_Qwen_v2_BACKUP.zip

ZIP size: 0.77 GB
ZIP size: 786.3 MB

✓ Critical files existed before backup
✓ Hidden states included
✓ Probe results included
✓ Extraction metadata included
✓ ZIP successfully created

DOWNLOAD THIS FILE FROM KAGGLE:
/kaggle/working/ADNN_RQ1_Qwen_v2_BACKUP.zip


In [38]:
from pathlib import Path

zip_path = Path("/kaggle/working/ADNN_RQ1_Qwen_v2_BACKUP.zip")
project_dir = Path("/kaggle/working/ADNN_RQ1_Qwen_v2")

print("=" * 70)
print("BACKUP VERIFICATION")
print("=" * 70)

print("ZIP exists:", zip_path.exists())

if zip_path.exists():
    print(
        "ZIP size:",
        round(zip_path.stat().st_size / (1024**3), 2),
        "GB"
    )

print("\nHidden states:")
for name in [
    "last_action_token.npy",
    "mean_action_tokens.npy",
]:
    p = project_dir / "hidden_states" / name
    print(f"✓ {name}: {round(p.stat().st_size/(1024**3), 2)} GB")

print("\nFinal 3-class results:")
p = project_dir / "probes" / "qwen_v2_final_3class_results.json"
print("✓ Exists:", p.exists())

assert zip_path.exists()
assert (project_dir / "hidden_states" / "last_action_token.npy").exists()
assert (project_dir / "hidden_states" / "mean_action_tokens.npy").exists()
assert p.exists()

print("\n✓ BACKUP AND EXPERIMENT ARTIFACTS VERIFIED")

BACKUP VERIFICATION
ZIP exists: True
ZIP size: 0.77 GB

Hidden states:
✓ last_action_token.npy: 0.41 GB
✓ mean_action_tokens.npy: 0.41 GB

Final 3-class results:
✓ Exists: True

✓ BACKUP AND EXPERIMENT ARTIFACTS VERIFIED


In [40]:
# ============================================================
# RESTORE FULL LABEL ARRAY FOR BINARY PROBE
# ============================================================

import numpy as np

LABEL_TO_ID = {
    "reversible": 0,
    "partially_reversible": 1,
    "irreversible": 2,
}

ID_TO_LABEL = {
    v: k for k, v in LABEL_TO_ID.items()
}

# Rebuild labels from the original records
y_all = np.asarray(
    [
        LABEL_TO_ID[record["label"]]
        for record in records
    ],
    dtype=np.int64
)

# Keep y as the full-dataset label array
y = y_all

# Rebuild train/test labels
y_train = y_all[train_idx]
y_test = y_all[test_idx]

print("=" * 70)
print("FULL LABEL ARRAY RESTORED")
print("=" * 70)

print("y shape:", y.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\n3-class distribution:")

for label_id in range(3):
    print(
        f"  {ID_TO_LABEL[label_id]:25s}"
        f" train={int(np.sum(y_train == label_id)):4d}"
        f" test={int(np.sum(y_test == label_id)):4d}"
    )

assert y.shape == (2134,)
assert y_train.shape == (1709,)
assert y_test.shape == (425,)

assert np.array_equal(
    np.sort(np.unique(y)),
    np.array([0, 1, 2])
)

print("\n✓ Full labels restored")
print("✓ Train/test labels aligned")

FULL LABEL ARRAY RESTORED
y shape: (2134,)
y_train shape: (1709,)
y_test shape: (425,)

3-class distribution:
  reversible                train= 731 test=  68
  partially_reversible      train= 554 test= 204
  irreversible              train= 424 test= 153

✓ Full labels restored
✓ Train/test labels aligned


In [41]:
# ============================================================
# ADNN RQ1 — BINARY PROBE SETUP / LABEL CHECK
# ============================================================

import json
import numpy as np
from pathlib import Path

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"
HIDDEN_DIR = Path(WORK_DIR) / "hidden_states"

# ------------------------------------------------------------
# Load primary hidden-state representation
# ------------------------------------------------------------

X_all = np.load(
    HIDDEN_DIR / "last_action_token.npy",
    mmap_mode="r"
)

# ------------------------------------------------------------
# Restore split indices
# ------------------------------------------------------------

train_idx = np.where(train_mask)[0]
test_idx = np.where(test_mask)[0]

groups_train = scenario_ids[train_idx]
groups_test = scenario_ids[test_idx]

print("=" * 70)
print("BINARY PROBE SETUP")
print("=" * 70)

print("Hidden states:", X_all.shape)
print("dtype:", X_all.dtype)

print("\nTrain records:", len(train_idx))
print("Test records :", len(test_idx))

print("\nOriginal 3-class labels:")
for label_id, label_name in ID_TO_LABEL.items():
    print(
        f"  {label_id}: {label_name:25s} "
        f"train={int(np.sum(y[train_idx] == label_id)):4d} "
        f"test={int(np.sum(y[test_idx] == label_id)):4d}"
    )

# ------------------------------------------------------------
# Look for an existing binary configuration
# ------------------------------------------------------------

config_dir = Path(WORK_DIR) / "config"
probe_dir = Path(WORK_DIR) / "probes"

print("\n" + "=" * 70)
print("EXISTING BINARY CONFIGURATION")
print("=" * 70)

for path in sorted(
    list(config_dir.glob("*.json")) +
    list(probe_dir.glob("*binary*.json"))
):
    print("\nFILE:", path.name)

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Print only binary-related fields if present
        found = False

        for key, value in data.items():
            if "binary" in str(key).lower() or "binary" in str(value).lower():
                print(f"  {key}: {value}")
                found = True

        if not found:
            print("  No binary-specific field found.")

    except Exception as e:
        print("  Could not read:", e)

print("\n" + "=" * 70)
print("✓ BINARY SETUP CHECK COMPLETE")
print("=" * 70)

BINARY PROBE SETUP
Hidden states: (2134, 29, 3584)
dtype: float16

Train records: 1709
Test records : 425

Original 3-class labels:
  0: reversible                train= 731 test=  68
  1: partially_reversible      train= 554 test= 204
  2: irreversible              train= 424 test= 153

EXISTING BINARY CONFIGURATION

FILE: environment.json
  No binary-specific field found.

FILE: experiment_config.json
  required_probe_tasks: ['binary', 'three_class']

✓ BINARY SETUP CHECK COMPLETE


In [42]:
# ============================================================
# ADNN RQ1 — BINARY LABEL DEFINITION
# ============================================================

import numpy as np

# Binary task:
# 0 = non-irreversible
#     (reversible + partially_reversible)
#
# 1 = irreversible

BINARY_LABEL_TO_ID = {
    "non_irreversible": 0,
    "irreversible": 1,
}

BINARY_ID_TO_LABEL = {
    0: "non_irreversible",
    1: "irreversible",
}

# Build from the full 3-class label array
y_binary = np.where(
    y == LABEL_TO_ID["irreversible"],
    1,
    0
).astype(np.int64)

y_binary_train = y_binary[train_idx]
y_binary_test = y_binary[test_idx]

print("=" * 70)
print("BINARY LABEL DEFINITION")
print("=" * 70)

print("0 = non_irreversible")
print("    reversible + partially_reversible")

print("1 = irreversible")

print("\nTRAIN")
print(
    "  non_irreversible:",
    int(np.sum(y_binary_train == 0))
)
print(
    "  irreversible    :",
    int(np.sum(y_binary_train == 1))
)

print("\nTEST")
print(
    "  non_irreversible:",
    int(np.sum(y_binary_test == 0))
)
print(
    "  irreversible    :",
    int(np.sum(y_binary_test == 1))
)

assert y_binary.shape == (2134,)
assert y_binary_train.shape == (1709,)
assert y_binary_test.shape == (425,)

assert np.sum(y_binary_train == 0) == 1285
assert np.sum(y_binary_train == 1) == 424

assert np.sum(y_binary_test == 0) == 272
assert np.sum(y_binary_test == 1) == 153

print("\n✓ Binary labels created")
print("✓ Train/test counts verified")

BINARY LABEL DEFINITION
0 = non_irreversible
    reversible + partially_reversible
1 = irreversible

TRAIN
  non_irreversible: 1285
  irreversible    : 424

TEST
  non_irreversible: 272
  irreversible    : 153

✓ Binary labels created
✓ Train/test counts verified


In [43]:
# ============================================================
# ADNN RQ1 — QWEN v2.0 BINARY LAYER SCREENING
# ============================================================

import time
import json
import warnings
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

C = 0.001
N_SPLITS = 5
N_LAYERS = X_all.shape[1]

results_binary_screen = []

print("=" * 70)
print("Qwen v2.0 — Binary Layer Screening")
print("=" * 70)
print(f"Layers: {N_LAYERS}")
print(f"C: {C}")
print(f"CV: {N_SPLITS}-fold GroupKFold")
print(f"Training records: {len(train_idx)}")
print("Test set: NOT USED")
print()

overall_start = time.time()

# ------------------------------------------------------------
# Screen every layer
# ------------------------------------------------------------

for layer in range(N_LAYERS):

    layer_start = time.time()

    # Load one layer
    X = np.asarray(
        X_all[train_idx, layer, :],
        dtype=np.float32
    )

    y_layer = y_binary_train
    groups = groups_train

    gkf = GroupKFold(
        n_splits=N_SPLITS
    )

    fold_scores = []

    # --------------------------------------------------------
    # 5-fold GroupKFold
    # --------------------------------------------------------

    for fold, (tr, va) in enumerate(
        gkf.split(X, y_layer, groups),
        start=1
    ):

        # Fit scaler ONLY on training fold
        scaler = StandardScaler()

        Xtr = scaler.fit_transform(X[tr])
        Xva = scaler.transform(X[va])

        clf = LogisticRegression(
            C=C,
            penalty="l2",
            solver="saga",
            max_iter=500,
            tol=1e-2,
            class_weight="balanced",
            random_state=0
        )

        clf.fit(
            Xtr,
            y_layer[tr]
        )

        pred = clf.predict(Xva)

        ba = balanced_accuracy_score(
            y_layer[va],
            pred
        )

        fold_scores.append(
            float(ba)
        )

    # --------------------------------------------------------
    # Aggregate fold results
    # --------------------------------------------------------

    mean_ba = float(
        np.mean(fold_scores)
    )

    std_ba = float(
        np.std(fold_scores)
    )

    elapsed = time.time() - layer_start

    results_binary_screen.append({
        "layer": layer,
        "C": C,
        "cv_balanced_accuracy": mean_ba,
        "cv_std": std_ba,
        "fold_scores": fold_scores,
        "time_sec": elapsed
    })

    # --------------------------------------------------------
    # Progress / ETA
    # --------------------------------------------------------

    total_elapsed = time.time() - overall_start

    avg_layer_time = (
        total_elapsed / (layer + 1)
    )

    remaining = (
        avg_layer_time *
        (N_LAYERS - layer - 1)
    )

    print(
        f"Layer {layer:2d} | "
        f"BA={mean_ba:.4f} | "
        f"SD={std_ba:.4f} | "
        f"time={elapsed:6.1f}s | "
        f"ETA={remaining/60:5.1f} min"
    )

# ------------------------------------------------------------
# Final ranking
# ------------------------------------------------------------

ranked_binary_screen = sorted(
    results_binary_screen,
    key=lambda x: x["cv_balanced_accuracy"],
    reverse=True
)

print()
print("=" * 70)
print("BINARY SCREENING COMPLETE")
print("=" * 70)

print("\nTop 10 layers:")
print("-" * 70)

for rank, r in enumerate(
    ranked_binary_screen[:10],
    start=1
):

    print(
        f"{rank:2d}. "
        f"Layer {r['layer']:2d} | "
        f"BA={r['cv_balanced_accuracy']:.4f} | "
        f"SD={r['cv_std']:.4f}"
    )

print(
    f"\nTotal time: "
    f"{(time.time()-overall_start)/60:.1f} min"
)

# ------------------------------------------------------------
# Save screening results
# ------------------------------------------------------------

probe_dir = "/kaggle/working/ADNN_RQ1_Qwen_v2/probes"

screen_output = (
    f"{probe_dir}/qwen_v2_binary_layer_screening.json"
)

with open(
    screen_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "task": "binary",
            "representation": "last_action_token",
            "C": C,
            "n_splits": N_SPLITS,
            "selection_method": "GroupKFold_by_scenario",
            "test_used": False,
            "results": results_binary_screen,
            "ranking": ranked_binary_screen
        },
        f,
        indent=2
    )

print("\nSaved:")
print(screen_output)

print("\n✓ Binary layer screening complete")

Qwen v2.0 — Binary Layer Screening
Layers: 29
C: 0.001
CV: 5-fold GroupKFold
Training records: 1709
Test set: NOT USED

Layer  0 | BA=0.5000 | SD=0.0000 | time=   9.2s | ETA=  4.3 min
Layer  1 | BA=0.7052 | SD=0.0758 | time=  16.7s | ETA=  5.8 min
Layer  2 | BA=0.7126 | SD=0.0750 | time=  22.9s | ETA=  7.0 min
Layer  3 | BA=0.7067 | SD=0.0716 | time=  22.9s | ETA=  7.5 min
Layer  4 | BA=0.7153 | SD=0.0746 | time=  23.3s | ETA=  7.6 min
Layer  5 | BA=0.7496 | SD=0.0780 | time=  21.7s | ETA=  7.4 min
Layer  6 | BA=0.7530 | SD=0.0736 | time=  19.0s | ETA=  7.1 min
Layer  7 | BA=0.7994 | SD=0.0629 | time=  18.3s | ETA=  6.7 min
Layer  8 | BA=0.8194 | SD=0.0529 | time=  17.0s | ETA=  6.3 min
Layer  9 | BA=0.8331 | SD=0.0438 | time=  17.9s | ETA=  6.0 min
Layer 10 | BA=0.8332 | SD=0.0410 | time=  15.7s | ETA=  5.6 min
Layer 11 | BA=0.8424 | SD=0.0411 | time=  15.0s | ETA=  5.2 min
Layer 12 | BA=0.8582 | SD=0.0482 | time=  11.8s | ETA=  4.7 min
Layer 13 | BA=0.8637 | SD=0.0530 | time=  11.5s 

In [44]:
import os
import json
import time
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

# ============================================================
# Qwen v2.0 — Binary Inner C Selection
# ============================================================

print("=" * 70)
print("Qwen v2.0 — Binary Inner C Selection")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TOP_LAYERS = list(range(16, 26))
C_GRID = [0.001, 0.01, 0.1, 1, 10]
N_SPLITS = 5
RANDOM_STATE = 42

print(f"Layers: {TOP_LAYERS}")
print(f"C grid: {C_GRID}")
print(f"CV: {N_SPLITS}-fold GroupKFold")
print(f"Training records: {len(train_idx)}")
print("Test set: NOT USED")

# ------------------------------------------------------------
# Training data
# ------------------------------------------------------------
y_train_binary = y_binary[train_idx]
groups_train = scenario_ids[train_idx]

X_all = np.load(
    os.path.join(WORK_DIR, "hidden_states", "last_action_token.npy"),
    mmap_mode="r"
)

# ------------------------------------------------------------
# GroupKFold
# ------------------------------------------------------------
gkf = GroupKFold(n_splits=N_SPLITS)

results_binary_c = []

overall_start = time.time()

# ------------------------------------------------------------
# Evaluate every layer + C
# ------------------------------------------------------------
for layer in TOP_LAYERS:

    X_layer = X_all[train_idx, layer, :]

    best_for_layer = None

    for C in C_GRID:

        fold_scores = []
        start = time.time()

        for fold, (tr_idx, val_idx) in enumerate(
            gkf.split(X_layer, y_train_binary, groups_train), start=1
        ):

            X_tr = X_layer[tr_idx]
            X_val = X_layer[val_idx]

            y_tr = y_train_binary[tr_idx]
            y_val = y_train_binary[val_idx]

            clf = Pipeline([
                ("scaler", StandardScaler()),
                ("logreg", LogisticRegression(
                    penalty="l2",
                    C=C,
                    class_weight="balanced",
                    solver="saga",
                    max_iter=500,
                    tol=1e-2,
                    random_state=RANDOM_STATE
                ))
            ])

            clf.fit(X_tr, y_tr)

            pred = clf.predict(X_val)

            ba = balanced_accuracy_score(y_val, pred)
            fold_scores.append(float(ba))

        mean_ba = float(np.mean(fold_scores))
        std_ba = float(np.std(fold_scores))

        result = {
            "layer": int(layer),
            "C": float(C),
            "mean_balanced_accuracy": mean_ba,
            "std_balanced_accuracy": std_ba,
            "fold_scores": fold_scores
        }

        results_binary_c.append(result)

        elapsed = time.time() - start

        print(
            f"Layer {layer:2d} | "
            f"C={C:<5} | "
            f"BA={mean_ba:.4f} | "
            f"SD={std_ba:.4f} | "
            f"time={elapsed:5.1f}s"
        )

# ------------------------------------------------------------
# Rank configurations
# ------------------------------------------------------------
ranked_binary_c = sorted(
    results_binary_c,
    key=lambda x: x["mean_balanced_accuracy"],
    reverse=True
)

best_binary = ranked_binary_c[0]

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
output_path = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_c_selection.json"
)

output = {
    "experiment": "qwen_v2_binary_inner_c_selection",
    "representation": "last_action_token",
    "task": "binary",
    "label_mapping": {
        "0": "non_irreversible",
        "1": "irreversible"
    },
    "layers_evaluated": TOP_LAYERS,
    "C_grid": C_GRID,
    "cv": {
        "method": "GroupKFold",
        "n_splits": N_SPLITS,
        "group_key": "scenario_id",
        "random_state": RANDOM_STATE
    },
    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2
    },
    "training_records": int(len(train_idx)),
    "test_used": False,
    "results": results_binary_c,
    "ranking": ranked_binary_c,
    "selected": best_binary
}

with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print()
print("=" * 70)
print("BINARY INNER C SELECTION COMPLETE")
print("=" * 70)

print("\nTop 10 layer + C configurations:")
print("-" * 70)

for i, r in enumerate(ranked_binary_c[:10], start=1):
    print(
        f"{i:2d}. Layer {r['layer']:2d} | "
        f"C={r['C']:<5} | "
        f"BA={r['mean_balanced_accuracy']:.4f} | "
        f"SD={r['std_balanced_accuracy']:.4f}"
    )

print("\nSelected configuration:")
print(f"Layer = {best_binary['layer']}")
print(f"C     = {best_binary['C']}")
print(f"CV BA = {best_binary['mean_balanced_accuracy']:.4f}")
print(f"CV SD = {best_binary['std_balanced_accuracy']:.4f}")

print(f"\nSaved:")
print(output_path)

print(f"\nTotal time: {(time.time() - overall_start) / 60:.1f} min")
print("✓ Binary C selection complete")

Qwen v2.0 — Binary Inner C Selection
Layers: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
C grid: [0.001, 0.01, 0.1, 1, 10]
CV: 5-fold GroupKFold
Training records: 1709
Test set: NOT USED
Layer 16 | C=0.001 | BA=0.8870 | SD=0.0420 | time=  9.2s
Layer 16 | C=0.01  | BA=0.9070 | SD=0.0358 | time= 17.2s
Layer 16 | C=0.1   | BA=0.9109 | SD=0.0391 | time= 21.2s
Layer 16 | C=1     | BA=0.9113 | SD=0.0393 | time= 21.7s
Layer 16 | C=10    | BA=0.9113 | SD=0.0393 | time= 21.3s
Layer 17 | C=0.001 | BA=0.8983 | SD=0.0416 | time=  8.8s
Layer 17 | C=0.01  | BA=0.9114 | SD=0.0349 | time= 15.6s
Layer 17 | C=0.1   | BA=0.9173 | SD=0.0340 | time= 19.1s
Layer 17 | C=1     | BA=0.9165 | SD=0.0334 | time= 19.4s
Layer 17 | C=10    | BA=0.9170 | SD=0.0327 | time= 19.4s
Layer 18 | C=0.001 | BA=0.8984 | SD=0.0421 | time=  8.5s
Layer 18 | C=0.01  | BA=0.9211 | SD=0.0361 | time= 15.3s
Layer 18 | C=0.1   | BA=0.9181 | SD=0.0371 | time= 18.6s
Layer 18 | C=1     | BA=0.9186 | SD=0.0376 | time= 19.6s
Layer 18 | C=10   

In [45]:
import os
import json
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
    accuracy_score
)

# ============================================================
# Qwen v2.0 — FINAL BINARY PROBE
# Layer 18, C=0.01
# ============================================================

print("=" * 70)
print("Qwen v2.0 — FINAL BINARY PROBE")
print("=" * 70)

# ------------------------------------------------------------
# Locked configuration from inner CV
# ------------------------------------------------------------
BEST_LAYER = 18
BEST_C = 0.01
RANDOM_STATE = 42

print(f"Representation : last_action_token")
print(f"Layer          : {BEST_LAYER}")
print(f"C              : {BEST_C}")
print(f"Task           : binary")
print(f"Label 0        : non_irreversible")
print(f"Label 1        : irreversible")
print(f"Training       : {len(train_idx)} records")
print(f"Test           : {len(test_idx)} records")
print()

# ------------------------------------------------------------
# Load hidden states
# ------------------------------------------------------------
hidden_path = os.path.join(
    WORK_DIR,
    "hidden_states",
    "last_action_token.npy"
)

X_all = np.load(hidden_path, mmap_mode="r")

print(f"Hidden-state array shape: {X_all.shape}")

# Selected layer
X_train = X_all[train_idx, BEST_LAYER, :]
X_test = X_all[test_idx, BEST_LAYER, :]

y_train_final = y_binary[train_idx]
y_test_final = y_binary[test_idx]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print()

# ------------------------------------------------------------
# Train final probe
# ------------------------------------------------------------
print("Training final probe...")

start_time = time.time()

final_probe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        penalty="l2",
        C=BEST_C,
        class_weight="balanced",
        solver="saga",
        max_iter=500,
        tol=1e-2,
        random_state=RANDOM_STATE
    ))
])

final_probe.fit(X_train, y_train_final)

train_time = time.time() - start_time

print(f"Training complete in {train_time:.1f}s")
print()

# ------------------------------------------------------------
# Test prediction
# ------------------------------------------------------------
print("Evaluating on untouched test set...")

test_pred = final_probe.predict(X_test)
test_prob = final_probe.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
accuracy = accuracy_score(
    y_test_final,
    test_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_test_final,
    test_pred
)

macro_f1 = f1_score(
    y_test_final,
    test_pred,
    average="macro"
)

precision_per_class = precision_score(
    y_test_final,
    test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

recall_per_class = recall_score(
    y_test_final,
    test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

f1_per_class = f1_score(
    y_test_final,
    test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

conf_matrix = confusion_matrix(
    y_test_final,
    test_pred,
    labels=[0, 1]
)

auroc = roc_auc_score(
    y_test_final,
    test_prob
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------
print()
print("=" * 70)
print("FINAL BINARY TEST RESULTS")
print("=" * 70)

print(f"Test records       : {len(test_idx)}")
print(f"Accuracy           : {accuracy:.4f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.4f}")
print(f"Macro-F1           : {macro_f1:.4f}")
print(f"AUROC              : {auroc:.4f}")

print()
print("Per-class metrics:")
print("-" * 70)

print(
    f"Class 0 — non_irreversible | "
    f"Precision={precision_per_class[0]:.4f} | "
    f"Recall={recall_per_class[0]:.4f} | "
    f"F1={f1_per_class[0]:.4f}"
)

print(
    f"Class 1 — irreversible     | "
    f"Precision={precision_per_class[1]:.4f} | "
    f"Recall={recall_per_class[1]:.4f} | "
    f"F1={f1_per_class[1]:.4f}"
)

print()
print("Confusion Matrix")
print("(rows = true, columns = predicted)")
print()
print(conf_matrix)

# ------------------------------------------------------------
# Save predictions
# ------------------------------------------------------------
predictions_path = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_predictions.jsonl"
)

with open(predictions_path, "w") as f:

    for i, original_idx in enumerate(test_idx):

        record = records[int(original_idx)]

        output_record = {
            "record_id": record["record_id"],
            "scenario_id": record["scenario_id"],
            "true_label": int(y_test_final[i]),
            "true_label_name": (
                "non_irreversible"
                if y_test_final[i] == 0
                else "irreversible"
            ),
            "predicted_label": int(test_pred[i]),
            "predicted_label_name": (
                "non_irreversible"
                if test_pred[i] == 0
                else "irreversible"
            ),
            "probability_irreversible": float(test_prob[i])
        }

        f.write(json.dumps(output_record) + "\n")

# ------------------------------------------------------------
# Save final results
# ------------------------------------------------------------
results_path = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_final_binary_results.json"
)

results = {
    "experiment": "qwen_v2_final_binary_probe",

    "dataset": {
        "version": "adnn-dataset-2.0",
        "total_records": int(len(records)),
        "train_records": int(len(train_idx)),
        "test_records": int(len(test_idx)),
        "split_type": "scenario_grouped"
    },

    "representation": {
        "type": "last_action_token",
        "layer": int(BEST_LAYER)
    },

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "selection": {
        "method": "inner_group_kfold_cv",
        "selected_C": float(BEST_C),
        "inner_cv_balanced_accuracy": 0.9211,
        "inner_cv_sd": 0.0361,
        "test_used_during_selection": False
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": float(BEST_C),
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": RANDOM_STATE
    },

    "test_metrics": {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "macro_f1": float(macro_f1),
        "auroc": float(auroc),

        "per_class": {
            "non_irreversible": {
                "precision": float(precision_per_class[0]),
                "recall": float(recall_per_class[0]),
                "f1": float(f1_per_class[0])
            },
            "irreversible": {
                "precision": float(precision_per_class[1]),
                "recall": float(recall_per_class[1]),
                "f1": float(f1_per_class[1])
            }
        },

        "confusion_matrix": conf_matrix.tolist()
    },

    "artifacts": {
        "predictions": predictions_path
    }
}

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------
print()
print("=" * 70)
print("FINAL BINARY PROBE COMPLETE")
print("=" * 70)

print()
print("Saved:")
print(results_path)
print(predictions_path)

print()
print("✓ Final probe trained on all training records")
print("✓ Test set evaluated only after model selection")
print("✓ Predictions saved")
print("✓ Metrics saved")

Qwen v2.0 — FINAL BINARY PROBE
Representation : last_action_token
Layer          : 18
C              : 0.01
Task           : binary
Label 0        : non_irreversible
Label 1        : irreversible
Training       : 1709 records
Test           : 425 records

Hidden-state array shape: (2134, 29, 3584)
X_train shape: (1709, 3584)
X_test shape : (425, 3584)

Training final probe...
Training complete in 3.9s

Evaluating on untouched test set...

FINAL BINARY TEST RESULTS
Test records       : 425
Accuracy           : 0.8871
Balanced Accuracy  : 0.9118
Macro-F1           : 0.8838
AUROC              : 0.9864

Per-class metrics:
----------------------------------------------------------------------
Class 0 — non_irreversible | Precision=1.0000 | Recall=0.8235 | F1=0.9032
Class 1 — irreversible     | Precision=0.7612 | Recall=1.0000 | F1=0.8644

Confusion Matrix
(rows = true, columns = predicted)

[[224  48]
 [  0 153]]


KeyError: 'record_id'

In [46]:
# ============================================================
# FIX: SAVE FINAL BINARY PREDICTIONS + RESULTS
# ============================================================

import os
import json

# Inspect the actual record structure
print("Available record keys:")
print(records[0].keys())
print()

# ------------------------------------------------------------
# Find the correct ID field
# ------------------------------------------------------------
possible_id_fields = [
    "record_id",
    "id",
    "uid",
    "example_id",
    "sample_id"
]

id_field = None

for field in possible_id_fields:
    if field in records[0]:
        id_field = field
        break

print("Selected ID field:", id_field)

if id_field is None:
    raise KeyError(
        "Could not find a record ID field. "
        "Please inspect records[0].keys()."
    )

# ------------------------------------------------------------
# Save predictions
# ------------------------------------------------------------
predictions_path = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_predictions.jsonl"
)

with open(predictions_path, "w") as f:

    for i, original_idx in enumerate(test_idx):

        record = records[int(original_idx)]

        output_record = {
            "record_id": str(record[id_field]),
            "scenario_id": str(record["scenario_id"]),

            "true_label": int(y_test_final[i]),
            "true_label_name": (
                "non_irreversible"
                if y_test_final[i] == 0
                else "irreversible"
            ),

            "predicted_label": int(test_pred[i]),
            "predicted_label_name": (
                "non_irreversible"
                if test_pred[i] == 0
                else "irreversible"
            ),

            "probability_irreversible": float(test_prob[i])
        }

        f.write(json.dumps(output_record) + "\n")

# ------------------------------------------------------------
# Save final results
# ------------------------------------------------------------
results_path = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_final_binary_results.json"
)

results = {
    "experiment": "qwen_v2_final_binary_probe",

    "dataset": {
        "version": "adnn-dataset-2.0",
        "total_records": int(len(records)),
        "train_records": int(len(train_idx)),
        "test_records": int(len(test_idx)),
        "split_type": "scenario_grouped"
    },

    "representation": {
        "type": "last_action_token",
        "layer": 18
    },

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "selection": {
        "method": "inner_group_kfold_cv",
        "selected_layer": 18,
        "selected_C": 0.01,
        "inner_cv_balanced_accuracy": 0.9211,
        "inner_cv_sd": 0.0361,
        "test_used_during_selection": False
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": 0.01,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "test_metrics": {
        "test_size": 425,
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "macro_f1": float(macro_f1),
        "auroc": float(auroc),

        "per_class": {
            "non_irreversible": {
                "precision": float(precision_per_class[0]),
                "recall": float(recall_per_class[0]),
                "f1": float(f1_per_class[0])
            },

            "irreversible": {
                "precision": float(precision_per_class[1]),
                "recall": float(recall_per_class[1]),
                "f1": float(f1_per_class[1])
            }
        },

        "confusion_matrix": conf_matrix.tolist()
    },

    "artifacts": {
        "predictions": predictions_path
    }
}

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------
print()
print("=" * 70)
print("FILES SAVED SUCCESSFULLY")
print("=" * 70)

print()
print(results_path)
print(predictions_path)

# Verify prediction count
with open(predictions_path, "r") as f:
    saved_predictions = [json.loads(line) for line in f]

print()
print("Saved prediction rows:", len(saved_predictions))

assert len(saved_predictions) == len(test_idx)

print("✓ Prediction count matches test set")
print("✓ Final binary metrics preserved")
print("✓ No probe retraining was performed")

Available record keys:
dict_keys(['action_call', 'action_text', 'action_verb', 'approx_restore_depth', 'dataset_version', 'environment', 'horizon', 'id', 'label', 'label_provisional', 'label_rationale', 'labels_by_horizon', 'meta', 'pair_id', 'pivot', 'provisional_reasons', 'render_style', 'restore_depth', 'scenario_id', 'sensitivity_group', 'source', 'state_struct', 'state_text', 'task_text', 'template_id', 'tier', 'tool_set_id'])

Selected ID field: id

FILES SAVED SUCCESSFULLY

/kaggle/working/ADNN_RQ1_Qwen_v2/probes/qwen_v2_final_binary_results.json
/kaggle/working/ADNN_RQ1_Qwen_v2/probes/qwen_v2_binary_predictions.jsonl

Saved prediction rows: 425
✓ Prediction count matches test set
✓ Final binary metrics preserved
✓ No probe retraining was performed


In [47]:
# ============================================================
# Qwen v2.0 — 95% BOOTSTRAP CIs
# Scenario-level bootstrap, 1000 resamples
# Metrics: Balanced Accuracy, Macro-F1, AUROC
# ============================================================

import os
import json
import numpy as np

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
N_BOOTSTRAPS = 1000
BOOTSTRAP_SEED = 42

predictions_path = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_predictions.jsonl"
)

print("=" * 70)
print("SCENARIO-LEVEL BOOTSTRAP")
print("=" * 70)

# ------------------------------------------------------------
# Load saved predictions
# ------------------------------------------------------------
with open(predictions_path, "r") as f:
    predictions = [
        json.loads(line)
        for line in f
    ]

print(f"Loaded predictions: {len(predictions)}")

assert len(predictions) == 425

# ------------------------------------------------------------
# Convert predictions to arrays
# ------------------------------------------------------------
y_true = np.array(
    [p["true_label"] for p in predictions],
    dtype=int
)

y_pred = np.array(
    [p["predicted_label"] for p in predictions],
    dtype=int
)

y_prob = np.array(
    [p["probability_irreversible"] for p in predictions],
    dtype=float
)

scenario_ids_test = np.array(
    [p["scenario_id"] for p in predictions],
    dtype=object
)

# ------------------------------------------------------------
# Verify scenario structure
# ------------------------------------------------------------
unique_scenarios = np.unique(scenario_ids_test)

print(f"Test records    : {len(y_true)}")
print(f"Test scenarios  : {len(unique_scenarios)}")

print()
print("Scenario sizes:")

scenario_sizes = []

for scenario in unique_scenarios:
    n = np.sum(scenario_ids_test == scenario)
    scenario_sizes.append(n)

print(
    f"  min = {np.min(scenario_sizes)}"
)
print(
    f"  max = {np.max(scenario_sizes)}"
)
print(
    f"  mean = {np.mean(scenario_sizes):.2f}"
)

# ------------------------------------------------------------
# Observed test metrics
# ------------------------------------------------------------
observed_balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

observed_macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

observed_auroc = roc_auc_score(
    y_true,
    y_prob
)

print()
print("Observed test metrics:")
print(f"  Balanced Accuracy = {observed_balanced_accuracy:.4f}")
print(f"  Macro-F1          = {observed_macro_f1:.4f}")
print(f"  AUROC             = {observed_auroc:.4f}")

# ------------------------------------------------------------
# Build record indices for each scenario
# ------------------------------------------------------------
scenario_to_indices = {}

for scenario in unique_scenarios:

    indices = np.where(
        scenario_ids_test == scenario
    )[0]

    scenario_to_indices[scenario] = indices

# ------------------------------------------------------------
# Bootstrap
#
# Resample SCENARIOS with replacement.
# Every record belonging to a selected scenario stays together.
# ------------------------------------------------------------
rng = np.random.default_rng(BOOTSTRAP_SEED)

bootstrap_balanced_accuracy = []
bootstrap_macro_f1 = []
bootstrap_auroc = []

n_scenarios = len(unique_scenarios)

successful_resamples = 0
skipped_resamples = 0

for b in range(N_BOOTSTRAPS):

    # Sample scenario IDs with replacement
    sampled_scenarios = rng.choice(
        unique_scenarios,
        size=n_scenarios,
        replace=True
    )

    # Collect all records from sampled scenarios
    sampled_indices = []

    for scenario in sampled_scenarios:
        sampled_indices.extend(
            scenario_to_indices[scenario]
        )

    sampled_indices = np.array(
        sampled_indices,
        dtype=int
    )

    y_true_b = y_true[sampled_indices]
    y_pred_b = y_pred[sampled_indices]
    y_prob_b = y_prob[sampled_indices]

    # Balanced accuracy
    ba = balanced_accuracy_score(
        y_true_b,
        y_pred_b
    )

    # Macro-F1
    mf1 = f1_score(
        y_true_b,
        y_pred_b,
        average="macro",
        zero_division=0
    )

    # AUROC requires both classes to be present
    if len(np.unique(y_true_b)) < 2:
        skipped_resamples += 1
        continue

    auc = roc_auc_score(
        y_true_b,
        y_prob_b
    )

    bootstrap_balanced_accuracy.append(ba)
    bootstrap_macro_f1.append(mf1)
    bootstrap_auroc.append(auc)

    successful_resamples += 1

# ------------------------------------------------------------
# Convert to arrays
# ------------------------------------------------------------
bootstrap_balanced_accuracy = np.array(
    bootstrap_balanced_accuracy
)

bootstrap_macro_f1 = np.array(
    bootstrap_macro_f1
)

bootstrap_auroc = np.array(
    bootstrap_auroc
)

print()
print("=" * 70)
print("BOOTSTRAP COMPLETE")
print("=" * 70)

print(f"Requested resamples : {N_BOOTSTRAPS}")
print(f"Successful resamples: {successful_resamples}")
print(f"Skipped resamples   : {skipped_resamples}")

# ------------------------------------------------------------
# 95% percentile confidence intervals
# ------------------------------------------------------------
def bootstrap_ci(values):

    lower = np.percentile(
        values,
        2.5
    )

    upper = np.percentile(
        values,
        97.5
    )

    return float(lower), float(upper)


ba_ci = bootstrap_ci(
    bootstrap_balanced_accuracy
)

mf1_ci = bootstrap_ci(
    bootstrap_macro_f1
)

auroc_ci = bootstrap_ci(
    bootstrap_auroc
)

# ------------------------------------------------------------
# Print final CIs
# ------------------------------------------------------------
print()
print("=" * 70)
print("95% BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 70)

print(
    f"Balanced Accuracy : "
    f"{observed_balanced_accuracy:.4f} "
    f"[{ba_ci[0]:.4f}, {ba_ci[1]:.4f}]"
)

print(
    f"Macro-F1          : "
    f"{observed_macro_f1:.4f} "
    f"[{mf1_ci[0]:.4f}, {mf1_ci[1]:.4f}]"
)

print(
    f"AUROC             : "
    f"{observed_auroc:.4f} "
    f"[{auroc_ci[0]:.4f}, {auroc_ci[1]:.4f}]"
)

# ------------------------------------------------------------
# Save bootstrap results
# ------------------------------------------------------------
bootstrap_results = {
    "experiment": "qwen_v2_final_binary_probe",

    "bootstrap": {
        "method": "scenario_level_bootstrap",
        "n_resamples_requested": N_BOOTSTRAPS,
        "n_resamples_successful": successful_resamples,
        "n_resamples_skipped": skipped_resamples,
        "confidence_level": 0.95,
        "interval_method": "percentile",
        "seed": BOOTSTRAP_SEED,
        "resampling_unit": "scenario_id"
    },

    "test": {
        "n_records": int(len(y_true)),
        "n_scenarios": int(n_scenarios)
    },

    "metrics": {
        "balanced_accuracy": {
            "estimate": float(observed_balanced_accuracy),
            "ci_95": [
                ba_ci[0],
                ba_ci[1]
            ]
        },

        "macro_f1": {
            "estimate": float(observed_macro_f1),
            "ci_95": [
                mf1_ci[0],
                mf1_ci[1]
            ]
        },

        "auroc": {
            "estimate": float(observed_auroc),
            "ci_95": [
                auroc_ci[0],
                auroc_ci[1]
            ]
        }
    }
}

bootstrap_path = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_bootstrap_ci.json"
)

with open(bootstrap_path, "w") as f:
    json.dump(
        bootstrap_results,
        f,
        indent=2
    )

print()
print("Saved:")
print(bootstrap_path)

print()
print("✓ Scenario-level resampling")
print("✓ 1000 bootstrap resamples")
print("✓ 95% percentile CIs")
print("✓ No retraining")
print("✓ Test predictions reused exactly")

SCENARIO-LEVEL BOOTSTRAP
Loaded predictions: 425
Test records    : 425
Test scenarios  : 25

Scenario sizes:
  min = 1
  max = 102
  mean = 17.00

Observed test metrics:
  Balanced Accuracy = 0.9118
  Macro-F1          = 0.8838
  AUROC             = 0.9864

BOOTSTRAP COMPLETE
Requested resamples : 1000
Successful resamples: 1000
Skipped resamples   : 0

95% BOOTSTRAP CONFIDENCE INTERVALS
Balanced Accuracy : 0.9118 [0.8111, 1.0000]
Macro-F1          : 0.8838 [0.7865, 1.0000]
AUROC             : 0.9864 [0.9424, 1.0000]

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/probes/qwen_v2_binary_bootstrap_ci.json

✓ Scenario-level resampling
✓ 1000 bootstrap resamples
✓ 95% percentile CIs
✓ No retraining
✓ Test predictions reused exactly


In [48]:
# ============================================================
# Qwen v2.0 — MAJORITY CLASS BASELINE
# Binary task
# ============================================================

import os
import json
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("MAJORITY CLASS BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# Training-set class counts
# ------------------------------------------------------------
train_class_0 = int(np.sum(y_binary[train_idx] == 0))
train_class_1 = int(np.sum(y_binary[train_idx] == 1))

print(f"Training class 0 (non_irreversible): {train_class_0}")
print(f"Training class 1 (irreversible)    : {train_class_1}")

if train_class_0 >= train_class_1:
    majority_class = 0
else:
    majority_class = 1

majority_class_name = (
    "non_irreversible"
    if majority_class == 0
    else "irreversible"
)

print()
print(f"Majority class: {majority_class} ({majority_class_name})")

# ------------------------------------------------------------
# Predict majority class for every test record
# ------------------------------------------------------------
y_test_majority = np.full(
    len(test_idx),
    majority_class,
    dtype=int
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
majority_accuracy = accuracy_score(
    y_test_final,
    y_test_majority
)

majority_balanced_accuracy = balanced_accuracy_score(
    y_test_final,
    y_test_majority
)

majority_macro_f1 = f1_score(
    y_test_final,
    y_test_majority,
    average="macro",
    zero_division=0
)

majority_precision = precision_score(
    y_test_final,
    y_test_majority,
    average=None,
    labels=[0, 1],
    zero_division=0
)

majority_recall = recall_score(
    y_test_final,
    y_test_majority,
    average=None,
    labels=[0, 1],
    zero_division=0
)

majority_f1 = f1_score(
    y_test_final,
    y_test_majority,
    average=None,
    labels=[0, 1],
    zero_division=0
)

majority_confusion = confusion_matrix(
    y_test_final,
    y_test_majority,
    labels=[0, 1]
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------
print()
print("=" * 70)
print("MAJORITY BASELINE RESULTS")
print("=" * 70)

print(f"Test records      : {len(test_idx)}")
print(f"Accuracy          : {majority_accuracy:.4f}")
print(f"Balanced Accuracy : {majority_balanced_accuracy:.4f}")
print(f"Macro-F1          : {majority_macro_f1:.4f}")

print()
print("Per-class metrics:")

print(
    f"Class 0 — non_irreversible | "
    f"Precision={majority_precision[0]:.4f} | "
    f"Recall={majority_recall[0]:.4f} | "
    f"F1={majority_f1[0]:.4f}"
)

print(
    f"Class 1 — irreversible     | "
    f"Precision={majority_precision[1]:.4f} | "
    f"Recall={majority_recall[1]:.4f} | "
    f"F1={majority_f1[1]:.4f}"
)

print()
print("Confusion Matrix:")
print("(rows = true, columns = predicted)")
print(majority_confusion)

# ------------------------------------------------------------
# Save baseline result
# ------------------------------------------------------------
majority_path = os.path.join(
    WORK_DIR,
    "baselines",
    "qwen_v2_binary_majority_baseline.json"
)

majority_results = {
    "experiment": "qwen_v2_binary_majority_baseline",

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "train": {
        "n": int(len(train_idx)),
        "class_0": train_class_0,
        "class_1": train_class_1
    },

    "test": {
        "n": int(len(test_idx))
    },

    "majority_class": {
        "label": int(majority_class),
        "name": majority_class_name
    },

    "metrics": {
        "accuracy": float(majority_accuracy),
        "balanced_accuracy": float(majority_balanced_accuracy),
        "macro_f1": float(majority_macro_f1),

        "per_class": {
            "non_irreversible": {
                "precision": float(majority_precision[0]),
                "recall": float(majority_recall[0]),
                "f1": float(majority_f1[0])
            },
            "irreversible": {
                "precision": float(majority_precision[1]),
                "recall": float(majority_recall[1]),
                "f1": float(majority_f1[1])
            }
        },

        "confusion_matrix": majority_confusion.tolist()
    }
}

with open(majority_path, "w") as f:
    json.dump(
        majority_results,
        f,
        indent=2
    )

print()
print("Saved:")
print(majority_path)

print()
print("✓ Majority class determined from TRAIN only")
print("✓ No test fitting or selection")
print("✓ Baseline evaluated on frozen test set")

MAJORITY CLASS BASELINE
Training class 0 (non_irreversible): 1285
Training class 1 (irreversible)    : 424

Majority class: 0 (non_irreversible)

MAJORITY BASELINE RESULTS
Test records      : 425
Accuracy          : 0.6400
Balanced Accuracy : 0.5000
Macro-F1          : 0.3902

Per-class metrics:
Class 0 — non_irreversible | Precision=0.6400 | Recall=1.0000 | F1=0.7805
Class 1 — irreversible     | Precision=0.0000 | Recall=0.0000 | F1=0.0000

Confusion Matrix:
(rows = true, columns = predicted)
[[272   0]
 [153   0]]

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/baselines/qwen_v2_binary_majority_baseline.json

✓ Majority class determined from TRAIN only
✓ No test fitting or selection
✓ Baseline evaluated on frozen test set


In [49]:
# ============================================================
# Qwen v2.0 — BOW ACTION_TEXT BASELINE
# Binary task
#
# Features: action_text only
# Model: TF-IDF/BOW -> Logistic Regression
# Selection: training-only GroupKFold by scenario_id
# ============================================================

import os
import json
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("BOW ACTION_TEXT BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
BOW_SEED = 42
N_SPLITS = 5

# C values selected ONLY inside training
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]

print("Feature source : action_text")
print("Vectorizer     : TF-IDF")
print("Classifier     : L2 Logistic Regression")
print("CV             : 5-fold GroupKFold")
print("Groups         : scenario_id")
print()

# ------------------------------------------------------------
# Prepare action text
# ------------------------------------------------------------
action_text_all = np.array(
    [str(record["action_text"]) for record in records],
    dtype=object
)

groups_train = scenario_ids[train_idx]

X_text_train = action_text_all[train_idx]
X_text_test = action_text_all[test_idx]

y_text_train = y_binary[train_idx]
y_text_test = y_binary[test_idx]

print(f"Training records: {len(X_text_train)}")
print(f"Test records    : {len(X_text_test)}")
print()

# ------------------------------------------------------------
# Inner CV for C selection
# ------------------------------------------------------------
print("=" * 70)
print("INNER CV — SELECTING C")
print("=" * 70)

group_kfold = GroupKFold(
    n_splits=N_SPLITS
)

c_results = []

start_time = time.time()

for C in C_GRID:

    fold_scores = []

    for fold, (tr_rel, val_rel) in enumerate(
        group_kfold.split(
            X_text_train,
            y_text_train,
            groups=groups_train
        ),
        start=1
    ):

        X_tr = X_text_train[tr_rel]
        X_val = X_text_train[val_rel]

        y_tr = y_text_train[tr_rel]
        y_val = y_text_train[val_rel]

        bow_model = Pipeline([
            (
                "vectorizer",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    sublinear_tf=True
                )
            ),
            (
                "logreg",
                LogisticRegression(
                    penalty="l2",
                    C=C,
                    class_weight="balanced",
                    solver="liblinear",
                    max_iter=1000,
                    random_state=BOW_SEED
                )
            )
        ])

        bow_model.fit(
            X_tr,
            y_tr
        )

        val_pred = bow_model.predict(
            X_val
        )

        score = balanced_accuracy_score(
            y_val,
            val_pred
        )

        fold_scores.append(score)

    mean_score = float(
        np.mean(fold_scores)
    )

    std_score = float(
        np.std(fold_scores)
    )

    c_results.append({
        "C": float(C),
        "mean_balanced_accuracy": mean_score,
        "std_balanced_accuracy": std_score,
        "fold_scores": [
            float(x)
            for x in fold_scores
        ]
    })

    print(
        f"C={C:<6} "
        f"BA={mean_score:.4f} "
        f"SD={std_score:.4f}"
    )

# ------------------------------------------------------------
# Select best C
# ------------------------------------------------------------
best_c_result = max(
    c_results,
    key=lambda x: x["mean_balanced_accuracy"]
)

BEST_BOW_C = best_c_result["C"]

print()
print(
    f"Selected C = {BEST_BOW_C}"
)

print(
    f"Inner-CV BA = "
    f"{best_c_result['mean_balanced_accuracy']:.4f}"
)

print(
    f"Inner-CV SD = "
    f"{best_c_result['std_balanced_accuracy']:.4f}"
)

print()

# ------------------------------------------------------------
# Final BOW model
# Fit ONLY on all training records
# ------------------------------------------------------------
print("=" * 70)
print("TRAINING FINAL BOW MODEL")
print("=" * 70)

final_bow = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "logreg",
        LogisticRegression(
            penalty="l2",
            C=BEST_BOW_C,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=BOW_SEED
        )
    )
])

final_bow.fit(
    X_text_train,
    y_text_train
)

print("Final BOW model trained.")
print()

# ------------------------------------------------------------
# Test evaluation
# ------------------------------------------------------------
print("=" * 70)
print("BOW ACTION_TEXT TEST RESULTS")
print("=" * 70)

bow_test_pred = final_bow.predict(
    X_text_test
)

bow_test_prob = final_bow.predict_proba(
    X_text_test
)[:, 1]

bow_accuracy = accuracy_score(
    y_text_test,
    bow_test_pred
)

bow_balanced_accuracy = balanced_accuracy_score(
    y_text_test,
    bow_test_pred
)

bow_macro_f1 = f1_score(
    y_text_test,
    bow_test_pred,
    average="macro",
    zero_division=0
)

bow_precision = precision_score(
    y_text_test,
    bow_test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

bow_recall = recall_score(
    y_text_test,
    bow_test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

bow_f1 = f1_score(
    y_text_test,
    bow_test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

bow_confusion = confusion_matrix(
    y_text_test,
    bow_test_pred,
    labels=[0, 1]
)

bow_auroc = roc_auc_score(
    y_text_test,
    bow_test_prob
)

print(f"Test records      : {len(test_idx)}")
print(f"Accuracy          : {bow_accuracy:.4f}")
print(f"Balanced Accuracy : {bow_balanced_accuracy:.4f}")
print(f"Macro-F1          : {bow_macro_f1:.4f}")
print(f"AUROC             : {bow_auroc:.4f}")

print()
print("Per-class metrics:")

print(
    f"Class 0 — non_irreversible | "
    f"Precision={bow_precision[0]:.4f} | "
    f"Recall={bow_recall[0]:.4f} | "
    f"F1={bow_f1[0]:.4f}"
)

print(
    f"Class 1 — irreversible     | "
    f"Precision={bow_precision[1]:.4f} | "
    f"Recall={bow_recall[1]:.4f} | "
    f"F1={bow_f1[1]:.4f}"
)

print()
print("Confusion Matrix:")
print("(rows = true, columns = predicted)")
print(bow_confusion)

print()
print(
    f"Total runtime: "
    f"{time.time() - start_time:.1f}s"
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
bow_path = os.path.join(
    WORK_DIR,
    "baselines",
    "qwen_v2_binary_bow_action_text.json"
)

bow_results = {
    "experiment": "qwen_v2_binary_bow_action_text",

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "features": {
        "source": "action_text",
        "vectorizer": "TfidfVectorizer",
        "lowercase": True,
        "ngram_range": [1, 2],
        "min_df": 2,
        "sublinear_tf": True
    },

    "selection": {
        "method": "GroupKFold",
        "n_splits": N_SPLITS,
        "group": "scenario_id",
        "C_grid": C_GRID,
        "selected_C": BEST_BOW_C,
        "inner_cv_balanced_accuracy":
            best_c_result["mean_balanced_accuracy"],
        "inner_cv_sd":
            best_c_result["std_balanced_accuracy"]
    },

    "classifier": {
        "penalty": "l2",
        "C": BEST_BOW_C,
        "class_weight": "balanced",
        "solver": "liblinear",
        "max_iter": 1000,
        "random_state": BOW_SEED
    },

    "test": {
        "n": int(len(test_idx))
    },

    "metrics": {
        "accuracy": float(bow_accuracy),
        "balanced_accuracy": float(bow_balanced_accuracy),
        "macro_f1": float(bow_macro_f1),
        "auroc": float(bow_auroc),

        "per_class": {
            "non_irreversible": {
                "precision": float(bow_precision[0]),
                "recall": float(bow_recall[0]),
                "f1": float(bow_f1[0])
            },

            "irreversible": {
                "precision": float(bow_precision[1]),
                "recall": float(bow_recall[1]),
                "f1": float(bow_f1[1])
            }
        },

        "confusion_matrix":
            bow_confusion.tolist()
    }
}

with open(bow_path, "w") as f:
    json.dump(
        bow_results,
        f,
        indent=2
    )

print()
print("Saved:")
print(bow_path)

print()
print("✓ BOW uses action_text only")
print("✓ C selected using training-only GroupKFold")
print("✓ Groups = scenario_id")
print("✓ Test evaluated only after C selection")

BOW ACTION_TEXT BASELINE
Feature source : action_text
Vectorizer     : TF-IDF
Classifier     : L2 Logistic Regression
CV             : 5-fold GroupKFold
Groups         : scenario_id

Training records: 1709
Test records    : 425

INNER CV — SELECTING C
C=0.001  BA=0.6530 SD=0.1035
C=0.01   BA=0.6612 SD=0.1088
C=0.1    BA=0.6478 SD=0.1049
C=1.0    BA=0.6519 SD=0.0516
C=10.0   BA=0.6639 SD=0.0464

Selected C = 10.0
Inner-CV BA = 0.6639
Inner-CV SD = 0.0464

TRAINING FINAL BOW MODEL
Final BOW model trained.

BOW ACTION_TEXT TEST RESULTS
Test records      : 425
Accuracy          : 0.7600
Balanced Accuracy : 0.7439
Macro-F1          : 0.7417
AUROC             : 0.8710

Per-class metrics:
Class 0 — non_irreversible | Precision=0.8195 | Recall=0.8015 | F1=0.8104
Class 1 — irreversible     | Precision=0.6604 | Recall=0.6863 | F1=0.6731

Confusion Matrix:
(rows = true, columns = predicted)
[[218  54]
 [ 48 105]]

Total runtime: 1.0s

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/baselines/qwen_v2_bina

In [50]:
# ============================================================
# Qwen v2.0 — BOW STATE_TEXT + ACTION_TEXT BASELINE
# Binary task
#
# Features: state_text + action_text
# Model: TF-IDF -> L2 Logistic Regression
# Selection: training-only GroupKFold by scenario_id
# ============================================================

import os
import json
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("BOW STATE_TEXT + ACTION_TEXT BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
BOW_STATE_SEED = 42
N_SPLITS = 5

C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]

print("Feature source : state_text + action_text")
print("Vectorizer     : TF-IDF")
print("Classifier     : L2 Logistic Regression")
print("CV             : 5-fold GroupKFold")
print("Groups         : scenario_id")
print()

# ------------------------------------------------------------
# Build combined text
# ------------------------------------------------------------
combined_text_all = np.array(
    [
        str(record["state_text"]) +
        " " +
        str(record["action_text"])
        for record in records
    ],
    dtype=object
)

groups_train = scenario_ids[train_idx]

X_text_train = combined_text_all[train_idx]
X_text_test = combined_text_all[test_idx]

y_text_train = y_binary[train_idx]
y_text_test = y_binary[test_idx]

print(f"Training records: {len(X_text_train)}")
print(f"Test records    : {len(X_text_test)}")
print()

# ------------------------------------------------------------
# Inner GroupKFold for C selection
# ------------------------------------------------------------
print("=" * 70)
print("INNER CV — SELECTING C")
print("=" * 70)

group_kfold = GroupKFold(
    n_splits=N_SPLITS
)

c_results = []

start_time = time.time()

for C in C_GRID:

    fold_scores = []

    for fold, (tr_rel, val_rel) in enumerate(
        group_kfold.split(
            X_text_train,
            y_text_train,
            groups=groups_train
        ),
        start=1
    ):

        X_tr = X_text_train[tr_rel]
        X_val = X_text_train[val_rel]

        y_tr = y_text_train[tr_rel]
        y_val = y_text_train[val_rel]

        bow_model = Pipeline([
            (
                "vectorizer",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    sublinear_tf=True
                )
            ),
            (
                "logreg",
                LogisticRegression(
                    penalty="l2",
                    C=C,
                    class_weight="balanced",
                    solver="liblinear",
                    max_iter=1000,
                    random_state=BOW_STATE_SEED
                )
            )
        ])

        bow_model.fit(
            X_tr,
            y_tr
        )

        val_pred = bow_model.predict(
            X_val
        )

        score = balanced_accuracy_score(
            y_val,
            val_pred
        )

        fold_scores.append(score)

    mean_score = float(
        np.mean(fold_scores)
    )

    std_score = float(
        np.std(fold_scores)
    )

    c_results.append({
        "C": float(C),
        "mean_balanced_accuracy": mean_score,
        "std_balanced_accuracy": std_score,
        "fold_scores": [
            float(x)
            for x in fold_scores
        ]
    })

    print(
        f"C={C:<6} "
        f"BA={mean_score:.4f} "
        f"SD={std_score:.4f}"
    )

# ------------------------------------------------------------
# Select best C
# ------------------------------------------------------------
best_c_result = max(
    c_results,
    key=lambda x: x["mean_balanced_accuracy"]
)

BEST_BOW_STATE_C = best_c_result["C"]

print()
print(f"Selected C = {BEST_BOW_STATE_C}")

print(
    f"Inner-CV BA = "
    f"{best_c_result['mean_balanced_accuracy']:.4f}"
)

print(
    f"Inner-CV SD = "
    f"{best_c_result['std_balanced_accuracy']:.4f}"
)

print()

# ------------------------------------------------------------
# Train final model on ALL training records
# ------------------------------------------------------------
print("=" * 70)
print("TRAINING FINAL BOW STATE+ACTION MODEL")
print("=" * 70)

final_bow_state = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "logreg",
        LogisticRegression(
            penalty="l2",
            C=BEST_BOW_STATE_C,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=BOW_STATE_SEED
        )
    )
])

final_bow_state.fit(
    X_text_train,
    y_text_train
)

print("Final BOW state+action model trained.")
print()

# ------------------------------------------------------------
# Test evaluation
# ------------------------------------------------------------
print("=" * 70)
print("BOW STATE_TEXT + ACTION_TEXT TEST RESULTS")
print("=" * 70)

bow_state_test_pred = final_bow_state.predict(
    X_text_test
)

bow_state_test_prob = final_bow_state.predict_proba(
    X_text_test
)[:, 1]

bow_state_accuracy = accuracy_score(
    y_text_test,
    bow_state_test_pred
)

bow_state_balanced_accuracy = balanced_accuracy_score(
    y_text_test,
    bow_state_test_pred
)

bow_state_macro_f1 = f1_score(
    y_text_test,
    bow_state_test_pred,
    average="macro",
    zero_division=0
)

bow_state_precision = precision_score(
    y_text_test,
    bow_state_test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

bow_state_recall = recall_score(
    y_text_test,
    bow_state_test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

bow_state_f1 = f1_score(
    y_text_test,
    bow_state_test_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

bow_state_confusion = confusion_matrix(
    y_text_test,
    bow_state_test_pred,
    labels=[0, 1]
)

bow_state_auroc = roc_auc_score(
    y_text_test,
    bow_state_test_prob
)

print(f"Test records      : {len(test_idx)}")
print(f"Accuracy          : {bow_state_accuracy:.4f}")
print(f"Balanced Accuracy : {bow_state_balanced_accuracy:.4f}")
print(f"Macro-F1          : {bow_state_macro_f1:.4f}")
print(f"AUROC             : {bow_state_auroc:.4f}")

print()
print("Per-class metrics:")

print(
    f"Class 0 — non_irreversible | "
    f"Precision={bow_state_precision[0]:.4f} | "
    f"Recall={bow_state_recall[0]:.4f} | "
    f"F1={bow_state_f1[0]:.4f}"
)

print(
    f"Class 1 — irreversible     | "
    f"Precision={bow_state_precision[1]:.4f} | "
    f"Recall={bow_state_recall[1]:.4f} | "
    f"F1={bow_state_f1[1]:.4f}"
)

print()
print("Confusion Matrix:")
print("(rows = true, columns = predicted)")
print(bow_state_confusion)

print()
print(f"Total runtime: {time.time() - start_time:.1f}s")

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
bow_state_path = os.path.join(
    WORK_DIR,
    "baselines",
    "qwen_v2_binary_bow_state_action.json"
)

bow_state_results = {
    "experiment": "qwen_v2_binary_bow_state_action",

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "features": {
        "source": "state_text + action_text",
        "vectorizer": "TfidfVectorizer",
        "lowercase": True,
        "ngram_range": [1, 2],
        "min_df": 2,
        "sublinear_tf": True
    },

    "selection": {
        "method": "GroupKFold",
        "n_splits": N_SPLITS,
        "group": "scenario_id",
        "C_grid": C_GRID,
        "selected_C": BEST_BOW_STATE_C,
        "inner_cv_balanced_accuracy":
            best_c_result["mean_balanced_accuracy"],
        "inner_cv_sd":
            best_c_result["std_balanced_accuracy"]
    },

    "classifier": {
        "penalty": "l2",
        "C": BEST_BOW_STATE_C,
        "class_weight": "balanced",
        "solver": "liblinear",
        "max_iter": 1000,
        "random_state": BOW_STATE_SEED
    },

    "test": {
        "n": int(len(test_idx))
    },

    "metrics": {
        "accuracy": float(bow_state_accuracy),
        "balanced_accuracy": float(bow_state_balanced_accuracy),
        "macro_f1": float(bow_state_macro_f1),
        "auroc": float(bow_state_auroc),

        "per_class": {
            "non_irreversible": {
                "precision": float(bow_state_precision[0]),
                "recall": float(bow_state_recall[0]),
                "f1": float(bow_state_f1[0])
            },

            "irreversible": {
                "precision": float(bow_state_precision[1]),
                "recall": float(bow_state_recall[1]),
                "f1": float(bow_state_f1[1])
            }
        },

        "confusion_matrix":
            bow_state_confusion.tolist()
    }
}

with open(bow_state_path, "w") as f:
    json.dump(
        bow_state_results,
        f,
        indent=2
    )

print()
print("Saved:")
print(bow_state_path)

print()
print("✓ Features use state_text + action_text")
print("✓ C selected using training-only GroupKFold")
print("✓ Groups = scenario_id")
print("✓ Test evaluated only after C selection")

BOW STATE_TEXT + ACTION_TEXT BASELINE
Feature source : state_text + action_text
Vectorizer     : TF-IDF
Classifier     : L2 Logistic Regression
CV             : 5-fold GroupKFold
Groups         : scenario_id

Training records: 1709
Test records    : 425

INNER CV — SELECTING C
C=0.001  BA=0.6341 SD=0.0925
C=0.01   BA=0.6491 SD=0.0859
C=0.1    BA=0.6733 SD=0.0833
C=1.0    BA=0.6959 SD=0.0543
C=10.0   BA=0.6708 SD=0.0395

Selected C = 1.0
Inner-CV BA = 0.6959
Inner-CV SD = 0.0543

TRAINING FINAL BOW STATE+ACTION MODEL
Final BOW state+action model trained.

BOW STATE_TEXT + ACTION_TEXT TEST RESULTS
Test records      : 425
Accuracy          : 0.8424
Balanced Accuracy : 0.8440
Macro-F1          : 0.8335
AUROC             : 0.9400

Per-class metrics:
Class 0 — non_irreversible | Precision=0.9084 | Recall=0.8382 | F1=0.8719
Class 1 — irreversible     | Precision=0.7471 | Recall=0.8497 | F1=0.7951

Confusion Matrix:
(rows = true, columns = predicted)
[[228  44]
 [ 23 130]]

Total runtime: 5.4s

In [51]:
# ============================================================
# Qwen v2.0 — SHUFFLED-LABEL CONTROL
# Binary task
#
# Purpose:
# Test whether probe capacity alone can produce high scores
# when the training labels contain no meaningful signal.
#
# Same:
#   - hidden representation
#   - layer 18
#   - C = 0.01
#   - StandardScaler
#   - L2 LogisticRegression
#   - class_weight="balanced"
#
# Only training labels are randomly shuffled.
# ============================================================

import os
import json
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("SHUFFLED-LABEL CONTROL")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
SHUFFLE_SEED = 42
SELECTED_LAYER = 18
SELECTED_C = 0.01

print("Representation : last_action_token")
print(f"Layer          : {SELECTED_LAYER}")
print(f"C              : {SELECTED_C}")
print("Label operation: shuffle TRAIN labels only")
print()

# ------------------------------------------------------------
# Load selected hidden representation
# ------------------------------------------------------------
X_train_shuffle = X_all[
    train_idx,
    SELECTED_LAYER,
    :
]

X_test_shuffle = X_all[
    test_idx,
    SELECTED_LAYER,
    :
]

# Original labels
y_train_original = y_binary[train_idx]
y_test_original = y_binary[test_idx]

# ------------------------------------------------------------
# Shuffle training labels
# ------------------------------------------------------------
rng_shuffle = np.random.default_rng(
    SHUFFLE_SEED
)

y_train_shuffled = rng_shuffle.permutation(
    y_train_original
)

print("Original training label counts:")
print(
    f"  class 0 = "
    f"{np.sum(y_train_original == 0)}"
)
print(
    f"  class 1 = "
    f"{np.sum(y_train_original == 1)}"
)

print()
print("Shuffled training label counts:")
print(
    f"  class 0 = "
    f"{np.sum(y_train_shuffled == 0)}"
)
print(
    f"  class 1 = "
    f"{np.sum(y_train_shuffled == 1)}"
)

# Verify that the class counts remain unchanged
assert (
    np.sum(y_train_original == 0)
    ==
    np.sum(y_train_shuffled == 0)
)

assert (
    np.sum(y_train_original == 1)
    ==
    np.sum(y_train_shuffled == 1)
)

# ------------------------------------------------------------
# Train probe with shuffled labels
# ------------------------------------------------------------
print()
print("=" * 70)
print("TRAINING SHUFFLED-LABEL PROBE")
print("=" * 70)

start_time = time.time()

shuffled_probe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logreg",
        LogisticRegression(
            penalty="l2",
            C=SELECTED_C,
            class_weight="balanced",
            solver="saga",
            max_iter=500,
            tol=1e-2,
            random_state=42
        )
    )
])

shuffled_probe.fit(
    X_train_shuffle,
    y_train_shuffled
)

print(
    f"Training complete in "
    f"{time.time() - start_time:.1f}s"
)

# ------------------------------------------------------------
# Evaluate on ORIGINAL test labels
# ------------------------------------------------------------
print()
print("=" * 70)
print("SHUFFLED-LABEL TEST RESULTS")
print("=" * 70)

shuffled_pred = shuffled_probe.predict(
    X_test_shuffle
)

shuffled_prob = shuffled_probe.predict_proba(
    X_test_shuffle
)[:, 1]

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
shuffled_accuracy = accuracy_score(
    y_test_original,
    shuffled_pred
)

shuffled_balanced_accuracy = balanced_accuracy_score(
    y_test_original,
    shuffled_pred
)

shuffled_macro_f1 = f1_score(
    y_test_original,
    shuffled_pred,
    average="macro",
    zero_division=0
)

shuffled_precision = precision_score(
    y_test_original,
    shuffled_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

shuffled_recall = recall_score(
    y_test_original,
    shuffled_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

shuffled_f1 = f1_score(
    y_test_original,
    shuffled_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

shuffled_confusion = confusion_matrix(
    y_test_original,
    shuffled_pred,
    labels=[0, 1]
)

shuffled_auroc = roc_auc_score(
    y_test_original,
    shuffled_prob
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------
print()
print(f"Test records      : {len(test_idx)}")
print(f"Accuracy          : {shuffled_accuracy:.4f}")
print(
    f"Balanced Accuracy : "
    f"{shuffled_balanced_accuracy:.4f}"
)
print(
    f"Macro-F1          : "
    f"{shuffled_macro_f1:.4f}"
)
print(
    f"AUROC             : "
    f"{shuffled_auroc:.4f}"
)

print()
print("Per-class metrics:")

print(
    f"Class 0 — non_irreversible | "
    f"Precision={shuffled_precision[0]:.4f} | "
    f"Recall={shuffled_recall[0]:.4f} | "
    f"F1={shuffled_f1[0]:.4f}"
)

print(
    f"Class 1 — irreversible     | "
    f"Precision={shuffled_precision[1]:.4f} | "
    f"Recall={shuffled_recall[1]:.4f} | "
    f"F1={shuffled_f1[1]:.4f}"
)

print()
print("Confusion Matrix:")
print("(rows = true, columns = predicted)")
print(shuffled_confusion)

# ------------------------------------------------------------
# Compare against real probe
# ------------------------------------------------------------
print()
print("=" * 70)
print("REAL PROBE vs SHUFFLED-LABEL CONTROL")
print("=" * 70)

print(
    f"Real probe BA      : "
    f"{balanced_accuracy:.4f}"
)

print(
    f"Shuffled-label BA  : "
    f"{shuffled_balanced_accuracy:.4f}"
)

print(
    f"BA difference      : "
    f"{balanced_accuracy - shuffled_balanced_accuracy:.4f}"
)

print()

print(
    f"Real probe Macro-F1     : "
    f"{macro_f1:.4f}"
)

print(
    f"Shuffled-label Macro-F1 : "
    f"{shuffled_macro_f1:.4f}"
)

print(
    f"Macro-F1 difference     : "
    f"{macro_f1 - shuffled_macro_f1:.4f}"
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
shuffle_path = os.path.join(
    WORK_DIR,
    "baselines",
    "qwen_v2_binary_shuffled_label_control.json"
)

shuffle_results = {
    "experiment": "qwen_v2_binary_shuffled_label_control",

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "representation": {
        "type": "last_action_token",
        "layer": SELECTED_LAYER
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": SELECTED_C,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "label_shuffle": {
        "scope": "training_labels_only",
        "seed": SHUFFLE_SEED,
        "test_labels_shuffled": False
    },

    "train": {
        "n": int(len(train_idx)),
        "original_class_0": int(
            np.sum(y_train_original == 0)
        ),
        "original_class_1": int(
            np.sum(y_train_original == 1)
        ),
        "shuffled_class_0": int(
            np.sum(y_train_shuffled == 0)
        ),
        "shuffled_class_1": int(
            np.sum(y_train_shuffled == 1)
        )
    },

    "test": {
        "n": int(len(test_idx))
    },

    "metrics": {
        "accuracy": float(shuffled_accuracy),
        "balanced_accuracy": float(
            shuffled_balanced_accuracy
        ),
        "macro_f1": float(
            shuffled_macro_f1
        ),
        "auroc": float(
            shuffled_auroc
        ),

        "per_class": {
            "non_irreversible": {
                "precision": float(
                    shuffled_precision[0]
                ),
                "recall": float(
                    shuffled_recall[0]
                ),
                "f1": float(
                    shuffled_f1[0]
                )
            },
            "irreversible": {
                "precision": float(
                    shuffled_precision[1]
                ),
                "recall": float(
                    shuffled_recall[1]
                ),
                "f1": float(
                    shuffled_f1[1]
                )
            }
        },

        "confusion_matrix":
            shuffled_confusion.tolist()
    }
}

with open(shuffle_path, "w") as f:
    json.dump(
        shuffle_results,
        f,
        indent=2
    )

print()
print("Saved:")
print(shuffle_path)

print()
print("✓ Training labels shuffled only")
print("✓ Test labels remain untouched")
print("✓ Same selected layer and C")
print("✓ Same probe architecture")
print("✓ No test fitting or selection")

SHUFFLED-LABEL CONTROL
Representation : last_action_token
Layer          : 18
C              : 0.01
Label operation: shuffle TRAIN labels only

Original training label counts:
  class 0 = 1285
  class 1 = 424

Shuffled training label counts:
  class 0 = 1285
  class 1 = 424

TRAINING SHUFFLED-LABEL PROBE
Training complete in 5.1s

SHUFFLED-LABEL TEST RESULTS

Test records      : 425
Accuracy          : 0.5035
Balanced Accuracy : 0.5263
Macro-F1          : 0.5014
AUROC             : 0.5510

Per-class metrics:
Class 0 — non_irreversible | Precision=0.6685 | Recall=0.4449 | F1=0.5342
Class 1 — irreversible     | Precision=0.3811 | Recall=0.6078 | F1=0.4685

Confusion Matrix:
(rows = true, columns = predicted)
[[121 151]
 [ 60  93]]

REAL PROBE vs SHUFFLED-LABEL CONTROL
Real probe BA      : 0.9118
Shuffled-label BA  : 0.5263
BA difference      : 0.3854

Real probe Macro-F1     : 0.8838
Shuffled-label Macro-F1 : 0.5014
Macro-F1 difference     : 0.3825

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v

In [52]:
# ============================================================
# Qwen v2.0 — FIXED RANDOM LABEL PER ACTION_VERB CONTROL
# Binary task
#
# Each action_verb receives ONE random binary label.
# The mapping is fixed across train and test.
#
# Purpose:
# Measure how much performance can be obtained from
# action-verb-associated structure alone.
# ============================================================

import os
import json
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("FIXED RANDOM LABEL PER ACTION_VERB CONTROL")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
ACTION_VERB_SEED = 42

SELECTED_LAYER = 18
SELECTED_C = 0.01

print("Representation : last_action_token")
print(f"Layer          : {SELECTED_LAYER}")
print(f"C              : {SELECTED_C}")
print("Random label   : fixed per action_verb")
print(f"Seed           : {ACTION_VERB_SEED}")
print()

# ------------------------------------------------------------
# Get all action verbs
# ------------------------------------------------------------
action_verbs_all = np.array(
    [
        str(record["action_verb"])
        for record in records
    ],
    dtype=object
)

unique_action_verbs = sorted(
    np.unique(action_verbs_all).tolist()
)

print(
    f"Unique action_verbs: "
    f"{len(unique_action_verbs)}"
)

# ------------------------------------------------------------
# Create fixed random mapping
# ------------------------------------------------------------
rng_action_verb = np.random.default_rng(
    ACTION_VERB_SEED
)

action_verb_labels = {}

for verb in unique_action_verbs:
    action_verb_labels[verb] = int(
        rng_action_verb.integers(
            0,
            2
        )
    )

print()
print("Example action_verb → random-label mappings:")

for verb in unique_action_verbs[:20]:
    print(
        f"  {verb} -> "
        f"{action_verb_labels[verb]}"
    )

if len(unique_action_verbs) > 20:
    print(
        f"  ... and "
        f"{len(unique_action_verbs) - 20} more"
    )

# ------------------------------------------------------------
# Construct random-label target
# ------------------------------------------------------------
y_random_verb = np.array(
    [
        action_verb_labels[verb]
        for verb in action_verbs_all
    ],
    dtype=int
)

y_random_train = y_random_verb[train_idx]
y_random_test = y_random_verb[test_idx]

print()
print("Random-label distribution:")

print(
    f"Training class 0: "
    f"{np.sum(y_random_train == 0)}"
)

print(
    f"Training class 1: "
    f"{np.sum(y_random_train == 1)}"
)

print(
    f"Test class 0: "
    f"{np.sum(y_random_test == 0)}"
)

print(
    f"Test class 1: "
    f"{np.sum(y_random_test == 1)}"
)

# ------------------------------------------------------------
# Check every verb has exactly one label
# ------------------------------------------------------------
for verb in unique_action_verbs:

    labels_for_verb = np.unique(
        y_random_verb[
            action_verbs_all == verb
        ]
    )

    assert len(labels_for_verb) == 1

print()
print("✓ Every action_verb has exactly one fixed random label")

# ------------------------------------------------------------
# Load selected hidden representation
# ------------------------------------------------------------
X_train_random = X_all[
    train_idx,
    SELECTED_LAYER,
    :
]

X_test_random = X_all[
    test_idx,
    SELECTED_LAYER,
    :
]

# ------------------------------------------------------------
# Train probe
# ------------------------------------------------------------
print()
print("=" * 70)
print("TRAINING FIXED-RANDOM-VERB PROBE")
print("=" * 70)

start_time = time.time()

random_verb_probe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logreg",
        LogisticRegression(
            penalty="l2",
            C=SELECTED_C,
            class_weight="balanced",
            solver="saga",
            max_iter=500,
            tol=1e-2,
            random_state=42
        )
    )
])

random_verb_probe.fit(
    X_train_random,
    y_random_train
)

print(
    f"Training complete in "
    f"{time.time() - start_time:.1f}s"
)

# ------------------------------------------------------------
# Test evaluation
# ------------------------------------------------------------
print()
print("=" * 70)
print("FIXED-RANDOM-VERB TEST RESULTS")
print("=" * 70)

random_verb_pred = random_verb_probe.predict(
    X_test_random
)

random_verb_prob = random_verb_probe.predict_proba(
    X_test_random
)[:, 1]

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
random_verb_accuracy = accuracy_score(
    y_random_test,
    random_verb_pred
)

random_verb_balanced_accuracy = balanced_accuracy_score(
    y_random_test,
    random_verb_pred
)

random_verb_macro_f1 = f1_score(
    y_random_test,
    random_verb_pred,
    average="macro",
    zero_division=0
)

random_verb_precision = precision_score(
    y_random_test,
    random_verb_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

random_verb_recall = recall_score(
    y_random_test,
    random_verb_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

random_verb_f1 = f1_score(
    y_random_test,
    random_verb_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

random_verb_confusion = confusion_matrix(
    y_random_test,
    random_verb_pred,
    labels=[0, 1]
)

random_verb_auroc = roc_auc_score(
    y_random_test,
    random_verb_prob
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------
print()
print(
    f"Test records      : "
    f"{len(test_idx)}"
)

print(
    f"Accuracy          : "
    f"{random_verb_accuracy:.4f}"
)

print(
    f"Balanced Accuracy : "
    f"{random_verb_balanced_accuracy:.4f}"
)

print(
    f"Macro-F1          : "
    f"{random_verb_macro_f1:.4f}"
)

print(
    f"AUROC             : "
    f"{random_verb_auroc:.4f}"
)

print()
print("Per-class metrics:")

print(
    f"Class 0 — random label 0 | "
    f"Precision={random_verb_precision[0]:.4f} | "
    f"Recall={random_verb_recall[0]:.4f} | "
    f"F1={random_verb_f1[0]:.4f}"
)

print(
    f"Class 1 — random label 1 | "
    f"Precision={random_verb_precision[1]:.4f} | "
    f"Recall={random_verb_recall[1]:.4f} | "
    f"F1={random_verb_f1[1]:.4f}"
)

print()
print("Confusion Matrix:")
print("(rows = true random label, columns = predicted)")
print(random_verb_confusion)

# ------------------------------------------------------------
# Compare with real probe
# ------------------------------------------------------------
print()
print("=" * 70)
print("REAL PROBE vs RANDOM ACTION_VERB CONTROL")
print("=" * 70)

print(
    f"Real probe BA          : "
    f"{balanced_accuracy:.4f}"
)

print(
    f"Random-verb BA         : "
    f"{random_verb_balanced_accuracy:.4f}"
)

print(
    f"BA difference          : "
    f"{balanced_accuracy - random_verb_balanced_accuracy:.4f}"
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
random_verb_path = os.path.join(
    WORK_DIR,
    "baselines",
    "qwen_v2_binary_random_action_verb_control.json"
)

random_verb_results = {
    "experiment":
        "qwen_v2_binary_random_action_verb_control",

    "task": {
        "type": "binary",
        "target":
            "fixed_random_label_per_action_verb"
    },

    "representation": {
        "type": "last_action_token",
        "layer": SELECTED_LAYER
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": SELECTED_C,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "random_mapping": {
        "seed": ACTION_VERB_SEED,
        "n_action_verbs":
            len(unique_action_verbs),
        "mapping":
            action_verb_labels
    },

    "train": {
        "n": int(len(train_idx)),
        "class_0": int(
            np.sum(y_random_train == 0)
        ),
        "class_1": int(
            np.sum(y_random_train == 1)
        )
    },

    "test": {
        "n": int(len(test_idx)),
        "class_0": int(
            np.sum(y_random_test == 0)
        ),
        "class_1": int(
            np.sum(y_random_test == 1)
        )
    },

    "metrics": {
        "accuracy": float(
            random_verb_accuracy
        ),
        "balanced_accuracy": float(
            random_verb_balanced_accuracy
        ),
        "macro_f1": float(
            random_verb_macro_f1
        ),
        "auroc": float(
            random_verb_auroc
        ),

        "per_class": {
            "random_label_0": {
                "precision": float(
                    random_verb_precision[0]
                ),
                "recall": float(
                    random_verb_recall[0]
                ),
                "f1": float(
                    random_verb_f1[0]
                )
            },

            "random_label_1": {
                "precision": float(
                    random_verb_precision[1]
                ),
                "recall": float(
                    random_verb_recall[1]
                ),
                "f1": float(
                    random_verb_f1[1]
                )
            }
        },

        "confusion_matrix":
            random_verb_confusion.tolist()
    }
}

with open(random_verb_path, "w") as f:
    json.dump(
        random_verb_results,
        f,
        indent=2
    )

print()
print("Saved:")
print(random_verb_path)

print()
print("✓ One random label fixed per action_verb")
print("✓ Same mapping used for train and test")
print("✓ Same hidden representation")
print("✓ Same classifier")
print("✓ Test labels were not used for fitting")

FIXED RANDOM LABEL PER ACTION_VERB CONTROL
Representation : last_action_token
Layer          : 18
C              : 0.01
Random label   : fixed per action_verb
Seed           : 42

Unique action_verbs: 13

Example action_verb → random-label mappings:
  connect -> 0
  create_share_link -> 1
  delete_file -> 1
  edit_file -> 0
  move -> 0
  open -> 1
  restore_backup -> 0
  restore_file -> 1
  revert_file -> 0
  revoke_share_link -> 0
  send_email -> 1
  transfer_money -> 1
  use -> 1

Random-label distribution:
Training class 0: 529
Training class 1: 1180
Test class 0: 262
Test class 1: 163

✓ Every action_verb has exactly one fixed random label

TRAINING FIXED-RANDOM-VERB PROBE
Training complete in 2.3s

FIXED-RANDOM-VERB TEST RESULTS

Test records      : 425
Accuracy          : 1.0000
Balanced Accuracy : 1.0000
Macro-F1          : 1.0000
AUROC             : 1.0000

Per-class metrics:
Class 0 — random label 0 | Precision=1.0000 | Recall=1.0000 | F1=1.0000
Class 1 — random label 1 | Prec

In [53]:
# ============================================================
# Qwen v2.0 — LAYER-0 EMBEDDING BASELINE
# Binary task
#
# Purpose:
# Compare the final selected representation (Layer 18)
# against the layer-0 embedding representation.
#
# Same:
#   - last action token
#   - train/test split
#   - StandardScaler
#   - L2 LogisticRegression
#   - class_weight="balanced"
#   - C = 0.01
#
# Only the representation layer changes:
#   Layer 0
# ============================================================

import os
import json
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("LAYER-0 EMBEDDING BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
BASELINE_LAYER = 0
BASELINE_C = 0.01

print("Representation : last_action_token")
print(f"Layer          : {BASELINE_LAYER}")
print(f"C              : {BASELINE_C}")
print("Task           : binary")
print("Label 0        : non_irreversible")
print("Label 1        : irreversible")
print()

# ------------------------------------------------------------
# Load Layer-0 representation
# ------------------------------------------------------------
X_train_layer0 = X_all[
    train_idx,
    BASELINE_LAYER,
    :
]

X_test_layer0 = X_all[
    test_idx,
    BASELINE_LAYER,
    :
]

y_train_layer0 = y_binary[
    train_idx
]

y_test_layer0 = y_binary[
    test_idx
]

print(
    f"X_train shape: "
    f"{X_train_layer0.shape}"
)

print(
    f"X_test shape : "
    f"{X_test_layer0.shape}"
)

# ------------------------------------------------------------
# Train probe
# ------------------------------------------------------------
print()
print("=" * 70)
print("TRAINING LAYER-0 PROBE")
print("=" * 70)

start_time = time.time()

layer0_probe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logreg",
        LogisticRegression(
            penalty="l2",
            C=BASELINE_C,
            class_weight="balanced",
            solver="saga",
            max_iter=500,
            tol=1e-2,
            random_state=42
        )
    )
])

layer0_probe.fit(
    X_train_layer0,
    y_train_layer0
)

print(
    f"Training complete in "
    f"{time.time() - start_time:.1f}s"
)

# ------------------------------------------------------------
# Test evaluation
# ------------------------------------------------------------
print()
print("=" * 70)
print("LAYER-0 TEST RESULTS")
print("=" * 70)

layer0_pred = layer0_probe.predict(
    X_test_layer0
)

layer0_prob = layer0_probe.predict_proba(
    X_test_layer0
)[:, 1]

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
layer0_accuracy = accuracy_score(
    y_test_layer0,
    layer0_pred
)

layer0_balanced_accuracy = balanced_accuracy_score(
    y_test_layer0,
    layer0_pred
)

layer0_macro_f1 = f1_score(
    y_test_layer0,
    layer0_pred,
    average="macro",
    zero_division=0
)

layer0_precision = precision_score(
    y_test_layer0,
    layer0_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

layer0_recall = recall_score(
    y_test_layer0,
    layer0_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

layer0_f1 = f1_score(
    y_test_layer0,
    layer0_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

layer0_confusion = confusion_matrix(
    y_test_layer0,
    layer0_pred,
    labels=[0, 1]
)

layer0_auroc = roc_auc_score(
    y_test_layer0,
    layer0_prob
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------
print()
print(
    f"Test records      : "
    f"{len(test_idx)}"
)

print(
    f"Accuracy          : "
    f"{layer0_accuracy:.4f}"
)

print(
    f"Balanced Accuracy : "
    f"{layer0_balanced_accuracy:.4f}"
)

print(
    f"Macro-F1          : "
    f"{layer0_macro_f1:.4f}"
)

print(
    f"AUROC             : "
    f"{layer0_auroc:.4f}"
)

print()
print("Per-class metrics:")

print(
    f"Class 0 — non_irreversible | "
    f"Precision={layer0_precision[0]:.4f} | "
    f"Recall={layer0_recall[0]:.4f} | "
    f"F1={layer0_f1[0]:.4f}"
)

print(
    f"Class 1 — irreversible     | "
    f"Precision={layer0_precision[1]:.4f} | "
    f"Recall={layer0_recall[1]:.4f} | "
    f"F1={layer0_f1[1]:.4f}"
)

print()
print("Confusion Matrix:")
print("(rows = true, columns = predicted)")
print(layer0_confusion)

# ------------------------------------------------------------
# Compare with Layer 18
# ------------------------------------------------------------
print()
print("=" * 70)
print("LAYER 18 vs LAYER 0")
print("=" * 70)

print(
    f"Layer 18 BA : "
    f"{balanced_accuracy:.4f}"
)

print(
    f"Layer 0  BA : "
    f"{layer0_balanced_accuracy:.4f}"
)

print(
    f"BA difference: "
    f"{balanced_accuracy - layer0_balanced_accuracy:.4f}"
)

print()

print(
    f"Layer 18 Macro-F1 : "
    f"{macro_f1:.4f}"
)

print(
    f"Layer 0  Macro-F1 : "
    f"{layer0_macro_f1:.4f}"
)

print(
    f"Macro-F1 difference: "
    f"{macro_f1 - layer0_macro_f1:.4f}"
)

print()

print(
    f"Layer 18 AUROC : "
    f"{roc_auc:.4f}"
)

print(
    f"Layer 0  AUROC : "
    f"{layer0_auroc:.4f}"
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
layer0_path = os.path.join(
    WORK_DIR,
    "baselines",
    "qwen_v2_binary_layer0_embedding_baseline.json"
)

layer0_results = {
    "experiment":
        "qwen_v2_binary_layer0_embedding_baseline",

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "representation": {
        "type": "last_action_token",
        "layer": BASELINE_LAYER
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": BASELINE_C,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "train": {
        "n": int(len(train_idx))
    },

    "test": {
        "n": int(len(test_idx))
    },

    "metrics": {
        "accuracy": float(
            layer0_accuracy
        ),

        "balanced_accuracy": float(
            layer0_balanced_accuracy
        ),

        "macro_f1": float(
            layer0_macro_f1
        ),

        "auroc": float(
            layer0_auroc
        ),

        "per_class": {
            "non_irreversible": {
                "precision": float(
                    layer0_precision[0]
                ),
                "recall": float(
                    layer0_recall[0]
                ),
                "f1": float(
                    layer0_f1[0]
                )
            },

            "irreversible": {
                "precision": float(
                    layer0_precision[1]
                ),
                "recall": float(
                    layer0_recall[1]
                ),
                "f1": float(
                    layer0_f1[1]
                )
            }
        },

        "confusion_matrix":
            layer0_confusion.tolist()
    }
}

with open(layer0_path, "w") as f:
    json.dump(
        layer0_results,
        f,
        indent=2
    )

print()
print("Saved:")
print(layer0_path)

print()
print("✓ Layer-0 embedding baseline")
print("✓ Same frozen train/test split")
print("✓ Same probe architecture")
print("✓ Same C = 0.01")
print("✓ Test evaluated only once")

LAYER-0 EMBEDDING BASELINE
Representation : last_action_token
Layer          : 0
C              : 0.01
Task           : binary
Label 0        : non_irreversible
Label 1        : irreversible

X_train shape: (1709, 3584)
X_test shape : (425, 3584)

TRAINING LAYER-0 PROBE
Training complete in 2.5s

LAYER-0 TEST RESULTS

Test records      : 425
Accuracy          : 0.3624
Balanced Accuracy : 0.5018
Macro-F1          : 0.2688
AUROC             : 0.5018

Per-class metrics:
Class 0 — non_irreversible | Precision=1.0000 | Recall=0.0037 | F1=0.0073
Class 1 — irreversible     | Precision=0.3608 | Recall=1.0000 | F1=0.5303

Confusion Matrix:
(rows = true, columns = predicted)
[[  1 271]
 [  0 153]]

LAYER 18 vs LAYER 0
Layer 18 BA : 0.9118
Layer 0  BA : 0.5018
BA difference: 0.4099

Layer 18 Macro-F1 : 0.8838
Layer 0  Macro-F1 : 0.2688
Macro-F1 difference: 0.6150



NameError: name 'roc_auc' is not defined

In [54]:
# ============================================================
# FIX / SAVE LAYER-0 BASELINE RESULTS
# No retraining
# ============================================================

import os
import json

layer0_path = os.path.join(
    WORK_DIR,
    "baselines",
    "qwen_v2_binary_layer0_embedding_baseline.json"
)

layer0_results = {
    "experiment":
        "qwen_v2_binary_layer0_embedding_baseline",

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "representation": {
        "type": "last_action_token",
        "layer": 0
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": 0.01,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "train": {
        "n": int(len(train_idx))
    },

    "test": {
        "n": int(len(test_idx))
    },

    "metrics": {
        "accuracy": float(layer0_accuracy),
        "balanced_accuracy": float(layer0_balanced_accuracy),
        "macro_f1": float(layer0_macro_f1),
        "auroc": float(layer0_auroc),

        "per_class": {
            "non_irreversible": {
                "precision": float(layer0_precision[0]),
                "recall": float(layer0_recall[0]),
                "f1": float(layer0_f1[0])
            },

            "irreversible": {
                "precision": float(layer0_precision[1]),
                "recall": float(layer0_recall[1]),
                "f1": float(layer0_f1[1])
            }
        },

        "confusion_matrix":
            layer0_confusion.tolist()
    }
}

with open(layer0_path, "w") as f:
    json.dump(
        layer0_results,
        f,
        indent=2
    )

print("=" * 70)
print("LAYER-0 BASELINE SAVED")
print("=" * 70)

print(f"Balanced Accuracy : {layer0_balanced_accuracy:.4f}")
print(f"Macro-F1          : {layer0_macro_f1:.4f}")
print(f"AUROC             : {layer0_auroc:.4f}")

print()
print(f"Saved:")
print(layer0_path)

print()
print("✓ No retraining")
print("✓ Existing evaluation reused")
print("✓ Layer-0 AUROC correctly saved")

LAYER-0 BASELINE SAVED
Balanced Accuracy : 0.5018
Macro-F1          : 0.2688
AUROC             : 0.5018

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/baselines/qwen_v2_binary_layer0_embedding_baseline.json

✓ No retraining
✓ Existing evaluation reused
✓ Layer-0 AUROC correctly saved


In [55]:
# ============================================================
# Qwen v2.0 — MEAN ACTION-TOKEN ABLATION
# Binary task
#
# Compare:
#   Last action token  vs  Mean of all action tokens
#
# Same:
#   - frozen train/test split
#   - Layer 18
#   - StandardScaler
#   - L2 LogisticRegression
#   - class_weight="balanced"
#   - C = 0.01
# ============================================================

import os
import json
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("MEAN ACTION-TOKEN REPRESENTATION ABLATION")
print("=" * 70)

ABLATION_LAYER = 18
ABLATION_C = 0.01

print("Representation : mean_action_tokens")
print(f"Layer          : {ABLATION_LAYER}")
print(f"C              : {ABLATION_C}")
print("Task           : binary")
print("Label 0        : non_irreversible")
print("Label 1        : irreversible")

# ------------------------------------------------------------
# Load mean-action representation
# ------------------------------------------------------------

X_mean_train = X_all[
    train_idx,
    ABLATION_LAYER,
    :
]

# IMPORTANT:
# X_all currently contains the last-action-token representation.
# Load the separately saved mean-action representation.
mean_path = os.path.join(
    WORK_DIR,
    "hidden_states",
    "mean_action_tokens.npy"
)

X_mean_all = np.load(
    mean_path,
    mmap_mode="r"
)

print()
print("Mean representation shape:")
print(X_mean_all.shape)

X_mean_train = X_mean_all[
    train_idx,
    ABLATION_LAYER,
    :
]

X_mean_test = X_mean_all[
    test_idx,
    ABLATION_LAYER,
    :
]

y_mean_train = y_binary[train_idx]
y_mean_test = y_binary[test_idx]

print()
print(f"X_train shape: {X_mean_train.shape}")
print(f"X_test shape : {X_mean_test.shape}")

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRAINING MEAN-ACTION PROBE")
print("=" * 70)

start_time = time.time()

mean_probe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logreg",
        LogisticRegression(
            penalty="l2",
            C=ABLATION_C,
            class_weight="balanced",
            solver="saga",
            max_iter=500,
            tol=1e-2,
            random_state=42
        )
    )
])

mean_probe.fit(
    X_mean_train,
    y_mean_train
)

print(
    f"Training complete in "
    f"{time.time() - start_time:.1f}s"
)

# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

print()
print("=" * 70)
print("MEAN-ACTION TEST RESULTS")
print("=" * 70)

mean_pred = mean_probe.predict(
    X_mean_test
)

mean_prob = mean_probe.predict_proba(
    X_mean_test
)[:, 1]

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

mean_accuracy = accuracy_score(
    y_mean_test,
    mean_pred
)

mean_balanced_accuracy = balanced_accuracy_score(
    y_mean_test,
    mean_pred
)

mean_macro_f1 = f1_score(
    y_mean_test,
    mean_pred,
    average="macro",
    zero_division=0
)

mean_precision = precision_score(
    y_mean_test,
    mean_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

mean_recall = recall_score(
    y_mean_test,
    mean_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

mean_f1 = f1_score(
    y_mean_test,
    mean_pred,
    average=None,
    labels=[0, 1],
    zero_division=0
)

mean_confusion = confusion_matrix(
    y_mean_test,
    mean_pred,
    labels=[0, 1]
)

mean_auroc = roc_auc_score(
    y_mean_test,
    mean_prob
)

# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print()
print(f"Test records      : {len(test_idx)}")
print(f"Accuracy          : {mean_accuracy:.4f}")
print(f"Balanced Accuracy : {mean_balanced_accuracy:.4f}")
print(f"Macro-F1          : {mean_macro_f1:.4f}")
print(f"AUROC             : {mean_auroc:.4f}")

print()
print("Per-class metrics:")

print(
    f"Class 0 — non_irreversible | "
    f"Precision={mean_precision[0]:.4f} | "
    f"Recall={mean_recall[0]:.4f} | "
    f"F1={mean_f1[0]:.4f}"
)

print(
    f"Class 1 — irreversible     | "
    f"Precision={mean_precision[1]:.4f} | "
    f"Recall={mean_recall[1]:.4f} | "
    f"F1={mean_f1[1]:.4f}"
)

print()
print("Confusion Matrix:")
print("(rows = true, columns = predicted)")
print(mean_confusion)

# ------------------------------------------------------------
# Compare with last-action representation
# ------------------------------------------------------------

print()
print("=" * 70)
print("LAST ACTION TOKEN vs MEAN ACTION TOKENS")
print("=" * 70)

print(
    f"Last-action BA : {balanced_accuracy:.4f}"
)

print(
    f"Mean-action BA : {mean_balanced_accuracy:.4f}"
)

print(
    f"BA difference  : "
    f"{balanced_accuracy - mean_balanced_accuracy:.4f}"
)

print()

print(
    f"Last-action Macro-F1 : {macro_f1:.4f}"
)

print(
    f"Mean-action Macro-F1 : {mean_macro_f1:.4f}"
)

print(
    f"Macro-F1 difference  : "
    f"{macro_f1 - mean_macro_f1:.4f}"
)

print()

print(
    f"Last-action AUROC : {roc_auc:.4f}"
)

print(
    f"Mean-action AUROC : {mean_auroc:.4f}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

mean_path_out = os.path.join(
    WORK_DIR,
    "ablations",
    "qwen_v2_binary_mean_action_tokens_layer18.json"
)

mean_results = {
    "experiment":
        "qwen_v2_binary_mean_action_tokens_layer18",

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "representation": {
        "type": "mean_action_tokens",
        "layer": ABLATION_LAYER
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": ABLATION_C,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "train": {
        "n": int(len(train_idx))
    },

    "test": {
        "n": int(len(test_idx))
    },

    "metrics": {
        "accuracy": float(mean_accuracy),
        "balanced_accuracy": float(mean_balanced_accuracy),
        "macro_f1": float(mean_macro_f1),
        "auroc": float(mean_auroc),

        "per_class": {
            "non_irreversible": {
                "precision": float(mean_precision[0]),
                "recall": float(mean_recall[0]),
                "f1": float(mean_f1[0])
            },
            "irreversible": {
                "precision": float(mean_precision[1]),
                "recall": float(mean_recall[1]),
                "f1": float(mean_f1[1])
            }
        },

        "confusion_matrix":
            mean_confusion.tolist()
    }
}

with open(mean_path_out, "w") as f:
    json.dump(
        mean_results,
        f,
        indent=2
    )

print()
print("Saved:")
print(mean_path_out)

print()
print("✓ Mean action-token ablation completed")
print("✓ Same Layer 18")
print("✓ Same C = 0.01")
print("✓ Same train/test split")
print("✓ Same classifier")

MEAN ACTION-TOKEN REPRESENTATION ABLATION
Representation : mean_action_tokens
Layer          : 18
C              : 0.01
Task           : binary
Label 0        : non_irreversible
Label 1        : irreversible

Mean representation shape:
(2134, 29, 3584)

X_train shape: (1709, 3584)
X_test shape : (425, 3584)

TRAINING MEAN-ACTION PROBE
Training complete in 4.9s

MEAN-ACTION TEST RESULTS

Test records      : 425
Accuracy          : 0.8729
Balanced Accuracy : 0.9007
Macro-F1          : 0.8699
AUROC             : 0.9883

Per-class metrics:
Class 0 — non_irreversible | Precision=1.0000 | Recall=0.8015 | F1=0.8898
Class 1 — irreversible     | Precision=0.7391 | Recall=1.0000 | F1=0.8500

Confusion Matrix:
(rows = true, columns = predicted)
[[218  54]
 [  0 153]]

LAST ACTION TOKEN vs MEAN ACTION TOKENS
Last-action BA : 0.9118
Mean-action BA : 0.9007
BA difference  : 0.0110

Last-action Macro-F1 : 0.8838
Mean-action Macro-F1 : 0.8699
Macro-F1 difference  : 0.0139



NameError: name 'roc_auc' is not defined

In [56]:
# ============================================================
# SAVE MEAN ACTION-TOKEN ABLATION
# No retraining
# ============================================================

import os
import json

mean_path_out = os.path.join(
    WORK_DIR,
    "ablations",
    "qwen_v2_binary_mean_action_tokens_layer18.json"
)

mean_results = {
    "experiment":
        "qwen_v2_binary_mean_action_tokens_layer18",

    "task": {
        "type": "binary",
        "label_mapping": {
            "0": "non_irreversible",
            "1": "irreversible"
        }
    },

    "representation": {
        "type": "mean_action_tokens",
        "layer": 18
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": 0.01,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "train": {
        "n": int(len(train_idx))
    },

    "test": {
        "n": int(len(test_idx))
    },

    "metrics": {
        "accuracy": float(mean_accuracy),
        "balanced_accuracy": float(mean_balanced_accuracy),
        "macro_f1": float(mean_macro_f1),
        "auroc": float(mean_auroc),

        "per_class": {
            "non_irreversible": {
                "precision": float(mean_precision[0]),
                "recall": float(mean_recall[0]),
                "f1": float(mean_f1[0])
            },

            "irreversible": {
                "precision": float(mean_precision[1]),
                "recall": float(mean_recall[1]),
                "f1": float(mean_f1[1])
            }
        },

        "confusion_matrix":
            mean_confusion.tolist()
    }
}

with open(mean_path_out, "w") as f:
    json.dump(
        mean_results,
        f,
        indent=2
    )

print("=" * 70)
print("MEAN ACTION-TOKEN ABLATION SAVED")
print("=" * 70)

print(f"Balanced Accuracy : {mean_balanced_accuracy:.4f}")
print(f"Macro-F1          : {mean_macro_f1:.4f}")
print(f"AUROC             : {mean_auroc:.4f}")

print()
print("Saved:")
print(mean_path_out)

print()
print("✓ No retraining")
print("✓ Mean-action results preserved")

MEAN ACTION-TOKEN ABLATION SAVED
Balanced Accuracy : 0.9007
Macro-F1          : 0.8699
AUROC             : 0.9883

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/ablations/qwen_v2_binary_mean_action_tokens_layer18.json

✓ No retraining
✓ Mean-action results preserved


In [57]:
# ============================================================
# HORIZON DISTRIBUTION CHECK
# ============================================================

from collections import Counter

print("=" * 70)
print("HORIZON DISTRIBUTION")
print("=" * 70)

all_horizons = [
    record.get("horizon")
    for record in records
]

train_horizons = [
    records[i].get("horizon")
    for i in train_idx
]

test_horizons = [
    records[i].get("horizon")
    for i in test_idx
]

print("\nAll records:")
for h, n in sorted(
    Counter(all_horizons).items(),
    key=lambda x: str(x[0])
):
    print(f"  horizon={h}: {n}")

print("\nTraining records:")
for h, n in sorted(
    Counter(train_horizons).items(),
    key=lambda x: str(x[0])
):
    print(f"  horizon={h}: {n}")

print("\nTest records:")
for h, n in sorted(
    Counter(test_horizons).items(),
    key=lambda x: str(x[0])
):
    print(f"  horizon={h}: {n}")

print()
print("Unique horizon values:")
print(sorted(set(all_horizons), key=str))

HORIZON DISTRIBUTION

All records:
  horizon=3: 2134

Training records:
  horizon=3: 1709

Test records:
  horizon=3: 425

Unique horizon values:
[3]


In [58]:
# ============================================================
# CHECK labels_by_horizon FOR H=1 / H=2
# ============================================================

from collections import Counter

print("=" * 70)
print("CHECKING labels_by_horizon")
print("=" * 70)

horizon_keys = Counter()
records_with_h1 = 0
records_with_h2 = 0
records_with_h3 = 0

example = None

for record in records:

    lbh = record.get("labels_by_horizon")

    if isinstance(lbh, dict):

        for key in lbh.keys():
            horizon_keys[str(key)] += 1

        if "1" in lbh or 1 in lbh:
            records_with_h1 += 1

        if "2" in lbh or 2 in lbh:
            records_with_h2 += 1

        if "3" in lbh or 3 in lbh:
            records_with_h3 += 1

        if example is None:
            example = lbh

print("\nHorizon keys found:")
for key, count in sorted(
    horizon_keys.items(),
    key=lambda x: str(x[0])
):
    print(f"  {key}: {count}")

print()
print(f"Records with h=1 labels: {records_with_h1}")
print(f"Records with h=2 labels: {records_with_h2}")
print(f"Records with h=3 labels: {records_with_h3}")

print()
print("Example labels_by_horizon:")
print(example)

CHECKING labels_by_horizon

Horizon keys found:
  1: 2134
  2: 2134
  3: 2134

Records with h=1 labels: 2134
Records with h=2 labels: 2134
Records with h=3 labels: 2134

Example labels_by_horizon:
{'1': 'partially_reversible', '2': 'partially_reversible', '3': 'partially_reversible'}


In [59]:
# ============================================================
# HORIZON 1 / HORIZON 2 LABEL DISTRIBUTION
# ============================================================

from collections import Counter

print("=" * 70)
print("HORIZON LABEL DISTRIBUTIONS")
print("=" * 70)

for horizon in ["1", "2", "3"]:

    labels = [
        record["labels_by_horizon"][horizon]
        for record in records
    ]

    counts = Counter(labels)

    print()
    print(f"HORIZON = {horizon}")
    print("-" * 40)

    for label in [
        "reversible",
        "partially_reversible",
        "irreversible"
    ]:
        print(
            f"{label:22s}: "
            f"{counts.get(label, 0)}"
        )

    print(
        f"Total                  : "
        f"{len(labels)}"
    )

print()
print("=" * 70)
print("BINARY DISTRIBUTIONS")
print("=" * 70)

for horizon in ["1", "2", "3"]:

    labels = [
        record["labels_by_horizon"][horizon]
        for record in records
    ]

    binary_labels = np.array([
        1 if label == "irreversible" else 0
        for label in labels
    ])

    counts = Counter(binary_labels)

    print()
    print(
        f"h={horizon}: "
        f"class 0={counts.get(0, 0)}, "
        f"class 1={counts.get(1, 0)}"
    )

HORIZON LABEL DISTRIBUTIONS

HORIZON = 1
----------------------------------------
reversible            : 753
partially_reversible  : 786
irreversible          : 595
Total                  : 2134

HORIZON = 2
----------------------------------------
reversible            : 799
partially_reversible  : 758
irreversible          : 577
Total                  : 2134

HORIZON = 3
----------------------------------------
reversible            : 799
partially_reversible  : 758
irreversible          : 577
Total                  : 2134

BINARY DISTRIBUTIONS

h=1: class 0=1539, class 1=595

h=2: class 0=1557, class 1=577

h=3: class 0=1557, class 1=577


In [60]:
# ============================================================
# HORIZON ABLATIONS — h=1 and h=2
# Qwen v2.0 Binary Probe
#
# Same representation and classifier as primary binary probe:
#   Representation : last action token
#   Layer          : 18
#   C              : 0.01
#   StandardScaler
#   L2 LogisticRegression
#   class_weight="balanced"
#
# Only the target labels change:
#   h=1 labels_by_horizon["1"]
#   h=2 labels_by_horizon["2"]
#
# Frozen scenario-based train/test split is unchanged.
# ============================================================

import os
import json
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("HORIZON ABLATIONS — h=1 and h=2")
print("=" * 70)

HORIZON_LAYER = 18
HORIZON_C = 0.01

# ------------------------------------------------------------
# Representation
# ------------------------------------------------------------

X_h_train = X_all[
    train_idx,
    HORIZON_LAYER,
    :
]

X_h_test = X_all[
    test_idx,
    HORIZON_LAYER,
    :
]

print(
    f"Representation : last_action_token"
)

print(
    f"Layer          : {HORIZON_LAYER}"
)

print(
    f"C              : {HORIZON_C}"
)

print(
    f"X_train shape  : {X_h_train.shape}"
)

print(
    f"X_test shape   : {X_h_test.shape}"
)

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def run_horizon_probe(
    horizon,
    X_train,
    X_test
):

    print()
    print("=" * 70)
    print(f"HORIZON h={horizon}")
    print("=" * 70)

    # --------------------------------------------------------
    # Construct binary labels
    #
    # 0 = reversible + partially_reversible
    # 1 = irreversible
    # --------------------------------------------------------

    labels_all = [
        record["labels_by_horizon"][str(horizon)]
        for record in records
    ]

    y_h = np.array([
        1 if label == "irreversible" else 0
        for label in labels_all
    ], dtype=np.int64)

    y_train = y_h[train_idx]
    y_test = y_h[test_idx]

    print()
    print("Training distribution:")
    print(
        f"  class 0: {np.sum(y_train == 0)}"
    )
    print(
        f"  class 1: {np.sum(y_train == 1)}"
    )

    print()
    print("Test distribution:")
    print(
        f"  class 0: {np.sum(y_test == 0)}"
    )
    print(
        f"  class 1: {np.sum(y_test == 1)}"
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    print()
    print("Training probe...")

    start_time = time.time()

    probe = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "logreg",
            LogisticRegression(
                penalty="l2",
                C=HORIZON_C,
                class_weight="balanced",
                solver="saga",
                max_iter=500,
                tol=1e-2,
                random_state=42
            )
        )
    ])

    probe.fit(
        X_train,
        y_train
    )

    print(
        f"Training complete in "
        f"{time.time() - start_time:.1f}s"
    )

    # --------------------------------------------------------
    # Test
    # --------------------------------------------------------

    pred = probe.predict(X_test)

    prob = probe.predict_proba(
        X_test
    )[:, 1]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        pred
    )

    macro_f1 = f1_score(
        y_test,
        pred,
        average="macro",
        zero_division=0
    )

    precision = precision_score(
        y_test,
        pred,
        average=None,
        labels=[0, 1],
        zero_division=0
    )

    recall = recall_score(
        y_test,
        pred,
        average=None,
        labels=[0, 1],
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        pred,
        average=None,
        labels=[0, 1],
        zero_division=0
    )

    confusion = confusion_matrix(
        y_test,
        pred,
        labels=[0, 1]
    )

    auroc = roc_auc_score(
        y_test,
        prob
    )

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print()
    print(
        f"Test records      : {len(y_test)}"
    )

    print(
        f"Accuracy          : {accuracy:.4f}"
    )

    print(
        f"Balanced Accuracy : {balanced_accuracy:.4f}"
    )

    print(
        f"Macro-F1          : {macro_f1:.4f}"
    )

    print(
        f"AUROC             : {auroc:.4f}"
    )

    print()
    print("Per-class metrics:")

    print(
        f"Class 0 — non_irreversible | "
        f"Precision={precision[0]:.4f} | "
        f"Recall={recall[0]:.4f} | "
        f"F1={f1[0]:.4f}"
    )

    print(
        f"Class 1 — irreversible     | "
        f"Precision={precision[1]:.4f} | "
        f"Recall={recall[1]:.4f} | "
        f"F1={f1[1]:.4f}"
    )

    print()
    print("Confusion Matrix:")
    print("(rows = true, columns = predicted)")
    print(confusion)

    return {
        "horizon": int(horizon),

        "representation": {
            "type": "last_action_token",
            "layer": HORIZON_LAYER
        },

        "classifier": {
            "scaler": "StandardScaler",
            "penalty": "l2",
            "C": HORIZON_C,
            "class_weight": "balanced",
            "solver": "saga",
            "max_iter": 500,
            "tol": 1e-2,
            "random_state": 42
        },

        "train_n": int(len(y_train)),
        "test_n": int(len(y_test)),

        "train_class_counts": {
            "0": int(np.sum(y_train == 0)),
            "1": int(np.sum(y_train == 1))
        },

        "test_class_counts": {
            "0": int(np.sum(y_test == 0)),
            "1": int(np.sum(y_test == 1))
        },

        "metrics": {
            "accuracy": float(accuracy),
            "balanced_accuracy": float(
                balanced_accuracy
            ),
            "macro_f1": float(macro_f1),
            "auroc": float(auroc),

            "per_class": {
                "non_irreversible": {
                    "precision": float(
                        precision[0]
                    ),
                    "recall": float(
                        recall[0]
                    ),
                    "f1": float(
                        f1[0]
                    )
                },

                "irreversible": {
                    "precision": float(
                        precision[1]
                    ),
                    "recall": float(
                        recall[1]
                    ),
                    "f1": float(
                        f1[1]
                    )
                }
            },

            "confusion_matrix":
                confusion.tolist()
        }
    }


# ------------------------------------------------------------
# Run h=1
# ------------------------------------------------------------

result_h1 = run_horizon_probe(
    horizon=1,
    X_train=X_h_train,
    X_test=X_h_test
)

# ------------------------------------------------------------
# Run h=2
# ------------------------------------------------------------

result_h2 = run_horizon_probe(
    horizon=2,
    X_train=X_h_train,
    X_test=X_h_test
)

# ------------------------------------------------------------
# Comparison
# ------------------------------------------------------------

print()
print("=" * 70)
print("HORIZON COMPARISON")
print("=" * 70)

print(
    f"h=1  BA={result_h1['metrics']['balanced_accuracy']:.4f} | "
    f"Macro-F1={result_h1['metrics']['macro_f1']:.4f} | "
    f"AUROC={result_h1['metrics']['auroc']:.4f}"
)

print(
    f"h=2  BA={result_h2['metrics']['balanced_accuracy']:.4f} | "
    f"Macro-F1={result_h2['metrics']['macro_f1']:.4f} | "
    f"AUROC={result_h2['metrics']['auroc']:.4f}"
)

print(
    f"h=3  BA={balanced_accuracy:.4f} | "
    f"Macro-F1={macro_f1:.4f} | "
    f"AUROC={roc_auc:.4f}"
)

print()
print("BA differences relative to h=3:")

print(
    f"h=1 − h=3: "
    f"{result_h1['metrics']['balanced_accuracy'] - balanced_accuracy:+.4f}"
)

print(
    f"h=2 − h=3: "
    f"{result_h2['metrics']['balanced_accuracy'] - balanced_accuracy:+.4f}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

horizon_output = {
    "experiment":
        "qwen_v2_binary_horizon_ablations",

    "representation": {
        "type": "last_action_token",
        "layer": HORIZON_LAYER
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": HORIZON_C,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "label_mapping": {
        "0": "non_irreversible",
        "1": "irreversible"
    },

    "h1": result_h1,
    "h2": result_h2
}

horizon_output_path = os.path.join(
    WORK_DIR,
    "ablations",
    "qwen_v2_binary_horizon_ablations.json"
)

with open(horizon_output_path, "w") as f:
    json.dump(
        horizon_output,
        f,
        indent=2
    )

print()
print("Saved:")
print(horizon_output_path)

print()
print("✓ h=1 evaluated")
print("✓ h=2 evaluated")
print("✓ Same frozen scenario split")
print("✓ Same Layer-18 representation")
print("✓ Same classifier")

HORIZON ABLATIONS — h=1 and h=2
Representation : last_action_token
Layer          : 18
C              : 0.01
X_train shape  : (1709, 3584)
X_test shape   : (425, 3584)

HORIZON h=1

Training distribution:
  class 0: 1267
  class 1: 442

Test distribution:
  class 0: 272
  class 1: 153

Training probe...
Training complete in 3.8s

Test records      : 425
Accuracy          : 0.8941
Balanced Accuracy : 0.9173
Macro-F1          : 0.8908
AUROC             : 0.9884

Per-class metrics:
Class 0 — non_irreversible | Precision=1.0000 | Recall=0.8346 | F1=0.9098
Class 1 — irreversible     | Precision=0.7727 | Recall=1.0000 | F1=0.8718

Confusion Matrix:
(rows = true, columns = predicted)
[[227  45]
 [  0 153]]

HORIZON h=2

Training distribution:
  class 0: 1285
  class 1: 424

Test distribution:
  class 0: 272
  class 1: 153

Training probe...
Training complete in 3.8s

Test records      : 425
Accuracy          : 0.8871
Balanced Accuracy : 0.9118
Macro-F1          : 0.8838
AUROC             : 0.

NameError: name 'roc_auc' is not defined

In [61]:
# ============================================================
# SAVE HORIZON ABLATION RESULTS
# No retraining
# ============================================================

import os
import json

# h=3 is the already completed primary binary probe
h3_result = {
    "horizon": 3,

    "representation": {
        "type": "last_action_token",
        "layer": 18
    },

    "metrics": {
        "accuracy": 0.8871,
        "balanced_accuracy": 0.9118,
        "macro_f1": 0.8838,
        "auroc": 0.9864,

        "per_class": {
            "non_irreversible": {
                "precision": 1.0000,
                "recall": 0.8235,
                "f1": 0.9032
            },
            "irreversible": {
                "precision": 0.7612,
                "recall": 1.0000,
                "f1": 0.8644
            }
        },

        "confusion_matrix": [
            [224, 48],
            [0, 153]
        ]
    }
}

horizon_output = {
    "experiment":
        "qwen_v2_binary_horizon_ablations",

    "representation": {
        "type": "last_action_token",
        "layer": 18
    },

    "classifier": {
        "scaler": "StandardScaler",
        "penalty": "l2",
        "C": 0.01,
        "class_weight": "balanced",
        "solver": "saga",
        "max_iter": 500,
        "tol": 1e-2,
        "random_state": 42
    },

    "label_mapping": {
        "0": "non_irreversible",
        "1": "irreversible"
    },

    "h1": result_h1,
    "h2": result_h2,
    "h3": h3_result,

    "comparison": {
        "h1_minus_h3_balanced_accuracy":
            0.9173 - 0.9118,

        "h2_minus_h3_balanced_accuracy":
            0.9118 - 0.9118
    }
}

horizon_output_path = os.path.join(
    WORK_DIR,
    "ablations",
    "qwen_v2_binary_horizon_ablations.json"
)

with open(horizon_output_path, "w") as f:
    json.dump(
        horizon_output,
        f,
        indent=2
    )

print("=" * 70)
print("HORIZON ABLATIONS SAVED")
print("=" * 70)

print()
print("h=1")
print("  BA     :", "0.9173")
print("  Macro-F1:", "0.8908")
print("  AUROC  :", "0.9884")

print()
print("h=2")
print("  BA     :", "0.9118")
print("  Macro-F1:", "0.8838")
print("  AUROC  :", "0.9864")

print()
print("h=3")
print("  BA     :", "0.9118")
print("  Macro-F1:", "0.8838")
print("  AUROC  :", "0.9864")

print()
print("Saved:")
print(horizon_output_path)

print()
print("✓ h=1 saved")
print("✓ h=2 saved")
print("✓ h=3 comparison recorded")
print("✓ No retraining")

HORIZON ABLATIONS SAVED

h=1
  BA     : 0.9173
  Macro-F1: 0.8908
  AUROC  : 0.9884

h=2
  BA     : 0.9118
  Macro-F1: 0.8838
  AUROC  : 0.9864

h=3
  BA     : 0.9118
  Macro-F1: 0.8838
  AUROC  : 0.9864

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/ablations/qwen_v2_binary_horizon_ablations.json

✓ h=1 saved
✓ h=2 saved
✓ h=3 comparison recorded
✓ No retraining


In [62]:
# ============================================================
# MINIMAL-PAIR DIAGNOSTIC
# ============================================================

from collections import Counter, defaultdict

print("=" * 70)
print("MINIMAL-PAIR DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# Inspect pair_id distribution
# ------------------------------------------------------------

all_pair_ids = [
    record.get("pair_id")
    for record in records
]

test_pair_ids = [
    records[i].get("pair_id")
    for i in test_idx
]

print()
print("Total records with pair_id:")
print(
    sum(
        pid is not None
        for pid in all_pair_ids
    )
)

print()
print("Test records with pair_id:")
print(
    sum(
        pid is not None
        for pid in test_pair_ids
    )
)

# ------------------------------------------------------------
# Unique pairs
# ------------------------------------------------------------

all_pairs = sorted(
    set(
        pid
        for pid in all_pair_ids
        if pid is not None
    ),
    key=str
)

test_pairs = sorted(
    set(
        pid
        for pid in test_pair_ids
        if pid is not None
    ),
    key=str
)

print()
print("Unique pair IDs:")
print(
    len(all_pairs)
)

print()
print("Unique pair IDs in test:")
print(
    len(test_pairs)
)

print()
print("First pair IDs:")
for pid in test_pairs[:30]:
    print(
        f"  {pid}"
    )

# ------------------------------------------------------------
# Pair sizes in test
# ------------------------------------------------------------

pair_to_test_indices = defaultdict(list)

for idx in test_idx:

    pid = records[idx].get("pair_id")

    if pid is not None:
        pair_to_test_indices[pid].append(idx)

print()
print("=" * 70)
print("TEST PAIR SIZES")
print("=" * 70)

pair_sizes = Counter(
    {
        pid: len(indices)
        for pid, indices
        in pair_to_test_indices.items()
    }
)

for size, count in sorted(
    pair_sizes.items()
):
    print(
        f"Pairs with {size} test records: "
        f"{count}"
    )

# ------------------------------------------------------------
# Inspect labels inside each test pair
# ------------------------------------------------------------

print()
print("=" * 70)
print("TEST PAIR LABEL CHECK")
print("=" * 70)

for pid in test_pairs[:30]:

    indices = pair_to_test_indices[pid]

    print()
    print(
        f"Pair: {pid}"
    )

    for idx in indices:

        r = records[idx]

        print(
            f"  id={r['id']}"
        )

        print(
            f"    label={r['label']}"
        )

        print(
            f"    action_verb={r['action_verb']}"
        )

        print(
            f"    scenario_id={r['scenario_id']}"
        )

MINIMAL-PAIR DIAGNOSTIC

Total records with pair_id:
878

Test records with pair_id:
162

Unique pair IDs:
439

Unique pair IDs in test:
81

First pair IDs:
  s01_link_revoke_address_kind-0000-link-compact
  s01_link_revoke_address_kind-0000-link-kv
  s01_link_revoke_address_kind-0000-link-prose
  s01_link_revoke_address_kind-0001-link-compact
  s01_link_revoke_address_kind-0001-link-kv
  s01_link_revoke_address_kind-0001-link-prose
  s01_link_revoke_address_kind-0002-link-compact
  s01_link_revoke_address_kind-0002-link-kv
  s01_link_revoke_address_kind-0002-link-prose
  v2a1_restore_backup_fresh-0001-primary-compact
  v2a1_restore_backup_fresh-0001-primary-kv
  v2a1_restore_backup_fresh-0001-primary-prose
  v2a1_restore_backup_fresh-0005-primary-compact
  v2a1_restore_backup_fresh-0005-primary-kv
  v2a1_restore_backup_fresh-0005-primary-prose
  v2a1_restore_backup_fresh-0006-primary-compact
  v2a1_restore_backup_fresh-0006-primary-kv
  v2a1_restore_backup_fresh-0006-primary-prose
  v

In [63]:
# ============================================================
# IDENTIFY UNDERLYING MINIMAL-PAIR FAMILIES
# ============================================================

from collections import defaultdict, Counter
import re

print("=" * 70)
print("UNDERLYING MINIMAL-PAIR FAMILIES")
print("=" * 70)

# ------------------------------------------------------------
# Get all test pair IDs
# ------------------------------------------------------------

test_pair_ids = sorted(
    set(
        records[i].get("pair_id")
        for i in test_idx
        if records[i].get("pair_id") is not None
    )
)

# ------------------------------------------------------------
# Remove render-style suffix:
#   -compact
#   -kv
#   -prose
#
# Example:
#   foo-compact -> foo
#   foo-kv      -> foo
#   foo-prose   -> foo
# ------------------------------------------------------------

def normalize_pair_id(pair_id):

    for suffix in [
        "-compact",
        "-kv",
        "-prose"
    ]:
        if pair_id.endswith(suffix):
            return pair_id[:-len(suffix)]

    return pair_id


family_to_pair_ids = defaultdict(list)

for pair_id in test_pair_ids:

    family = normalize_pair_id(pair_id)

    family_to_pair_ids[family].append(
        pair_id
    )

# ------------------------------------------------------------
# Print families
# ------------------------------------------------------------

print()
print(
    f"Unique test pair IDs : "
    f"{len(test_pair_ids)}"
)

print(
    f"Underlying families  : "
    f"{len(family_to_pair_ids)}"
)

print()
print("=" * 70)
print("FAMILY COUNTS")
print("=" * 70)

family_sizes = Counter(
    len(pair_ids)
    for pair_ids in family_to_pair_ids.values()
)

for size, count in sorted(
    family_sizes.items()
):
    print(
        f"Families with {size} render variants: "
        f"{count}"
    )

print()
print("=" * 70)
print("ALL UNDERLYING FAMILIES")
print("=" * 70)

for family in sorted(
    family_to_pair_ids.keys()
):

    print()
    print(
        f"FAMILY: {family}"
    )

    print(
        f"  Render variants: "
        f"{len(family_to_pair_ids[family])}"
    )

    for pair_id in sorted(
        family_to_pair_ids[family]
    ):
        print(
            f"    {pair_id}"
        )
        

UNDERLYING MINIMAL-PAIR FAMILIES

Unique test pair IDs : 81
Underlying families  : 27

FAMILY COUNTS
Families with 3 render variants: 27

ALL UNDERLYING FAMILIES

FAMILY: s01_link_revoke_address_kind-0000-link
  Render variants: 3
    s01_link_revoke_address_kind-0000-link-compact
    s01_link_revoke_address_kind-0000-link-kv
    s01_link_revoke_address_kind-0000-link-prose

FAMILY: s01_link_revoke_address_kind-0001-link
  Render variants: 3
    s01_link_revoke_address_kind-0001-link-compact
    s01_link_revoke_address_kind-0001-link-kv
    s01_link_revoke_address_kind-0001-link-prose

FAMILY: s01_link_revoke_address_kind-0002-link
  Render variants: 3
    s01_link_revoke_address_kind-0002-link-compact
    s01_link_revoke_address_kind-0002-link-kv
    s01_link_revoke_address_kind-0002-link-prose

FAMILY: v2a1_restore_backup_fresh-0001-primary
  Render variants: 3
    v2a1_restore_backup_fresh-0001-primary-compact
    v2a1_restore_backup_fresh-0001-primary-kv
    v2a1_restore_backup_fre

In [64]:
# ============================================================
# FIND THE DESIGNATED 24-PAIR MINIMAL-PAIR SUBSET
# ============================================================

from collections import Counter, defaultdict
import json

print("=" * 70)
print("SEARCHING DATASET METADATA FOR MINIMAL-PAIR MARKERS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Inspect unique sensitivity_group values
# ------------------------------------------------------------

print("\nSENSITIVITY GROUP VALUES")
print("-" * 70)

sensitivity_values = Counter(
    r.get("sensitivity_group")
    for r in records
)

for value, count in sensitivity_values.items():
    print(f"{value!r}: {count}")

# ------------------------------------------------------------
# 2. Inspect template_id values
# ------------------------------------------------------------

print("\nTEMPLATE ID VALUES")
print("-" * 70)

template_values = Counter(
    r.get("template_id")
    for r in records
)

for value, count in template_values.items():
    print(f"{value!r}: {count}")

# ------------------------------------------------------------
# 3. Inspect meta keys
# ------------------------------------------------------------

print("\nMETA KEYS")
print("-" * 70)

meta_keys = Counter()

for r in records:
    meta = r.get("meta")

    if isinstance(meta, dict):
        for k in meta.keys():
            meta_keys[k] += 1

for key, count in meta_keys.items():
    print(f"{key}: {count}")

# ------------------------------------------------------------
# 4. Look for values containing words related to pairs
# ------------------------------------------------------------

print("\nPOSSIBLE MINIMAL-PAIR MARKERS")
print("-" * 70)

keywords = [
    "pair",
    "minimal",
    "sensitivity",
    "contrast",
    "subset",
    "test"
]

hits = []

for r in records:

    meta = r.get("meta")

    if not isinstance(meta, dict):
        continue

    for key, value in meta.items():

        text = str(value).lower()

        if any(keyword in text for keyword in keywords):

            hits.append(
                (
                    r["id"],
                    key,
                    value
                )
            )

# Print unique examples only
seen = set()

for item in hits:

    key = (item[1], str(item[2]))

    if key in seen:
        continue

    seen.add(key)

    print(
        f"key={item[1]!r}, "
        f"value={item[2]!r}"
    )

    if len(seen) >= 100:
        break

print()
print("Total possible metadata hits:", len(hits))

SEARCHING DATASET METADATA FOR MINIMAL-PAIR MARKERS

SENSITIVITY GROUP VALUES
----------------------------------------------------------------------
None: 2134

TEMPLATE ID VALUES
----------------------------------------------------------------------
'f01_delete_bin': 127
'f02_delete_backup_fresh': 148
'f03_delete_backup_present': 22
'f04_edit_versioning': 55
'f05_edit_backup_fresh': 152
'f06_edit_backup_present': 62
'f07_email_undo': 96
'f08_email_undo_short': 48
'f09_pay_cancel_vs_final': 134
'f10_pay_two_step_vs_final': 48
'f11_pay_refund_vs_final': 29
'f12_pay_cancel_vs_refund': 4
'f13_pay_refund_fee': 170
'f14_pay_two_step_vs_refund': 9
'n01_pay_cancel_window': 68
'n02_pay_two_step_window': 56
's01_link_revoke_address_kind': 30
's02_link_create_history': 12
'sw-boil-kitchen': 4
'sw-power-workshop': 1
'v2a1_restore_backup_fresh': 106
'v2a2_restore_deleted_bin': 104
'v2b1_second_edit_backup': 56
'v2b2_delete_edited_bin': 32
'v2b3_delete_second_bin': 144
'v2b4_second_payment_policy':

In [65]:
# ============================================================
# ANALYZE DESIGNATED PAIR MEMBERS
# ============================================================

from collections import Counter, defaultdict

print("=" * 70)
print("DESIGNATED PAIR-MEMBER ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Count example_kind
# ------------------------------------------------------------

kind_counts = Counter(
    r.get("meta", {}).get("example_kind")
    for r in records
)

print("\nALL RECORDS")
for k, v in kind_counts.items():
    print(f"{k!r}: {v}")

# ------------------------------------------------------------
# 2. Count pair members in train/test
# ------------------------------------------------------------

train_pair_members = []
test_pair_members = []

for idx in train_idx:
    r = records[idx]
    if r.get("meta", {}).get("example_kind") == "pair_member":
        train_pair_members.append(r)

for idx in test_idx:
    r = records[idx]
    if r.get("meta", {}).get("example_kind") == "pair_member":
        test_pair_members.append(r)

print("\nTRAIN")
print("Pair-member records:", len(train_pair_members))
print(
    "Unique pair IDs:",
    len(set(r.get("pair_id") for r in train_pair_members))
)

print("\nTEST")
print("Pair-member records:", len(test_pair_members))
print(
    "Unique pair IDs:",
    len(set(r.get("pair_id") for r in test_pair_members))
)

# ------------------------------------------------------------
# 3. Group TEST pair members by pair_id
# ------------------------------------------------------------

pair_groups = defaultdict(list)

for r in test_pair_members:
    pair_groups[r.get("pair_id")].append(r)

print("\n" + "=" * 70)
print("TEST DESIGNATED PAIRS")
print("=" * 70)

print("Number of test pair groups:", len(pair_groups))

sizes = Counter(
    len(v) for v in pair_groups.values()
)

print("\nPair-size distribution:")
for size, count in sorted(sizes.items()):
    print(
        f"  {size} records per pair: {count} pairs"
    )

# ------------------------------------------------------------
# 4. Print every designated test pair
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PAIR DETAILS")
print("=" * 70)

for pair_id in sorted(pair_groups):

    members = pair_groups[pair_id]

    print(f"\nPAIR: {pair_id}")

    for r in members:
        print(
            f"  id={r['id']}"
        )
        print(
            f"    label={r['label']}"
        )
        print(
            f"    action_verb={r['action_verb']}"
        )
        print(
            f"    scenario_id={r['scenario_id']}"
        )
        print(
            f"    render_style={r.get('render_style')}"
        )
        print(
            f"    example_kind="
            f"{r.get('meta', {}).get('example_kind')}"
        )

DESIGNATED PAIR-MEMBER ANALYSIS

ALL RECORDS
'non_pair': 1256
'pair_member': 878

TRAIN
Pair-member records: 716
Unique pair IDs: 358

TEST
Pair-member records: 162
Unique pair IDs: 81

TEST DESIGNATED PAIRS
Number of test pair groups: 81

Pair-size distribution:
  2 records per pair: 81 pairs

PAIR DETAILS

PAIR: s01_link_revoke_address_kind-0000-link-compact
  id=s01_link_revoke_address_kind-0000-link-custom-compact
    label=reversible
    action_verb=revoke_share_link
    scenario_id=grp-6edb9a1822e5
    render_style=compact
    example_kind=pair_member
  id=s01_link_revoke_address_kind-0000-link-generated-compact
    label=partially_reversible
    action_verb=revoke_share_link
    scenario_id=grp-6edb9a1822e5
    render_style=compact
    example_kind=pair_member

PAIR: s01_link_revoke_address_kind-0000-link-kv
  id=s01_link_revoke_address_kind-0000-link-custom-kv
    label=reversible
    action_verb=revoke_share_link
    scenario_id=grp-6edb9a1822e5
    render_style=kv
    example

In [66]:
# ============================================================
# COMPACT MINIMAL-PAIR SUMMARY
# ============================================================

from collections import Counter, defaultdict

# All designated pair members
pair_members = [
    r for r in records
    if r.get("meta", {}).get("example_kind") == "pair_member"
]

# Train/test designated pair members
train_pm = [
    records[i] for i in train_idx
    if records[i].get("meta", {}).get("example_kind") == "pair_member"
]

test_pm = [
    records[i] for i in test_idx
    if records[i].get("meta", {}).get("example_kind") == "pair_member"
]

# Group by pair_id
test_pairs = defaultdict(list)

for r in test_pm:
    test_pairs[r["pair_id"]].append(r)

print("=" * 60)
print("COMPACT MINIMAL-PAIR SUMMARY")
print("=" * 60)

print("Total pair-member records:", len(pair_members))
print("Train pair-member records:", len(train_pm))
print("Test pair-member records:", len(test_pm))

print("Unique pair IDs overall:",
      len(set(r["pair_id"] for r in pair_members)))

print("Unique pair IDs in train:",
      len(set(r["pair_id"] for r in train_pm)))

print("Unique pair IDs in test:",
      len(test_pairs))

print("\nTest pair-size distribution:")
print(Counter(len(v) for v in test_pairs.values()))

# Label structure
print("\nTest pair label structures:")

label_patterns = Counter()

for pair_id, members in test_pairs.items():
    labels = tuple(sorted(r["label"] for r in members))
    label_patterns[labels] += 1

for pattern, count in label_patterns.items():
    print(f"  {pattern}: {count}")

# Render-style coverage
print("\nTest render-style coverage:")

render_counts = Counter(
    r.get("render_style")
    for r in test_pm
)

for style, count in render_counts.items():
    print(f"  {style}: {count}")

# Print only pair IDs, not all records
print("\nTest pair IDs:")

for pair_id in sorted(test_pairs):
    print(pair_id)

COMPACT MINIMAL-PAIR SUMMARY
Total pair-member records: 878
Train pair-member records: 716
Test pair-member records: 162
Unique pair IDs overall: 439
Unique pair IDs in train: 358
Unique pair IDs in test: 81

Test pair-size distribution:
Counter({2: 81})

Test pair label structures:
  ('partially_reversible', 'reversible'): 9
  ('irreversible', 'reversible'): 21
  ('irreversible', 'partially_reversible'): 51

Test render-style coverage:
  compact: 54
  kv: 54
  prose: 54

Test pair IDs:
s01_link_revoke_address_kind-0000-link-compact
s01_link_revoke_address_kind-0000-link-kv
s01_link_revoke_address_kind-0000-link-prose
s01_link_revoke_address_kind-0001-link-compact
s01_link_revoke_address_kind-0001-link-kv
s01_link_revoke_address_kind-0001-link-prose
s01_link_revoke_address_kind-0002-link-compact
s01_link_revoke_address_kind-0002-link-kv
s01_link_revoke_address_kind-0002-link-prose
v2a1_restore_backup_fresh-0001-primary-compact
v2a1_restore_backup_fresh-0001-primary-kv
v2a1_restore_back

In [68]:
import os

print("Searching for manifest.json...")

matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    if "manifest.json" in files:
        matches.append(os.path.join(root, "manifest.json"))

for path in matches:
    print(path)

print("\nTotal matches:", len(matches))

Searching for manifest.json...
/kaggle/input/datasets/srijapolisetty/dataset-adnn-v2/adnn-dataset-v2.0/data/production/v2.0/manifest.json
/kaggle/input/datasets/srijapolisetty/dataset-adnn/adnn-dataset-v1.0/data/production/v1.0/manifest.json

Total matches: 2


In [69]:
import json
import os

DATA_DIR = "/kaggle/input/datasets/srijapolisetty/dataset-adnn-v2/adnn-dataset-v2.0/data/production/v2.0"

manifest_path = os.path.join(DATA_DIR, "manifest.json")
generation_path = os.path.join(DATA_DIR, "generation_report.json")

with open(manifest_path, "r") as f:
    manifest = json.load(f)

with open(generation_path, "r") as f:
    generation_report = json.load(f)

print("=" * 70)
print("MANIFEST / GENERATION REPORT: MINIMAL-PAIR INFORMATION")
print("=" * 70)

def search_recursive(obj, path="root"):
    results = []

    if isinstance(obj, dict):
        for key, value in obj.items():

            key_text = str(key).lower()
            value_text = str(value).lower()

            if (
                "pair" in key_text
                or "minimal" in key_text
                or "24" in key_text
                or "pair" in value_text
                or "minimal" in value_text
            ):
                results.append(
                    (path + "." + str(key), value)
                )

            results.extend(
                search_recursive(
                    value,
                    path + "." + str(key)
                )
            )

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            results.extend(
                search_recursive(
                    value,
                    path + f"[{i}]"
                )
            )

    return results


for name, obj in [
    ("MANIFEST", manifest),
    ("GENERATION REPORT", generation_report)
]:

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    hits = search_recursive(obj)

    if not hits:
        print("No pair/minimal-pair information found.")

    for path, value in hits:

        text = repr(value)

        if len(text) > 1500:
            text = text[:1500] + "..."

        print(f"\n{path}")
        print(text)

MANIFEST / GENERATION REPORT: MINIMAL-PAIR INFORMATION

MANIFEST
No pair/minimal-pair information found.

GENERATION REPORT

root.build
{'admission': {'admitted_groups': {'pair': 238, 'single': 811}, 'design_mismatch_by_family': {}, 'drops': {'oq21_sensitive_group': 964, 'scenario_full': 15988}, 'gold_units': 1079, 'sensitivity_units_tier_p': 26}, 'render_drops': {'d5_signature_cap': 102, 'orphaned_pair_member': 46}, 'selection_drops': {'family_cap_non_pair_unit': 208}, 'trim_drops': {'family_cap_non_pair': 7, 'family_cap_pair': 32}}

root.build.admission
{'admitted_groups': {'pair': 238, 'single': 811}, 'design_mismatch_by_family': {}, 'drops': {'oq21_sensitive_group': 964, 'scenario_full': 15988}, 'gold_units': 1079, 'sensitivity_units_tier_p': 26}

root.build.admission.admitted_groups
{'pair': 238, 'single': 811}

root.build.admission.admitted_groups.pair
238

root.build.render_drops
{'d5_signature_cap': 102, 'orphaned_pair_member': 46}

root.build.render_drops.orphaned_pair_member


In [72]:
import json
import os
import numpy as np
from collections import defaultdict, Counter
from sklearn.metrics import balanced_accuracy_score, f1_score

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"

PRED_PATH = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_predictions.jsonl"
)

# ============================================================
# Load saved predictions
# ============================================================

pred_by_id = {}

with open(PRED_PATH, "r") as f:
    for line in f:
        p = json.loads(line)
        pred_by_id[p["record_id"]] = p

print("=" * 70)
print("MINIMAL-PAIR EVALUATION")
print("=" * 70)

print("Saved predictions:", len(pred_by_id))

# ============================================================
# Get test pair-member records
# ============================================================

test_pair_records = [
    records[i]
    for i in test_idx
    if records[i].get("meta", {}).get("example_kind")
    == "pair_member"
]

print("Test pair-member records:", len(test_pair_records))

# ============================================================
# Normalize pair ID -> underlying family
# ============================================================

def normalize_pair_id(pair_id):

    for suffix in ["-compact", "-kv", "-prose"]:
        if pair_id.endswith(suffix):
            return pair_id[:-len(suffix)]

    return pair_id


family_to_records = defaultdict(list)

for r in test_pair_records:

    family = normalize_pair_id(
        r["pair_id"]
    )

    family_to_records[family].append(r)

# ============================================================
# Select 24 underlying test pair families
# ============================================================

all_families = sorted(
    family_to_records.keys()
)

selected_families = all_families[:24]

print("\nTotal test families:", len(all_families))
print("Selected families:", len(selected_families))

print("\nExcluded families:")
for family in all_families[24:]:
    print(" ", family)

# ============================================================
# Build selected records
# ============================================================

selected_records = []

for family in selected_families:
    selected_records.extend(
        family_to_records[family]
    )

print(
    "\nSelected records:",
    len(selected_records)
)

# ============================================================
# Evaluate each pair/render instance
# ============================================================

pair_results = []

for family in selected_families:

    members = family_to_records[family]

    render_groups = defaultdict(list)

    for r in members:
        render_groups[r["render_style"]].append(r)

    for render_style, rs in sorted(
        render_groups.items()
    ):

        if len(rs) != 2:
            print(
                "WARNING:",
                family,
                render_style,
                "has",
                len(rs),
                "records"
            )
            continue

        true_labels = [
            1 if r["label"] == "irreversible"
            else 0
            for r in rs
        ]

        pred_labels = [
            pred_by_id[r["id"]]["predicted_label"]
            for r in rs
        ]

        valid_pair = (
            len(set(true_labels)) == 2
        )

        if valid_pair:

            correct = (
                pred_labels == true_labels
            )

            contrast_preserved = (
                len(set(pred_labels)) == 2
                and (
                    sorted(pred_labels)
                    == sorted(true_labels)
                )
            )

        else:

            correct = False
            contrast_preserved = False

        pair_results.append({
            "family": family,
            "render_style": render_style,
            "ids": [r["id"] for r in rs],
            "true_labels": true_labels,
            "pred_labels": pred_labels,
            "valid_pair": valid_pair,
            "correct": correct,
            "contrast_preserved": contrast_preserved
        })

# ============================================================
# Pair-level metrics
# ============================================================

valid_results = [
    x for x in pair_results
    if x["valid_pair"]
]

correct_results = [
    x for x in valid_results
    if x["correct"]
]

contrast_results = [
    x for x in valid_results
    if x["contrast_preserved"]
]

pair_accuracy = (
    len(correct_results)
    / len(valid_results)
)

contrast_accuracy = (
    len(contrast_results)
    / len(valid_results)
)

# ============================================================
# Record-level metrics on selected subset
# ============================================================

selected_true = []
selected_pred = []

for r in selected_records:

    selected_true.append(
        1 if r["label"] == "irreversible"
        else 0
    )

    selected_pred.append(
        pred_by_id[r["id"]]["predicted_label"]
    )

selected_true = np.array(selected_true)
selected_pred = np.array(selected_pred)

ba = balanced_accuracy_score(
    selected_true,
    selected_pred
)

macro_f1 = f1_score(
    selected_true,
    selected_pred,
    average="macro"
)

# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)

print(
    f"Valid minimal pairs: "
    f"{len(valid_results)}"
)

print(
    f"Correct pairs: "
    f"{len(correct_results)}"
)

print(
    f"Pair accuracy: "
    f"{pair_accuracy:.4f}"
)

print(
    f"Contrast-preservation accuracy: "
    f"{contrast_accuracy:.4f}"
)

print(
    f"Selected-subset Balanced Accuracy: "
    f"{ba:.4f}"
)

print(
    f"Selected-subset Macro-F1: "
    f"{macro_f1:.4f}"
)

# ============================================================
# BREAKDOWN BY RENDER STYLE
# ============================================================

print("\n" + "=" * 70)
print("BREAKDOWN BY RENDER STYLE")
print("=" * 70)

for style in ["compact", "kv", "prose"]:

    subset = [
        x for x in valid_results
        if x["render_style"] == style
    ]

    correct = sum(
        x["correct"]
        for x in subset
    )

    contrast = sum(
        x["contrast_preserved"]
        for x in subset
    )

    print(
        f"{style:8s} "
        f"pairs={len(subset):2d} "
        f"correct={correct:2d} "
        f"pair_acc={correct/len(subset):.4f} "
        f"contrast={contrast/len(subset):.4f}"
    )

# ============================================================
# Save results
# ============================================================

output = {
    "analysis": "minimal_pair_24_family_subset",
    "selection_rule":
        "first_24_underlying_test_pair_families_sorted",
    "total_test_families": len(all_families),
    "selected_families": selected_families,
    "excluded_families": all_families[24:],
    "selected_records": len(selected_records),
    "valid_pair_render_instances": len(valid_results),
    "correct_pair_render_instances": len(correct_results),
    "pair_accuracy": float(pair_accuracy),
    "contrast_preservation_accuracy":
        float(contrast_accuracy),
    "balanced_accuracy": float(ba),
    "macro_f1": float(macro_f1),
    "pair_results": pair_results
}

OUT_PATH = os.path.join(
    WORK_DIR,
    "analysis",
    "qwen_v2_minimal_pair_24_results.json"
)

with open(OUT_PATH, "w") as f:
    json.dump(
        output,
        f,
        indent=2
    )

print("\nSaved:")
print(OUT_PATH)

MINIMAL-PAIR EVALUATION
Saved predictions: 425
Test pair-member records: 162

Total test families: 27
Selected families: 24

Excluded families:
  v2b4_second_payment_policy-0097-primary
  v2b4_second_payment_policy-0141-primary
  v2b4_second_payment_policy-0187-primary

Selected records: 144

RESULTS
Valid minimal pairs: 63
Correct pairs: 20
Pair accuracy: 0.3175
Contrast-preservation accuracy: 0.3175
Selected-subset Balanced Accuracy: 0.7346
Selected-subset Macro-F1: 0.6921

BREAKDOWN BY RENDER STYLE
compact  pairs=21 correct= 2 pair_acc=0.0952 contrast=0.0952
kv       pairs=21 correct=11 pair_acc=0.5238 contrast=0.5238
prose    pairs=21 correct= 7 pair_acc=0.3333 contrast=0.3333

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/analysis/qwen_v2_minimal_pair_24_results.json


In [73]:
import json
import os
from collections import Counter, defaultdict

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"

RESULT_PATH = os.path.join(
    WORK_DIR,
    "analysis",
    "qwen_v2_minimal_pair_24_results.json"
)

with open(RESULT_PATH, "r") as f:
    mp_results = json.load(f)

pair_results = [
    x for x in mp_results["pair_results"]
    if x["valid_pair"]
]

print("=" * 70)
print("MINIMAL-PAIR FAILURE ANALYSIS")
print("=" * 70)

print("Valid pair-render instances:", len(pair_results))
print("Correct:", sum(x["correct"] for x in pair_results))
print("Failed:", sum(not x["correct"] for x in pair_results))

# ============================================================
# 1. FAILURE BY RENDER STYLE
# ============================================================

print("\n" + "=" * 70)
print("FAILURES BY RENDER STYLE")
print("=" * 70)

for style in ["compact", "kv", "prose"]:

    subset = [
        x for x in pair_results
        if x["render_style"] == style
    ]

    failures = [
        x for x in subset
        if not x["correct"]
    ]

    print(
        f"{style:8s} "
        f"total={len(subset):2d} "
        f"correct={len(subset)-len(failures):2d} "
        f"failed={len(failures):2d}"
    )

# ============================================================
# 2. FAILURE BY LABEL TRANSITION
# ============================================================

print("\n" + "=" * 70)
print("FAILURES BY TRUE LABEL TRANSITION")
print("=" * 70)

transition_total = Counter()
transition_failed = Counter()

for x in pair_results:

    labels = sorted(x["true_labels"])

    transition = f"{labels[0]} -> {labels[1]}"

    transition_total[transition] += 1

    if not x["correct"]:
        transition_failed[transition] += 1

for transition in sorted(transition_total):

    total = transition_total[transition]
    failed = transition_failed[transition]

    print(
        f"{transition:12s} "
        f"total={total:2d} "
        f"failed={failed:2d} "
        f"accuracy={(total-failed)/total:.4f}"
    )

# ============================================================
# 3. LOAD ORIGINAL RECORDS
# ============================================================

record_by_id = {
    r["id"]: r
    for r in records
}

# ============================================================
# 4. FAILURE BY ACTION VERB
# ============================================================

verb_total = Counter()
verb_failed = Counter()

for x in pair_results:

    for record_id in x["ids"]:

        r = record_by_id[record_id]

        verb = r["action_verb"]

        verb_total[verb] += 1

        if not x["correct"]:
            verb_failed[verb] += 1

print("\n" + "=" * 70)
print("FAILURES BY ACTION VERB")
print("=" * 70)

for verb, total in verb_total.most_common():

    failed = verb_failed[verb]

    print(
        f"{verb:22s} "
        f"records={total:3d} "
        f"failed_pair_records={failed:3d} "
        f"failure_rate={failed/total:.4f}"
    )

# ============================================================
# 5. FAILURE BY FAMILY
# ============================================================

family_total = Counter()
family_failed = Counter()

for x in pair_results:

    family = x["family"]

    family_total[family] += 1

    if not x["correct"]:
        family_failed[family] += 1

print("\n" + "=" * 70)
print("FAILURES BY FAMILY")
print("=" * 70)

for family, total in family_total.most_common():

    failed = family_failed[family]

    print(
        f"{family:55s} "
        f"total={total} "
        f"failed={failed}"
    )

# ============================================================
# 6. PRINT EVERY FAILED PAIR
# ============================================================

print("\n" + "=" * 70)
print("FAILED PAIR-RENDER INSTANCES")
print("=" * 70)

for i, x in enumerate(pair_results):

    if x["correct"]:
        continue

    print("\n" + "-" * 70)

    print(
        f"#{i+1} | "
        f"Family: {x['family']} | "
        f"Render: {x['render_style']}"
    )

    for record_id, true_label, pred_label in zip(
        x["ids"],
        x["true_labels"],
        x["pred_labels"]
    ):

        r = record_by_id[record_id]

        print("\nRecord:", record_id)

        print(
            "True:",
            r["label"],
            "| Binary:",
            true_label
        )

        print(
            "Predicted:",
            pred_label
        )

        print(
            "Action verb:",
            r["action_verb"]
        )

        print(
            "Pair ID:",
            r["pair_id"]
        )

        print(
            "Environment:",
            r["environment"]
        )

        print(
            "Task:",
            r["task_text"][:300]
        )

        print(
            "Action:",
            r["action_text"][:300]
        )

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

MINIMAL-PAIR FAILURE ANALYSIS
Valid pair-render instances: 63
Correct: 20
Failed: 43

FAILURES BY RENDER STYLE
compact  total=21 correct= 2 failed=19
kv       total=21 correct=11 failed=10
prose    total=21 correct= 7 failed=14

FAILURES BY TRUE LABEL TRANSITION
0 -> 1       total=63 failed=43 accuracy=0.3175

FAILURES BY ACTION VERB
transfer_money         records= 84 failed_pair_records= 44 failure_rate=0.5238
restore_backup         records= 42 failed_pair_records= 42 failure_rate=1.0000

FAILURES BY FAMILY
v2a1_restore_backup_fresh-0001-primary                  total=3 failed=3
v2a1_restore_backup_fresh-0005-primary                  total=3 failed=3
v2a1_restore_backup_fresh-0006-primary                  total=3 failed=3
v2a1_restore_backup_fresh-0008-primary                  total=3 failed=3
v2a1_restore_backup_fresh-0010-primary                  total=3 failed=3
v2a1_restore_backup_fresh-0016-primary                  total=3 failed=3
v2a1_restore_backup_fresh-0124-primary          

In [74]:
from collections import Counter

pair_results = [
    x for x in mp_results["pair_results"]
    if x["valid_pair"]
]

record_by_id = {
    r["id"]: r
    for r in records
}

# ------------------------------------------------------------
# Count each pair once, using the first member's action verb
# ------------------------------------------------------------

pair_verb_total = Counter()
pair_verb_failed = Counter()
pair_family_total = Counter()
pair_family_failed = Counter()

for x in pair_results:

    verb = record_by_id[x["ids"][0]]["action_verb"]

    pair_verb_total[verb] += 1

    if not x["correct"]:
        pair_verb_failed[verb] += 1

    pair_family_total[x["family"]] += 1

    if not x["correct"]:
        pair_family_failed[x["family"]] += 1

# ------------------------------------------------------------
# Print action-level summary
# ------------------------------------------------------------

print("=" * 75)
print("MINIMAL-PAIR SUMMARY BY ACTION VERB")
print("=" * 75)

for verb in sorted(pair_verb_total):

    total = pair_verb_total[verb]
    failed = pair_verb_failed[verb]
    correct = total - failed

    print(
        f"{verb:25s} "
        f"total={total:2d} "
        f"correct={correct:2d} "
        f"failed={failed:2d} "
        f"accuracy={correct/total:.3f}"
    )

# ------------------------------------------------------------
# Overall concentration
# ------------------------------------------------------------

total_failed = sum(pair_verb_failed.values())

print("\n" + "=" * 75)
print("FAILURE CONCENTRATION")
print("=" * 75)

for verb, failed in pair_verb_failed.most_common():

    print(
        f"{verb:25s} "
        f"failed={failed:2d} "
        f"of_all_failures={failed/total_failed:.3f}"
    )

# ------------------------------------------------------------
# Print only families with failures
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FAMILIES WITH FAILURES")
print("=" * 75)

for family in sorted(pair_family_total):

    total = pair_family_total[family]
    failed = pair_family_failed[family]

    if failed > 0:

        print(
            f"{family:55s} "
            f"total={total:2d} "
            f"failed={failed:2d} "
            f"accuracy={(total-failed)/total:.3f}"
        )

print("\nTotal valid pair-render instances:", len(pair_results))
print("Total failed:", total_failed)
print("Total correct:", len(pair_results) - total_failed)

MINIMAL-PAIR SUMMARY BY ACTION VERB
restore_backup            total=21 correct= 0 failed=21 accuracy=0.000
transfer_money            total=42 correct=20 failed=22 accuracy=0.476

FAILURE CONCENTRATION
transfer_money            failed=22 of_all_failures=0.512
restore_backup            failed=21 of_all_failures=0.488

FAMILIES WITH FAILURES
v2a1_restore_backup_fresh-0001-primary                  total= 3 failed= 3 accuracy=0.000
v2a1_restore_backup_fresh-0005-primary                  total= 3 failed= 3 accuracy=0.000
v2a1_restore_backup_fresh-0006-primary                  total= 3 failed= 3 accuracy=0.000
v2a1_restore_backup_fresh-0008-primary                  total= 3 failed= 3 accuracy=0.000
v2a1_restore_backup_fresh-0010-primary                  total= 3 failed= 3 accuracy=0.000
v2a1_restore_backup_fresh-0016-primary                  total= 3 failed= 3 accuracy=0.000
v2a1_restore_backup_fresh-0124-primary                  total= 3 failed= 3 accuracy=0.000
v2b4_second_payment_policy-00

In [75]:
import json
import os
import numpy as np
from collections import Counter, defaultdict
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"

PRED_PATH = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_predictions.jsonl"
)

# ============================================================
# Load predictions
# ============================================================

pred_by_id = {}

with open(PRED_PATH, "r") as f:
    for line in f:
        p = json.loads(line)
        pred_by_id[p["record_id"]] = p

# ============================================================
# Map records
# ============================================================

record_by_id = {
    r["id"]: r
    for r in records
}

# ============================================================
# Group test predictions by environment
# ============================================================

environment_data = defaultdict(list)

for r in records:

    record_id = r["id"]

    if record_id not in pred_by_id:
        continue

    p = pred_by_id[record_id]

    environment = r.get(
        "environment",
        "unknown"
    )

    environment_data[environment].append({
        "record_id": record_id,
        "true": int(p["true_label"]),
        "pred": int(p["predicted_label"]),
        "prob": float(p["probability_irreversible"]),
        "label": r["label"]
    })

# ============================================================
# Calculate metrics
# ============================================================

results = {}

for environment, rows in sorted(
    environment_data.items()
):

    y_true = np.array(
        [x["true"] for x in rows]
    )

    y_pred = np.array(
        [x["pred"] for x in rows]
    )

    y_prob = np.array(
        [x["prob"] for x in rows]
    )

    result = {
        "n": len(rows),
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro"
            )
        ),
        "class_counts": {
            "non_irreversible": int(
                np.sum(y_true == 0)
            ),
            "irreversible": int(
                np.sum(y_true == 1)
            )
        }
    }

    # AUROC requires both classes
    if len(np.unique(y_true)) == 2:
        result["auroc"] = float(
            roc_auc_score(
                y_true,
                y_prob
            )
        )
    else:
        result["auroc"] = None

    results[environment] = result

# ============================================================
# Print
# ============================================================

print("=" * 90)
print("QWEN V2.0 BINARY PROBE — ENVIRONMENT BREAKDOWN")
print("=" * 90)

print(
    f"{'Environment':20s}"
    f"{'N':>6s}"
    f"{'BA':>10s}"
    f"{'Macro-F1':>12s}"
    f"{'Accuracy':>12s}"
    f"{'AUROC':>10s}"
)

print("-" * 90)

for environment, result in results.items():

    auroc = (
        f"{result['auroc']:.4f}"
        if result["auroc"] is not None
        else "N/A"
    )

    print(
        f"{environment:20s}"
        f"{result['n']:6d}"
        f"{result['balanced_accuracy']:10.4f}"
        f"{result['macro_f1']:12.4f}"
        f"{result['accuracy']:12.4f}"
        f"{auroc:>10s}"
    )

# ============================================================
# Class distributions
# ============================================================

print("\n" + "=" * 90)
print("CLASS DISTRIBUTION BY ENVIRONMENT")
print("=" * 90)

for environment, result in results.items():

    counts = result["class_counts"]

    print(
        f"{environment:20s}"
        f"non_irreversible={counts['non_irreversible']:4d} "
        f"irreversible={counts['irreversible']:4d}"
    )

# ============================================================
# Save
# ============================================================

OUT_PATH = os.path.join(
    WORK_DIR,
    "analysis",
    "qwen_v2_binary_environment_breakdown.json"
)

with open(OUT_PATH, "w") as f:
    json.dump(
        results,
        f,
        indent=2
    )

print("\nSaved:")
print(OUT_PATH)

QWEN V2.0 BINARY PROBE — ENVIRONMENT BREAKDOWN
Environment              N        BA    Macro-F1    Accuracy     AUROC
------------------------------------------------------------------------------------------
sandbox                424    0.9114      0.8836      0.8868    0.9864
scienceworld             1    1.0000      1.0000      1.0000       N/A

CLASS DISTRIBUTION BY ENVIRONMENT
sandbox             non_irreversible= 271 irreversible= 153
scienceworld        non_irreversible=   1 irreversible=   0

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/analysis/qwen_v2_binary_environment_breakdown.json


In [76]:
import json
import os
from collections import Counter

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"

RESULT_PATH = os.path.join(
    WORK_DIR,
    "analysis",
    "qwen_v2_minimal_pair_24_results.json"
)

with open(RESULT_PATH, "r") as f:
    mp_results = json.load(f)

pair_results = [
    x for x in mp_results["pair_results"]
    if x["valid_pair"]
]

print("=" * 75)
print("MINIMAL-PAIR RENDER-STYLE ANALYSIS")
print("=" * 75)

styles = ["compact", "kv", "prose"]

render_summary = {}

for style in styles:

    subset = [
        x for x in pair_results
        if x["render_style"] == style
    ]

    total = len(subset)
    correct = sum(
        x["correct"]
        for x in subset
    )
    failed = total - correct

    accuracy = correct / total

    render_summary[style] = {
        "valid_pair_instances": total,
        "correct": correct,
        "failed": failed,
        "pair_accuracy": accuracy
    }

    print(
        f"{style:10s} "
        f"N={total:2d} "
        f"correct={correct:2d} "
        f"failed={failed:2d} "
        f"pair_accuracy={accuracy:.4f}"
    )

# ------------------------------------------------------------
# Relative differences
# ------------------------------------------------------------

compact = render_summary["compact"]["pair_accuracy"]
kv = render_summary["kv"]["pair_accuracy"]
prose = render_summary["prose"]["pair_accuracy"]

print("\n" + "=" * 75)
print("RENDER-STYLE DIFFERENCES")
print("=" * 75)

print(
    f"KV - Compact:   {kv - compact:+.4f}"
)

print(
    f"Prose - Compact: {prose - compact:+.4f}"
)

print(
    f"KV - Prose:     {kv - prose:+.4f}"
)

# ------------------------------------------------------------
# Save clean summary
# ------------------------------------------------------------

OUT_PATH = os.path.join(
    WORK_DIR,
    "analysis",
    "qwen_v2_minimal_pair_render_style.json"
)

with open(OUT_PATH, "w") as f:
    json.dump(
        {
            "source":
                "qwen_v2_minimal_pair_24_results.json",
            "render_style_results":
                render_summary,
            "differences": {
                "kv_minus_compact":
                    kv - compact,
                "prose_minus_compact":
                    prose - compact,
                "kv_minus_prose":
                    kv - prose
            }
        },
        f,
        indent=2
    )

print("\nSaved:")
print(OUT_PATH)

MINIMAL-PAIR RENDER-STYLE ANALYSIS
compact    N=21 correct= 2 failed=19 pair_accuracy=0.0952
kv         N=21 correct=11 failed=10 pair_accuracy=0.5238
prose      N=21 correct= 7 failed=14 pair_accuracy=0.3333

RENDER-STYLE DIFFERENCES
KV - Compact:   +0.4286
Prose - Compact: +0.2381
KV - Prose:     +0.1905

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/analysis/qwen_v2_minimal_pair_render_style.json


In [77]:
from collections import defaultdict, Counter

print("=" * 80)
print("TEST MINIMAL-PAIR FAMILY METADATA")
print("=" * 80)

family_records = defaultdict(list)

for i in test_idx:

    r = records[i]

    if r.get("meta", {}).get("example_kind") == "pair_member":

        family = normalize_pair_id(
            r["pair_id"]
        )

        family_records[family].append(r)

for family in sorted(family_records):

    rs = family_records[family]

    labels = Counter(
        r["label"]
        for r in rs
    )

    verbs = Counter(
        r["action_verb"]
        for r in rs
    )

    render_styles = sorted(
        set(
            r["render_style"]
            for r in rs
        )
    )

    pair_ids = sorted(
        set(
            r["pair_id"]
            for r in rs
        )
    )

    print("\n" + "-" * 80)
    print("FAMILY:", family)
    print("Records:", len(rs))
    print("Labels:", dict(labels))
    print("Action verbs:", dict(verbs))
    print("Render styles:", render_styles)
    print("Pair IDs:", pair_ids)

    # Show template IDs
    templates = sorted(
        set(
            r["template_id"]
            for r in rs
        )
    )

    print("Templates:", templates)

    # Show sensitivity/example metadata if present
    example_kinds = sorted(
        set(
            r.get("meta", {}).get("example_kind")
            for r in rs
        )
    )

    print("Example kinds:", example_kinds)

print("\n" + "=" * 80)
print("TOTAL TEST FAMILIES:", len(family_records))
print("=" * 80)

TEST MINIMAL-PAIR FAMILY METADATA

--------------------------------------------------------------------------------
FAMILY: s01_link_revoke_address_kind-0000-link
Records: 6
Labels: {'reversible': 3, 'partially_reversible': 3}
Action verbs: {'revoke_share_link': 6}
Render styles: ['compact', 'kv', 'prose']
Pair IDs: ['s01_link_revoke_address_kind-0000-link-compact', 's01_link_revoke_address_kind-0000-link-kv', 's01_link_revoke_address_kind-0000-link-prose']
Templates: ['s01_link_revoke_address_kind']
Example kinds: ['pair_member']

--------------------------------------------------------------------------------
FAMILY: s01_link_revoke_address_kind-0001-link
Records: 6
Labels: {'reversible': 3, 'partially_reversible': 3}
Action verbs: {'revoke_share_link': 6}
Render styles: ['compact', 'kv', 'prose']
Pair IDs: ['s01_link_revoke_address_kind-0001-link-compact', 's01_link_revoke_address_kind-0001-link-kv', 's01_link_revoke_address_kind-0001-link-prose']
Templates: ['s01_link_revoke_addres

In [78]:
import json
import os
from collections import defaultdict
from sklearn.metrics import balanced_accuracy_score, f1_score

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"

PRED_PATH = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_predictions.jsonl"
)

# ============================================================
# Load predictions
# ============================================================

pred_by_id = {}

with open(PRED_PATH, "r") as f:
    for line in f:
        p = json.loads(line)
        pred_by_id[p["record_id"]] = p

# ============================================================
# Build test pair families
# ============================================================

family_records = defaultdict(list)

for i in test_idx:

    r = records[i]

    if r.get("meta", {}).get("example_kind") != "pair_member":
        continue

    family = normalize_pair_id(r["pair_id"])

    family_records[family].append(r)

# ============================================================
# Canonical 24-family selection
#
# Evidence-based structure:
#   7 v2a1_restore_backup_fresh
#   17 v2b4_second_payment_policy
#   = 24 families
#
# The 3 s01 share-link families are excluded.
# ============================================================

canonical_families = sorted(
    family
    for family in family_records
    if (
        family.startswith(
            "v2a1_restore_backup_fresh-"
        )
        or
        family.startswith(
            "v2b4_second_payment_policy-"
        )
    )
)

excluded_families = sorted(
    family
    for family in family_records
    if family not in canonical_families
)

print("=" * 80)
print("CANONICAL 24-FAMILY MINIMAL-PAIR ANALYSIS")
print("=" * 80)

print(
    "Canonical families:",
    len(canonical_families)
)

print(
    "Excluded test families:",
    len(excluded_families)
)

print("\nExcluded:")
for family in excluded_families:
    print(" ", family)

assert len(canonical_families) == 24
assert len(excluded_families) == 3

# ============================================================
# Evaluate each family × render style
# ============================================================

pair_results = []

for family in canonical_families:

    members = family_records[family]

    render_groups = defaultdict(list)

    for r in members:
        render_groups[r["render_style"]].append(r)

    for render_style in ["compact", "kv", "prose"]:

        rs = render_groups[render_style]

        assert len(rs) == 2

        true_labels = [
            1 if r["label"] == "irreversible"
            else 0
            for r in rs
        ]

        pred_labels = [
            pred_by_id[r["id"]]["predicted_label"]
            for r in rs
        ]

        # Pair is valid when the two members have
        # different binary labels.
        valid_pair = (
            len(set(true_labels)) == 2
        )

        correct = (
            valid_pair
            and pred_labels == true_labels
        )

        contrast_preserved = (
            valid_pair
            and len(set(pred_labels)) == 2
            and sorted(pred_labels)
               == sorted(true_labels)
        )

        pair_results.append({
            "family": family,
            "render_style": render_style,
            "ids": [r["id"] for r in rs],
            "true_labels": true_labels,
            "pred_labels": pred_labels,
            "valid_pair": valid_pair,
            "correct": correct,
            "contrast_preserved":
                contrast_preserved
        })

# ============================================================
# Pair-level results
# ============================================================

valid_results = [
    x for x in pair_results
    if x["valid_pair"]
]

correct_results = [
    x for x in valid_results
    if x["correct"]
]

contrast_results = [
    x for x in valid_results
    if x["contrast_preserved"]
]

pair_accuracy = (
    len(correct_results)
    / len(valid_results)
)

contrast_accuracy = (
    len(contrast_results)
    / len(valid_results)
)

print("\n" + "=" * 80)
print("OVERALL RESULTS")
print("=" * 80)

print(
    "Underlying families:",
    len(canonical_families)
)

print(
    "Expected pair-render instances:",
    24 * 3
)

print(
    "Valid pair-render instances:",
    len(valid_results)
)

print(
    "Correct:",
    len(correct_results)
)

print(
    "Failed:",
    len(valid_results) - len(correct_results)
)

print(
    f"Pair accuracy: "
    f"{pair_accuracy:.4f}"
)

print(
    f"Contrast preservation: "
    f"{contrast_accuracy:.4f}"
)

# ============================================================
# Breakdown by action
# ============================================================

print("\n" + "=" * 80)
print("BY ACTION FAMILY")
print("=" * 80)

for prefix, name in [
    (
        "v2a1_restore_backup_fresh-",
        "restore_backup"
    ),
    (
        "v2b4_second_payment_policy-",
        "transfer_money"
    )
]:

    subset = [
        x for x in valid_results
        if x["family"].startswith(prefix)
    ]

    correct = sum(
        x["correct"]
        for x in subset
    )

    print(
        f"{name:20s} "
        f"N={len(subset):2d} "
        f"correct={correct:2d} "
        f"failed={len(subset)-correct:2d} "
        f"accuracy={correct/len(subset):.4f}"
    )

# ============================================================
# Breakdown by render style
# ============================================================

print("\n" + "=" * 80)
print("BY RENDER STYLE")
print("=" * 80)

render_summary = {}

for style in ["compact", "kv", "prose"]:

    subset = [
        x for x in valid_results
        if x["render_style"] == style
    ]

    correct = sum(
        x["correct"]
        for x in subset
    )

    accuracy = correct / len(subset)

    render_summary[style] = {
        "n": len(subset),
        "correct": correct,
        "failed": len(subset) - correct,
        "pair_accuracy": accuracy
    }

    print(
        f"{style:10s} "
        f"N={len(subset):2d} "
        f"correct={correct:2d} "
        f"failed={len(subset)-correct:2d} "
        f"accuracy={accuracy:.4f}"
    )

# ============================================================
# Record-level metrics on canonical subset
# ============================================================

selected_records = []

for family in canonical_families:
    selected_records.extend(
        family_records[family]
    )

y_true = []
y_pred = []

for r in selected_records:

    y_true.append(
        1 if r["label"] == "irreversible"
        else 0
    )

    y_pred.append(
        pred_by_id[r["id"]]["predicted_label"]
    )

ba = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print("\n" + "=" * 80)
print("RECORD-LEVEL METRICS")
print("=" * 80)

print(
    "Selected records:",
    len(selected_records)
)

print(
    f"Balanced Accuracy: {ba:.4f}"
)

print(
    f"Macro-F1: {macro_f1:.4f}"
)

# ============================================================
# Save canonical result
# ============================================================

output = {
    "analysis":
        "canonical_24_minimal_pair_subset",

    "selection_basis":
        "7 v2a1_restore_backup_fresh families "
        "+ 17 v2b4_second_payment_policy families",

    "canonical_family_count":
        len(canonical_families),

    "canonical_families":
        canonical_families,

    "excluded_families":
        excluded_families,

    "valid_pair_render_instances":
        len(valid_results),

    "correct_pair_render_instances":
        len(correct_results),

    "pair_accuracy":
        float(pair_accuracy),

    "contrast_preservation_accuracy":
        float(contrast_accuracy),

    "record_level_balanced_accuracy":
        float(ba),

    "record_level_macro_f1":
        float(macro_f1),

    "render_style_results":
        render_summary,

    "pair_results":
        pair_results
}

OUT_PATH = os.path.join(
    WORK_DIR,
    "analysis",
    "qwen_v2_canonical_24_minimal_pairs.json"
)

with open(OUT_PATH, "w") as f:
    json.dump(
        output,
        f,
        indent=2
    )

print("\nSaved:")
print(OUT_PATH)

CANONICAL 24-FAMILY MINIMAL-PAIR ANALYSIS
Canonical families: 24
Excluded test families: 3

Excluded:
  s01_link_revoke_address_kind-0000-link
  s01_link_revoke_address_kind-0001-link
  s01_link_revoke_address_kind-0002-link

OVERALL RESULTS
Underlying families: 24
Expected pair-render instances: 72
Valid pair-render instances: 72
Correct: 25
Failed: 47
Pair accuracy: 0.3472
Contrast preservation: 0.3472

BY ACTION FAMILY
restore_backup       N=21 correct= 0 failed=21 accuracy=0.0000
transfer_money       N=51 correct=25 failed=26 accuracy=0.4902

BY RENDER STYLE
compact    N=24 correct= 3 failed=21 accuracy=0.1250
kv         N=24 correct=13 failed=11 accuracy=0.5417
prose      N=24 correct= 9 failed=15 accuracy=0.3750

RECORD-LEVEL METRICS
Selected records: 144
Balanced Accuracy: 0.6736
Macro-F1: 0.6347

Saved:
/kaggle/working/ADNN_RQ1_Qwen_v2/analysis/qwen_v2_canonical_24_minimal_pairs.json


In [82]:
import os
import json
import hashlib
from datetime import datetime, timezone

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"

PROBES = os.path.join(WORK_DIR, "probes")
BASELINES = os.path.join(WORK_DIR, "baselines")
ABLATIONS = os.path.join(WORK_DIR, "ablations")
ANALYSIS = os.path.join(WORK_DIR, "analysis")
HIDDEN = os.path.join(WORK_DIR, "hidden_states")


# ============================================================
# HELPERS
# ============================================================

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def file_info(path):

    if not os.path.exists(path):
        return None

    return {
        "path": os.path.relpath(path, WORK_DIR),
        "size_bytes": os.path.getsize(path),
        "sha256": sha256(path)
    }


# ============================================================
# LOAD EXISTING RESULTS
# ============================================================

three_class = load_json(
    os.path.join(
        PROBES,
        "qwen_v2_final_3class_results.json"
    )
)

binary = load_json(
    os.path.join(
        PROBES,
        "qwen_v2_final_binary_results.json"
    )
)

bootstrap = load_json(
    os.path.join(
        PROBES,
        "qwen_v2_binary_bootstrap_ci.json"
    )
)

minimal_pairs = load_json(
    os.path.join(
        ANALYSIS,
        "qwen_v2_canonical_24_minimal_pairs.json"
    )
)

render_style = load_json(
    os.path.join(
        ANALYSIS,
        "qwen_v2_minimal_pair_render_style.json"
    )
)

environment = load_json(
    os.path.join(
        ANALYSIS,
        "qwen_v2_binary_environment_breakdown.json"
    )
)


# ============================================================
# 1. BEST LAYERS
# ============================================================

best_layers = {

    "model":
        "Qwen/Qwen2.5-7B-Instruct",

    "three_class": {

        "selected_layer":
            three_class["selected_layer"],

        "selected_C":
            three_class["selected_C"],

        "inner_cv_balanced_accuracy":
            three_class[
                "inner_cv_balanced_accuracy"
            ],

        "selection_method":
            three_class[
                "cv_selection_source"
            ]
    },

    "binary": {

        "selected_layer":
            binary["selection"][
                "selected_layer"
            ],

        "selected_C":
            binary["selection"][
                "selected_C"
            ],

        "inner_cv_balanced_accuracy":
            binary["selection"][
                "inner_cv_balanced_accuracy"
            ],

        "inner_cv_sd":
            binary["selection"][
                "inner_cv_sd"
            ],

        "selection_method":
            binary["selection"]["method"]
    },

    "representation": {

        "primary":
            "last_action_token",

        "ablation":
            "mean_action_tokens"
    }
}


best_layers_path = os.path.join(
    PROBES,
    "best_layers.json"
)

with open(best_layers_path, "w") as f:

    json.dump(
        best_layers,
        f,
        indent=2
    )


# ============================================================
# 2. METRICS
# ============================================================

metrics = {

    "model":
        "Qwen/Qwen2.5-7B-Instruct",

    # --------------------------------------------------------
    # Primary 3-class
    # --------------------------------------------------------

    "primary_three_class": {

        "layer":
            three_class["selected_layer"],

        "C":
            three_class["selected_C"],

        "balanced_accuracy":
            three_class[
                "test_metrics"
            ]["balanced_accuracy"],

        "macro_f1":
            three_class[
                "test_metrics"
            ]["macro_f1"],

        "auroc":
            three_class[
                "test_metrics"
            ]["macro_auroc"],

        "per_class":
            three_class["per_class"],

        "confusion_matrix":
            three_class["confusion_matrix"]
    },


    # --------------------------------------------------------
    # Primary binary
    # --------------------------------------------------------

    "primary_binary": {

        "layer":
            binary["selection"][
                "selected_layer"
            ],

        "C":
            binary["selection"][
                "selected_C"
            ],

        "accuracy":
            binary["test_metrics"][
                "accuracy"
            ],

        "balanced_accuracy":
            binary["test_metrics"][
                "balanced_accuracy"
            ],

        "macro_f1":
            binary["test_metrics"][
                "macro_f1"
            ],

        "auroc":
            binary["test_metrics"][
                "auroc"
            ],

        "per_class":
            binary["test_metrics"][
                "per_class"
            ],

        "confusion_matrix":
            binary["test_metrics"][
                "confusion_matrix"
            ]
    },


    # --------------------------------------------------------
    # Bootstrap
    # --------------------------------------------------------

    "bootstrap_95_ci":
        bootstrap,


    # --------------------------------------------------------
    # Controls
    # --------------------------------------------------------

    "controls": {

        "majority":
            load_json(
                os.path.join(
                    BASELINES,
                    "qwen_v2_binary_majority_baseline.json"
                )
            ),

        "bow_action_text":
            load_json(
                os.path.join(
                    BASELINES,
                    "qwen_v2_binary_bow_action_text.json"
                )
            ),

        "bow_state_action":
            load_json(
                os.path.join(
                    BASELINES,
                    "qwen_v2_binary_bow_state_action.json"
                )
            ),

        "shuffled_labels":
            load_json(
                os.path.join(
                    BASELINES,
                    "qwen_v2_binary_shuffled_label_control.json"
                )
            ),

        "random_action_verb":
            load_json(
                os.path.join(
                    BASELINES,
                    "qwen_v2_binary_random_action_verb_control.json"
                )
            ),

        "layer0":
            load_json(
                os.path.join(
                    BASELINES,
                    "qwen_v2_binary_layer0_embedding_baseline.json"
                )
            )
    },


    # --------------------------------------------------------
    # Ablations
    # --------------------------------------------------------

    "ablations": {

        "mean_action_tokens":
            load_json(
                os.path.join(
                    ABLATIONS,
                    "qwen_v2_binary_mean_action_tokens_layer18.json"
                )
            ),

        "horizon":
            load_json(
                os.path.join(
                    ABLATIONS,
                    "qwen_v2_binary_horizon_ablations.json"
                )
            )
    },


    # --------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------

    "diagnostics": {

        "canonical_24_minimal_pairs": {

            "underlying_families":
                minimal_pairs[
                    "canonical_family_count"
                ],

            "valid_pair_render_instances":
                minimal_pairs[
                    "valid_pair_render_instances"
                ],

            "correct":
                minimal_pairs[
                    "correct_pair_render_instances"
                ],

            "pair_accuracy":
                minimal_pairs[
                    "pair_accuracy"
                ],

            "contrast_preservation_accuracy":
                minimal_pairs[
                    "contrast_preservation_accuracy"
                ],

            "record_level_balanced_accuracy":
                minimal_pairs[
                    "record_level_balanced_accuracy"
                ],

            "record_level_macro_f1":
                minimal_pairs[
                    "record_level_macro_f1"
                ]
        },

        "render_style":
            render_style,

        "environment":
            environment
    }
}


metrics_path = os.path.join(
    PROBES,
    "metrics.json"
)

with open(metrics_path, "w") as f:

    json.dump(
        metrics,
        f,
        indent=2
    )


# ============================================================
# 3. MANIFEST
# ============================================================

important_files = [

    # Hidden states
    os.path.join(
        HIDDEN,
        "last_action_token.npy"
    ),

    os.path.join(
        HIDDEN,
        "mean_action_tokens.npy"
    ),

    os.path.join(
        HIDDEN,
        "extraction_index.jsonl"
    ),

    os.path.join(
        HIDDEN,
        "extraction_metadata.json"
    ),

    # Primary probe outputs
    os.path.join(
        PROBES,
        "qwen_v2_final_3class_results.json"
    ),

    os.path.join(
        PROBES,
        "qwen_v2_final_binary_results.json"
    ),

    os.path.join(
        PROBES,
        "qwen_v2_binary_predictions.jsonl"
    ),

    os.path.join(
        PROBES,
        "qwen_v2_binary_bootstrap_ci.json"
    ),

    best_layers_path,
    metrics_path,

    # Diagnostics
    os.path.join(
        ANALYSIS,
        "qwen_v2_canonical_24_minimal_pairs.json"
    ),

    os.path.join(
        ANALYSIS,
        "qwen_v2_minimal_pair_render_style.json"
    ),

    os.path.join(
        ANALYSIS,
        "qwen_v2_binary_environment_breakdown.json"
    )
]


manifest_files = []

missing_files = []

for path in important_files:

    info = file_info(path)

    if info is None:
        missing_files.append(
            os.path.relpath(
                path,
                WORK_DIR
            )
        )

    else:
        manifest_files.append(info)


manifest = {

    "artifact":
        "ADNN RQ1 Qwen handoff",

    "model":
        "Qwen/Qwen2.5-7B-Instruct",

    "status":
        "preliminary_v1.0",

    "created_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "dataset": {

        "version":
            "adnn-dataset-2.0",

        "records":
            2134,

        "scenarios":
            149,

        "train_records":
            1709,

        "test_records":
            425,

        "split":
            "scenario_grouped_frozen"
    },

    "hidden_states": {

        "dtype":
            "float16",

        "layers":
            29,

        "hidden_size":
            3584,

        "representations": [
            "last_action_token",
            "mean_action_tokens"
        ]
    },

    "probe_pipeline": {

        "scaler":
            "StandardScaler",

        "classifier":
            "L2 LogisticRegression",

        "class_weight":
            "balanced",

        "group_cv":
            "scenario_id",

        "layer_selection":
            "inner CV only",

        "C_selection":
            "inner CV only",

        "test_usage":
            "single final evaluation"
    },

    "files":
        manifest_files,

    "missing_files":
        missing_files
}


manifest_path = os.path.join(
    PROBES,
    "manifest.json"
)

with open(manifest_path, "w") as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ============================================================
# 4. FINAL SUMMARY
# ============================================================

summary = {

    "model":
        "Qwen/Qwen2.5-7B-Instruct",

    "status":
        "preliminary_v1.0",

    "primary_result": {

        "three_class_BA":
            three_class[
                "test_metrics"
            ]["balanced_accuracy"],

        "three_class_macro_F1":
            three_class[
                "test_metrics"
            ]["macro_f1"],

        "three_class_AUROC":
            three_class[
                "test_metrics"
            ]["macro_auroc"],

        "binary_BA":
            binary[
                "test_metrics"
            ]["balanced_accuracy"],

        "binary_macro_F1":
            binary[
                "test_metrics"
            ]["macro_f1"],

        "binary_AUROC":
            binary[
                "test_metrics"
            ]["auroc"]
    },


    "critical_diagnostics": {

        "random_action_verb_BA":
            1.0,

        "canonical_24_pair_accuracy":
            minimal_pairs[
                "pair_accuracy"
            ],

        "restore_backup_pair_accuracy":
            0.0,

        "transfer_money_pair_accuracy":
            25 / 51,

        "compact_pair_accuracy":
            0.125,

        "kv_pair_accuracy":
            13 / 24,

        "prose_pair_accuracy":
            9 / 24
    },


    "interpretation":
        "Aggregate reversibility decoding is strong, "
        "but minimal-pair and random-action-verb controls "
        "indicate substantial entanglement with action "
        "identity, context, and rendering factors."
}


summary_path = os.path.join(
    ANALYSIS,
    "qwen_v2_rq1_summary.json"
)

with open(summary_path, "w") as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# PRINT FINAL STATUS
# ============================================================

print("=" * 80)
print("RQ1 QWEN HANDOFF CREATED")
print("=" * 80)

print("\nCreated:")
print("1.", best_layers_path)
print("2.", metrics_path)
print("3.", manifest_path)
print("4.", summary_path)

print(
    "\nTracked artifact files:",
    len(manifest_files)
)

print(
    "Missing artifact files:",
    len(missing_files)
)

if missing_files:

    for path in missing_files:
        print("  MISSING:", path)


print("\nPrimary results:")

print(
    f"3-class BA:    "
    f"{summary['primary_result']['three_class_BA']:.4f}"
)

print(
    f"3-class Macro-F1:"
    f" {summary['primary_result']['three_class_macro_F1']:.4f}"
)

print(
    f"3-class AUROC: "
    f"{summary['primary_result']['three_class_AUROC']:.4f}"
)

print(
    f"Binary BA:     "
    f"{summary['primary_result']['binary_BA']:.4f}"
)

print(
    f"Binary Macro-F1:"
    f" {summary['primary_result']['binary_macro_F1']:.4f}"
)

print(
    f"Binary AUROC:  "
    f"{summary['primary_result']['binary_AUROC']:.4f}"
)

print(
    "\nCanonical 24-pair accuracy:",
    f"{summary['critical_diagnostics']['canonical_24_pair_accuracy']:.4f}"
)

print("\nStatus: preliminary_v1.0")

RQ1 QWEN HANDOFF CREATED

Created:
1. /kaggle/working/ADNN_RQ1_Qwen_v2/probes/best_layers.json
2. /kaggle/working/ADNN_RQ1_Qwen_v2/probes/metrics.json
3. /kaggle/working/ADNN_RQ1_Qwen_v2/probes/manifest.json
4. /kaggle/working/ADNN_RQ1_Qwen_v2/analysis/qwen_v2_rq1_summary.json

Tracked artifact files: 13
Missing artifact files: 0

Primary results:
3-class BA:    0.9216
3-class Macro-F1: 0.9209
3-class AUROC: 0.9951
Binary BA:     0.9118
Binary Macro-F1: 0.8838
Binary AUROC:  0.9864

Canonical 24-pair accuracy: 0.3472

Status: preliminary_v1.0


In [83]:
import os
import json
import hashlib
import numpy as np

WORK_DIR = "/kaggle/working/ADNN_RQ1_Qwen_v2"


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("=" * 80)
print("FINAL RQ1 QWEN INTEGRITY CHECK")
print("=" * 80)


# ============================================================
# 1. MANIFEST HASH CHECK
# ============================================================

manifest_path = os.path.join(
    WORK_DIR,
    "probes",
    "manifest.json"
)

with open(manifest_path, "r") as f:
    manifest = json.load(f)

hash_pass = True

for item in manifest["files"]:

    path = os.path.join(
        WORK_DIR,
        item["path"]
    )

    exists = os.path.exists(path)

    if not exists:

        print(
            f"❌ MISSING: {item['path']}"
        )

        hash_pass = False
        continue

    actual_hash = sha256(path)

    if actual_hash == item["sha256"]:

        print(
            f"✓ HASH OK: {item['path']}"
        )

    else:

        print(
            f"❌ HASH MISMATCH: {item['path']}"
        )

        hash_pass = False


# ============================================================
# 2. JSON READABILITY
# ============================================================

json_files = [

    "probes/best_layers.json",
    "probes/metrics.json",
    "probes/manifest.json",
    "analysis/qwen_v2_rq1_summary.json"
]

json_pass = True

for rel_path in json_files:

    path = os.path.join(
        WORK_DIR,
        rel_path
    )

    try:

        with open(path, "r") as f:
            json.load(f)

        print(
            f"✓ JSON OK: {rel_path}"
        )

    except Exception as e:

        print(
            f"❌ JSON ERROR: {rel_path}"
        )

        print(e)

        json_pass = False


# ============================================================
# 3. HIDDEN STATE SHAPES
# ============================================================

hidden_pass = True

expected_shape = (
    2134,
    29,
    3584
)

for filename in [
    "last_action_token.npy",
    "mean_action_tokens.npy"
]:

    path = os.path.join(
        WORK_DIR,
        "hidden_states",
        filename
    )

    arr = np.load(
        path,
        mmap_mode="r"
    )

    print(
        f"{filename}: "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )

    if arr.shape != expected_shape:

        print(
            f"❌ WRONG SHAPE: expected "
            f"{expected_shape}"
        )

        hidden_pass = False

    if arr.dtype != np.float16:

        print(
            "❌ WRONG DTYPE: expected float16"
        )

        hidden_pass = False

    if arr.shape == expected_shape and arr.dtype == np.float16:

        print(
            f"✓ HIDDEN STATE OK: {filename}"
        )


# ============================================================
# 4. PREDICTION COUNT
# ============================================================

prediction_path = os.path.join(
    WORK_DIR,
    "probes",
    "qwen_v2_binary_predictions.jsonl"
)

prediction_count = 0

with open(prediction_path, "r") as f:

    for line in f:

        if line.strip():

            prediction_count += 1


print(
    "\nBinary prediction count:",
    prediction_count
)

prediction_pass = (
    prediction_count == 425
)

if prediction_pass:

    print(
        "✓ Prediction count OK"
    )

else:

    print(
        "❌ Expected 425 predictions"
    )


# ============================================================
# 5. FINAL SUMMARY
# ============================================================

all_pass = (
    hash_pass
    and json_pass
    and hidden_pass
    and prediction_pass
)

print("\n" + "=" * 80)

if all_pass:

    print(
        "✅ ALL INTEGRITY CHECKS PASSED"
    )

    print(
        "Qwen RQ1 preliminary_v1.0 handoff "
        "is internally consistent."
    )

else:

    print(
        "❌ SOME INTEGRITY CHECKS FAILED"
    )

print("=" * 80)

FINAL RQ1 QWEN INTEGRITY CHECK
✓ HASH OK: hidden_states/last_action_token.npy
✓ HASH OK: hidden_states/mean_action_tokens.npy
✓ HASH OK: hidden_states/extraction_index.jsonl
✓ HASH OK: hidden_states/extraction_metadata.json
✓ HASH OK: probes/qwen_v2_final_3class_results.json
✓ HASH OK: probes/qwen_v2_final_binary_results.json
✓ HASH OK: probes/qwen_v2_binary_predictions.jsonl
✓ HASH OK: probes/qwen_v2_binary_bootstrap_ci.json
✓ HASH OK: probes/best_layers.json
✓ HASH OK: probes/metrics.json
✓ HASH OK: analysis/qwen_v2_canonical_24_minimal_pairs.json
✓ HASH OK: analysis/qwen_v2_minimal_pair_render_style.json
✓ HASH OK: analysis/qwen_v2_binary_environment_breakdown.json
✓ JSON OK: probes/best_layers.json
✓ JSON OK: probes/metrics.json
✓ JSON OK: probes/manifest.json
✓ JSON OK: analysis/qwen_v2_rq1_summary.json
last_action_token.npy: shape=(2134, 29, 3584), dtype=float16
✓ HIDDEN STATE OK: last_action_token.npy
mean_action_tokens.npy: shape=(2134, 29, 3584), dtype=float16
✓ HIDDEN STATE O

In [84]:
# ============================================================
# ADNN RQ1 — COMPLETE PROJECT BACKUP v2
# ============================================================

import os
import shutil
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/ADNN_RQ1_Qwen_v2")

# New backup name so the previous backup is preserved
ZIP_BASE = Path("/kaggle/working/ADNN_RQ1_Qwen_v2_BACKUP_2")

print("=" * 70)
print("ADNN RQ1 — CREATING UPDATED COMPLETE BACKUP")
print("=" * 70)

assert PROJECT_DIR.exists(), (
    f"Project folder not found: {PROJECT_DIR}"
)

# ------------------------------------------------------------
# Verify critical files BEFORE creating backup
# ------------------------------------------------------------

critical_files = [

    # --------------------------------------------------------
    # Hidden states
    # --------------------------------------------------------

    PROJECT_DIR / "hidden_states" / "last_action_token.npy",
    PROJECT_DIR / "hidden_states" / "mean_action_tokens.npy",

    PROJECT_DIR / "hidden_states" / "extraction_index.jsonl",
    PROJECT_DIR / "hidden_states" / "extraction_manifest.jsonl",
    PROJECT_DIR / "hidden_states" / "extraction_metadata.json",
    PROJECT_DIR / "hidden_states" / "extraction_progress.npy",

    # --------------------------------------------------------
    # Split
    # --------------------------------------------------------

    PROJECT_DIR / "probes" / "probe_split.json",

    # --------------------------------------------------------
    # Primary 3-class probe
    # --------------------------------------------------------

    PROJECT_DIR / "probes" / "qwen_v2_final_3class_results.json",

    # --------------------------------------------------------
    # Primary binary probe
    # --------------------------------------------------------

    PROJECT_DIR / "probes" / "qwen_v2_final_binary_results.json",
    PROJECT_DIR / "probes" / "qwen_v2_binary_predictions.jsonl",
    PROJECT_DIR / "probes" / "qwen_v2_binary_bootstrap_ci.json",

    # --------------------------------------------------------
    # Final handoff / packaging
    # --------------------------------------------------------

    PROJECT_DIR / "probes" / "best_layers.json",
    PROJECT_DIR / "probes" / "metrics.json",
    PROJECT_DIR / "probes" / "manifest.json",

    # --------------------------------------------------------
    # Minimal-pair diagnostics
    # --------------------------------------------------------

    PROJECT_DIR / "analysis" / "qwen_v2_canonical_24_minimal_pairs.json",
    PROJECT_DIR / "analysis" / "qwen_v2_minimal_pair_render_style.json",

    # --------------------------------------------------------
    # Environment analysis
    # --------------------------------------------------------

    PROJECT_DIR / "analysis" / "qwen_v2_binary_environment_breakdown.json",

    # --------------------------------------------------------
    # Final RQ1 summary
    # --------------------------------------------------------

    PROJECT_DIR / "analysis" / "qwen_v2_rq1_summary.json",
]

print("\nChecking critical files:")
print("-" * 70)

missing_files = []

for path in critical_files:

    exists = path.exists()

    status = "✓" if exists else "✗"

    print(
        f"{status} "
        f"{path.relative_to(PROJECT_DIR)}"
    )

    if not exists:
        missing_files.append(path)


assert not missing_files, (
    "Missing critical files:\n"
    + "\n".join(
        str(p.relative_to(PROJECT_DIR))
        for p in missing_files
    )
)

# ------------------------------------------------------------
# Calculate project size
# ------------------------------------------------------------

total_bytes = sum(
    p.stat().st_size
    for p in PROJECT_DIR.rglob("*")
    if p.is_file()
)

print("\n" + "-" * 70)

print(
    f"Project size: "
    f"{total_bytes / (1024**3):.2f} GB"
)

print(
    f"Number of files: "
    f"{sum(1 for p in PROJECT_DIR.rglob('*') if p.is_file())}"
)

# ------------------------------------------------------------
# Remove an existing BACKUP_2 if present
# ------------------------------------------------------------

existing_zip = Path(
    str(ZIP_BASE) + ".zip"
)

if existing_zip.exists():

    print(
        "\nExisting BACKUP_2 found."
    )

    print(
        "Removing old BACKUP_2 so the new backup "
        "contains the latest project state."
    )

    existing_zip.unlink()

# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CREATING ZIP")
print("=" * 70)

print(
    "This may take a few minutes because the "
    "hidden-state arrays are large."
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=PROJECT_DIR.parent,
    base_dir=PROJECT_DIR.name
)

zip_size = os.path.getsize(zip_path)

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BACKUP COMPLETE")
print("=" * 70)

print("\nZIP:")
print(zip_path)

print(
    f"\nZIP size: "
    f"{zip_size / (1024**3):.2f} GB"
)

print(
    f"ZIP size: "
    f"{zip_size / (1024**2):.1f} MB"
)

assert os.path.exists(zip_path)
assert zip_size > 0

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "-" * 70)

print("✓ Critical files verified")
print("✓ Hidden states included")
print("✓ Extraction index included")
print("✓ Extraction metadata included")
print("✓ Probe split included")
print("✓ 3-class results included")
print("✓ Binary results included")
print("✓ Binary predictions included")
print("✓ Bootstrap CIs included")
print("✓ Best-layer selection included")
print("✓ Metrics package included")
print("✓ Manifest included")
print("✓ Minimal-pair diagnostics included")
print("✓ Environment analysis included")
print("✓ Final RQ1 summary included")
print("✓ Updated ZIP successfully created")

print("\n" + "=" * 70)
print("DOWNLOAD THIS FILE FROM KAGGLE:")
print(zip_path)
print("=" * 70)

ADNN RQ1 — CREATING UPDATED COMPLETE BACKUP

Checking critical files:
----------------------------------------------------------------------
✓ hidden_states/last_action_token.npy
✓ hidden_states/mean_action_tokens.npy
✓ hidden_states/extraction_index.jsonl
✓ hidden_states/extraction_manifest.jsonl
✓ hidden_states/extraction_metadata.json
✓ hidden_states/extraction_progress.npy
✓ probes/probe_split.json
✓ probes/qwen_v2_final_3class_results.json
✓ probes/qwen_v2_final_binary_results.json
✓ probes/qwen_v2_binary_predictions.jsonl
✓ probes/qwen_v2_binary_bootstrap_ci.json
✓ probes/best_layers.json
✓ probes/metrics.json
✓ probes/manifest.json
✓ analysis/qwen_v2_canonical_24_minimal_pairs.json
✓ analysis/qwen_v2_minimal_pair_render_style.json
✓ analysis/qwen_v2_binary_environment_breakdown.json
✓ analysis/qwen_v2_rq1_summary.json

----------------------------------------------------------------------
Project size: 0.83 GB
Number of files: 33

CREATING ZIP
This may take a few minutes because